# ComfyUI on a free Colab T4 - the_frizzy1 runner

Four cells. Everything else happens in the ComfyUI UI, where it belongs.

| | Cell | What it does |
|---|---|---|
| **1** | Setup | hardware check, install, Drive, and your last session back |
| **2** | Launch | speed layer, a frontend your browser can run, ComfyUI, a URL that is watched and rebuilt if it dies |
| **3** | Save state | pushes workflows, outputs and your node list to Drive |
| **4** | Models + doctor | stage and download models onto the VM, move any that landed wrong, then a diagnostic that fixes what it can |

### What changed in this build

- **The install cache is off by default and marked experimental.** It packed incomplete
  archives, so a restored tree looked installed and died on import with
  `No module named 'comfy.ldm.models'`. The packing bug is fixed, old archives are ignored,
  and anything restored is verified and repaired from git before use, but cloning is only
  slow and never wrong, so it stays off until you decide otherwise. Your workflows,
  settings and custom node list still restore as before: that is a different mechanism and
  was never affected.
- **The workflow cell is gone.** Drag a workflow JSON onto the canvas. Anything it needs
  goes through the Models panel, which is where models were always supposed to come from.
- **Downloads are staged, not fired.** Press Download and you get a list: filename, size,
  destination, duplicates elsewhere, free space. Every folder is a dropdown you can change
  before anything starts.
- **Folder detection no longer dumps everything in `checkpoints`.** It reads the repo's own
  path first (`split_files/text_encoders/...` lands in `text_encoders`), then the filename,
  and only then the extension. Anything it worked out from the extension alone is flagged
  amber, because that guess is the one that used to be wrong.
- **Anything already on disk can be moved.** Both the Models panel and cell 4 list what you
  have, newest first, with a dropdown per file. Moving refreshes ComfyUI's model lists, no
  restart.
- **The tunnel is watched.** A quick tunnel that dies is rebuilt within about 90 seconds and
  the new address is printed where you are already looking. The URL also lands in
  `/content/tunnel_url.txt`. One request a minute keeps an idle tunnel from being reaped in
  the first place, and the notebook tab gets its own keepalive.
- **API compression is off.** It was making `/api/settings` unparseable through
  localhost.run, which is a settings dialog that will not open while generation works fine.
  Asset caching and gzip on the frontend are untouched, so the app still loads at a third of
  its size.
- **aria2 everywhere**, 16 connections per file, in the panel and in cell 4.

---

### Set these up once (key icon, left sidebar)

| Secret | For |
|---|---|
| `HF_TOKEN` | gated HuggingFace models, and it is passed to the in-UI downloader |
| `CIVITAI_TOKEN` | CivitAI downloads |
| `NGROK_TOKEN` | the only tunnel that doesn't depend on Colab's shared IP |
| `GITHUB_TOKEN` | your private repos, and 5000 API calls/hr instead of 60 |

### What a free T4 actually is

15 GB of VRAM on 2018 silicon. Compute capability 7.5, which means no bf16, no fp8, no
FlashAttention 2 and no SageAttention. Big models fit; they just don't fly. Prefer GGUF
Q4_K_M or Q5_K_S over Q8.

Session dies at 12 hours, or after ~90 minutes idle. No terminal and no background execution
on free tier, so closing the tab ends the run.


## 1 - Setup

Everything up to "ready to launch". Run this first, and again after any runtime restart.

Drive mounts to `frizzy-comfy` every session and your last one is restored when a snapshot
exists. Models are not part of that: they stay on the VM's session disk, which is both
faster and where they have to be for ComfyUI to load them at any speed.

pip runs in three escalating passes - constrained, unconstrained, then `--no-deps` - and
tells you which one worked, so you know whether Colab's torch is at risk. The frontend is a
separate pip package and gets verified on disk, because a missing one is what leaves you
staring at a logo.

`user/` stays on local disk and is rsynced to Drive. It is never symlinked there: the
ComfyUI frontend calls `/api/userdata` while booting, and those calls stall on the Drive
FUSE mount.


In [ ]:
#@title 1. Setup - hardware, install, restore { display-mode: "form" }

#@markdown Drive mounts every session, always to the same folder, and your last session
#@markdown comes back when there is one to come back to. None of that is a choice any more:
#@markdown without Drive nothing survives the runtime, and restoring is what makes cell 3
#@markdown worth running. Models stay on the VM's session disk (~60-100 GB), because that is
#@markdown the only disk fast enough to load them from.
USE_DRIVE = True
DRIVE_FOLDER = "frizzy-comfy"
RESTORE_LAST_SESSION = True
VERSION_PIN = "last known good"  #@param ["last known good", "latest", "custom"]
#@markdown `master` ships most days, so a session that worked yesterday can break for
#@markdown reasons that have nothing to do with you. `last known good` rebuilds the exact
#@markdown ComfyUI commit and frontend versions cell 3 recorded from a session that worked,
#@markdown falling back to a set verified on a free T4. Use `latest` for new features.
COMFY_REF = "master"  #@param {type:"string"}
#@markdown Only used when VERSION_PIN is `custom`.
EXTRA_NODES = ""  #@param {type:"string"}
#@markdown Comma-separated git URLs, for nodes you always want.
USE_INSTALL_CACHE = False  #@param {type:"boolean"}
#@markdown EXPERIMENTAL. Restores a tar of the ComfyUI tree from Drive instead of cloning,
#@markdown which saves a couple of minutes. An earlier version of this wrote incomplete
#@markdown archives and produced a tree that looked installed and died on import. The
#@markdown packing bug is fixed, old archives are ignored, and anything restored is checked
#@markdown and repaired from git before use. Cloning is only slow, never wrong, so this
#@markdown stays off unless you turn it on.
USE_WHEELHOUSE = True  #@param {type:"boolean"}
#@markdown If cell 3 cached the wheels to Drive, install from them: offline, faster, and
#@markdown byte-identical to the session that worked. Falls back to the network if absent.
MANAGER_OFFLINE = False  #@param {type:"boolean"}
#@markdown Stops ComfyUI-Manager fetching its node database at boot. Try it if the UI
#@markdown stalls, but it is not the default because it also changes Manager's behaviour.

from pathlib import Path
LIB = r'''
# frizzy_lib - shared runtime for the the_frizzy1 ComfyUI Colab runner
# No triple-quoted strings anywhere in this file: it is embedded verbatim
# inside a raw string in notebook cell 1. The build script enforces it.
__version__ = '1.0.0'
__build__ = '84819129'          # replaced at build time with a content fingerprint

import os, re, sys, time, json, base64, random, shutil, socket, ssl, subprocess, tempfile
import threading
import urllib.request, urllib.error, urllib.parse
from pathlib import Path

ROOT = Path('/content/ComfyUI')
LOG = Path('/content/comfy.log')
CONS = Path('/content/constraints.txt')
TUNLOG = Path('/content/tunnel.log')
PORT = 8188
UA = 'Mozilla/5.0 (frizzy-colab)'


# ---------------------------------------------------------------- output
class C:
    OK = '\033[92m'
    WARN = '\033[93m'
    FAIL = '\033[91m'
    DIM = '\033[90m'
    X = '\033[0m'


def rule(t=''):
    print('\n' + '=' * 68)
    if t:
        print('  ' + t)
        print('=' * 68)


def step(m):
    print('\n>>> ' + m)


def ok(m):
    print('  ' + C.OK + 'OK' + C.X + '   ' + str(m))


def warn(m):
    print('  ' + C.WARN + 'WARN' + C.X + ' ' + str(m))


def fail(m):
    print('  ' + C.FAIL + 'FAIL' + C.X + ' ' + str(m))


def info(m):
    print('       ' + str(m))


def human(n):
    n = float(n or 0)
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024:
            return '%.2f %s' % (n, u)
        n /= 1024
    return '%.2f PB' % n


def summary(rows):
    rule('STATUS')
    for st, label, detail in rows:
        c = {'OK': C.OK, 'WARN': C.WARN, 'FAIL': C.FAIL}.get(st, '')
        print('  %s%-5s%s %-22s %s' % (c, st, C.X, label, detail))
    print('=' * 68)


# ---------------------------------------------------------------- errors
# (regex, what, why, [numbered fixes])
TAXONOMY = [
    (r'non matching host and origin|Sec-Fetch-Site|\b403\b.*(?:cors|origin|cross-site)|enable-cors-header',
     'ComfyUI is returning 403 to your browser',
     'ComfyUI ships a middleware that rejects any request whose fetch metadata says cross-site, and any request where Host and Origin disagree. Clicking a tunnel link from the Colab output is a cross-site navigation, so the browser gets a 403 and shows a blank page. Nothing looks wrong from inside the VM, because command-line tools do not send those headers.',
     ['Cell 2 passes --enable-cors-header by default, which replaces that middleware',
      'If you disabled CORS_HEADER, turn it back on and relaunch',
      'As a stopgap: open the tunnel URL by pasting it into the address bar and '
      'pressing Enter, rather than clicking the link']),

    (r'assets?/[\w.-]+\.(?:js|mjs).*(?:404|Not Found)|references no javascript',
     'index.html loads but its JavaScript does not',
     'This is the white screen. The page arrives, the app never mounts. Either the frontend package is incomplete, a custom node injected broken JavaScript, or the browser is holding a stale cached bundle.',
     ['Turn SAFE_MODE on in cell 2 - that starts with every custom node disabled',
      'If safe mode loads, a custom node is the cause; remove it and relaunch',
      'Hard reload the page: Ctrl+Shift+R, or open it in a private window',
      'If safe mode is also white, rerun cell 1 to repair the frontend package']),

    (r'CUDA out of memory|torch\.cuda\.OutOfMemoryError|CUDA error: out of memory',
     'GPU ran out of VRAM',
     'Model plus activations did not fit. A T4 has 15 GB, and bf16 weights get cast up on Turing, which costs more VRAM, not less.',
     ['Fetch a smaller quant (Q4_K_M instead of Q8) from the Models panel',
      'Set VRAM_MODE = lowvram in cell 2',
      'Lower resolution or frame count in the workflow',
      'Cell 4 with FREE_VRAM on, then re-queue']),

    (r'\bKilled\b\s*(python|$)|SIGKILL|signal 9|exit (code|status) 137|\bMemoryError\b|Cannot allocate memory|Out of memory: Kill',
     'System RAM ran out, not VRAM',
     'Colab gives about 12.7 GB of host RAM. Text encoders and node caching live there. The process dies with no traceback, which is why this looks random.',
     ['Keep DISABLE_NODE_CACHE on in cell 2 (passes --cache-none)',
      'Use the fp8 text encoder, not fp16 (6.7 GB vs 11.4 GB)',
      'Runtime > Restart session, then rerun cells 1 and 3']),

    (r'No space left on device|OSError: \[Errno 28\]|Disk quota exceeded',
     'Disk full',
     'Free tier gives roughly 78 GB and video models eat it fast.',
     ['Cell 4 with CLEAR_PIP_CACHE and CLEAR_HF_CACHE on',
      'rm /content/ComfyUI/models/diffusion_models/<file>',
      'Stage the download first - it prices the whole set before it starts']),

    (r'\b401\b|Unauthorized|Authorization header|gated repo|awaiting a review|Access to model .* is restricted',
     'HuggingFace refused the download (gated or private repo)',
     'It needs an accepted licence plus a token. Without one the server returns an HTML error page, which gets saved under a .safetensors name and fails cryptically later.',
     ['Open the model page on huggingface.co and accept the licence',
      'Left sidebar > key icon > add secret HF_TOKEN, enable notebook access',
      'Rerun the cell']),

    (r'\b404\b|Not Found|EntryNotFound|RepositoryNotFound',
     'File or repo does not exist at that path',
     'Quant filenames get renamed on these repos. A hardcoded URL rots within weeks.',
     ['The Models panel HEAD-checks every URL before it queues it',
      'If it reports the file is missing, keep the manifest default',
      'Or fix the URL in your repo manifest - that is the source of truth']),

    (r'error while deserializing header|HeaderTooLarge|MetadataIncompleteBuffer|invalid load key',
     'The file is not a model, it is an error page',
     'A failed download wrote HTML or JSON to disk under a model filename.',
     ['Delete the file named in the error',
      'Download it again - every file is magic-byte checked on arrival']),

    (r'Could not find a version that satisfies|ResolutionImpossible|conflicting dependencies|has requirement .*, but you',
     'pip could not resolve dependencies',
     'Usually the constraints file pinning Colab torch against a package that wants a different build.',
     ['Cell 1 already retries unconstrained, then --no-deps',
      'If all three passes fail: Runtime > Restart session, rerun cell 1']),

    (r'Torch not compiled with CUDA|CUDA is not available|libcudart|undefined symbol.*cuda|no kernel image is available',
     'torch was replaced with a build that does not match the driver',
     'Something ran pip with an --extra-index-url and swapped Colab torch for a mismatched wheel. This is the most common way a Colab ComfyUI notebook dies.',
     ['Runtime > Restart session - mandatory, the broken torch is still loaded',
      'Rerun cell 1; it writes a constraints file that prevents this',
      'Never add --extra-index-url to any pip line in this notebook']),

    (r'comfyui-frontend-package is not installed|No package metadata was found for comfyui-frontend|Could not find the frontend',
     'The frontend pip package is missing',
     'The ComfyUI UI ships as a pip package, not in the git repo. The server starts fine and the browser shows a logo and nothing else.',
     ['Rerun cell 1 - it installs and verifies the frontend explicitly',
      'If it persists, set LEGACY_FRONTEND = True in cell 2']),

    (r'Temporary failure in name resolution|Connection reset|Network is unreachable|urlopen error timed out|EOF occurred in violation',
     'Network problem on the Colab side',
     'Colab egress hiccup, or the remote host is rate limiting this shared IP.',
     ['Rerun - every network call retries with backoff',
      'If one host stays down, wait it out; the mirrors cover the rest']),

    (r'could not read Username|Authentication failed|Permission denied \(publickey\)|Repository not found',
     'Git repo is private, renamed, or misspelled',
     'Colab has no credentials, so private repos cannot be cloned anonymously.',
     ['Check the owner/repo spelling',
      'For your own private repos, add a GITHUB_TOKEN secret']),

    (r'Address already in use|bind.*8188|\[Errno 98\]',
     'Port 8188 is already taken',
     'An earlier ComfyUI is still running in the background.',
     ['Cell 4 with KILL_COMFY on',
      'Then rerun cell 2 - it also frees the port on start']),

    (r'ModuleNotFoundError: No module named .([\w_.]+).',
     'A python package is missing',
     'A custom node needs something whose requirements install failed quietly.',
     ['pip install the module named in the error',
      'Or rerun cell 1 without that node to confirm it is the cause']),

    (r'IMPORT FAILED|Cannot import.*custom_nodes|Failed to load custom node',
     'A custom node failed to load',
     'ComfyUI still starts but that node is gone, and a slow or broken node can stall the UI boot on /api/object_info.',
     ['Cell 2 prints which ones failed',
      'Move it out of custom_nodes/ and rerun cell 2']),

    (r'API rate limit exceeded|\b403\b.*rate limit|X-RateLimit-Remaining: 0',
     'GitHub API rate limit',
     'Unauthenticated GitHub allows 60 requests an hour per IP, and Colab IPs are shared with everyone else on that host.',
     ['Add a GITHUB_TOKEN secret (raises it to 5000/hr)',
      'Add GITHUB_TOKEN to Secrets: 5000 API calls an hour instead of 60']),

    (r'\b1033\b|Argo Tunnel error|\b502\b|Bad gateway|\b1015\b|rate limited|error 530|\b504\b',
     'The tunnel URL resolves but will not serve',
     'Either cloudflared lost its connection to ComfyUI, or the quick tunnel is being throttled from this Colab egress IP.',
     ['Cell 2 verifies every URL and falls through automatically',
      'Rerun cell 2 - quick tunnels are reassigned each time',
      'Add an NGROK_TOKEN secret; it does not depend on the shared IP']),

    (r'Host not in allowlist|failed to unmarshal quick Tunnel|Requesting new quick Tunnel.*\n.*ERR',
     'cloudflared could not reach Cloudflare',
     'The quick-tunnel API was unreachable or refused from this Colab host. This is an egress problem, not a ComfyUI one.',
     ['Cell 2 falls through to the next transport by itself',
      'Rerun cell 2 - Colab hands out a different egress IP each session',
      'Add an NGROK_TOKEN secret for a transport that does not depend on the IP']),

    (r'websocket.*fail|101 expected|Upgrade required|reconnecting',
     'HTTP works but websockets do not',
     'ComfyUI needs /ws for progress and previews. The Colab kernel proxy does not forward websockets at all.',
     ['Use cloudflared, pinggy or ngrok - never the Colab proxy for real work',
      'Rerun cell 2 to get a different transport']),

    (r'mountpoint must not already contain|Mountpoint must not|drive\.mount|MessageError.*drive',
     'Google Drive would not mount',
     'Usually a stale mount, or leftover local files sitting at /content/drive.',
     ['Runtime > Restart session',
      'Rerun cell 1 - it moves stray local files aside before mounting']),
]

_COMPILED = [(re.compile(p, re.I | re.M), w, y, f) for p, w, y, f in TAXONOMY]


def explain(text, quiet=False, limit=2):
    # Match text against the taxonomy and print WHAT / WHY / FIX.
    t = str(text)
    hits = [(w, y, f) for rx, w, y, f in _COMPILED if rx.search(t)]
    if not hits:
        if not quiet:
            print('\n  ' + C.WARN + 'Unrecognised error.' + C.X + ' Raw tail:')
            for l in t.strip().splitlines()[-15:]:
                print('    ' + l[:150])
        return None
    for w, y, fx in hits[:limit]:
        print('\n  ' + C.FAIL + 'WHAT: ' + C.X + w)
        print('  WHY:  ' + y)
        print('  FIX:')
        for i, f in enumerate(fx, 1):
            print('    %d. %s' % (i, f))
    return hits[0][0]


# ---------------------------------------------------------------- control flow
def retry(fn, tries=3, base=2.0, label='', fatal=False, quiet=False):
    last = None
    for a in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            last = e
            if a < tries:
                d = base ** a + random.uniform(0, 1.5)
                if not quiet:
                    warn('%s failed (%d/%d): %s' % (label or 'call', a, tries, str(e)[:110]))
                    info('retrying in %.1fs' % d)
                time.sleep(d)
    if not quiet:
        fail('%s failed after %d attempts' % (label or 'call', tries))
        explain(last, quiet=True)
    if fatal:
        raise SystemExit('Stopping: ' + (label or 'call') + ' is required.')
    return None


def sh(cmd, cwd=None, tries=1, label='', check=True, show=False, timeout=None):
    # Run a shell command with retries. Returns (returncode, combined output).
    rc, out = 1, ''
    for a in range(1, tries + 1):
        try:
            p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, timeout=timeout,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            rc, out = p.returncode, (p.stdout or '')
        except subprocess.TimeoutExpired:
            rc, out = 124, 'timed out after %ss' % timeout
        if show and out:
            print('\n'.join('    ' + l for l in out.strip().splitlines()[-25:]))
        if rc == 0:
            return 0, out
        if a < tries:
            d = 2.0 ** a + random.uniform(0, 1.5)
            warn('%s rc=%d (%d/%d), retrying in %.1fs' % (label or cmd[:40], rc, a, tries, d))
            time.sleep(d)
    if check:
        fail((label or cmd[:60]) + ' returned %d' % rc)
        explain(out)
    return rc, out


def wait_for(pred, timeout=60, interval=1.0, dots=False, on_dead=None):
    # Poll pred() until truthy or timeout. on_dead() short-circuits with None.
    end = time.time() + timeout
    while time.time() < end:
        if on_dead is not None and on_dead():
            return None
        v = pred()
        if v:
            if dots:
                print()
            return v
        if dots:
            print('.', end='', flush=True)
        time.sleep(interval)
    if dots:
        print()
    return False


def spawn_capture(cmd, pattern, timeout=75, logfile=None, shell=None):
    # Start a long-running process, tee its output to a file, and poll that file
    # for a regex. Never blocks on readline, which is why v5's tunnels hung.
    lf = Path(logfile or tempfile.mktemp(prefix='spawn_', suffix='.log'))
    lf.write_text('')
    fh = open(lf, 'w')
    use_shell = shell if shell is not None else isinstance(cmd, str)
    proc = subprocess.Popen(cmd, shell=use_shell, stdout=fh, stderr=subprocess.STDOUT, text=True)
    rx = re.compile(pattern)
    end = time.time() + timeout
    while time.time() < end:
        time.sleep(1.0)
        try:
            body = lf.read_text(errors='ignore')
        except Exception:
            body = ''
        m = rx.search(body)
        if m:
            return m.group(0), proc, body
        if proc.poll() is not None:
            return None, proc, body + '\n[process exited rc=%s]' % proc.returncode
    try:
        proc.kill()
    except Exception:
        pass
    return None, proc, (lf.read_text(errors='ignore') if lf.exists() else '')


def _ancestors(pid=None):
    # Every PID from here up to init, so we never kill ourselves or the kernel.
    out, cur = set(), (pid if pid is not None else os.getpid())
    for _ in range(40):
        out.add(cur)
        try:
            with open('/proc/%d/stat' % cur) as f:
                cur = int(f.read().rsplit(')', 1)[1].split()[1])
        except Exception:
            break
        if cur <= 1:
            break
    out.add(os.getpid())
    return out


def pids_matching(pattern):
    # pgrep without a shell, so the pattern never appears in a command line we
    # then match against. pgrep already excludes itself.
    try:
        r = subprocess.run(['pgrep', '-f', pattern], capture_output=True, text=True, timeout=15)
    except Exception:
        return []
    safe = _ancestors()
    pids = []
    for tok in r.stdout.split():
        try:
            pid = int(tok)
        except ValueError:
            continue
        if pid in safe or pid <= 1:
            continue
        pids.append(pid)
    return pids


def kill_matching(pattern, grace=2.0):
    # SIGTERM, wait, then SIGKILL. Returns the PIDs it actually signalled.
    import signal
    pids = pids_matching(pattern)
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except Exception:
            pass
    if pids:
        time.sleep(grace)
        for pid in pids:
            try:
                os.kill(pid, signal.SIGKILL)
            except Exception:
                pass
    return pids


KILL_PATTERNS = ['main.py --listen', 'cloudflared tunnel', 'a.pinggy.io', 'ngrok http', 'lt --port']


def kill_all(port=PORT, extra=()):
    killed = []
    for pat in list(KILL_PATTERNS) + list(extra):
        killed += kill_matching(pat, grace=0.5)
    try:
        subprocess.run(['fuser', '-k', '%d/tcp' % port], capture_output=True, timeout=15)
    except Exception:
        pass
    time.sleep(1.0)
    return killed


# ---------------------------------------------------------------- net
def get_json(url, token=None, timeout=45, tries=3, label='', quiet=False):
    def go():
        req = urllib.request.Request(url, headers={'User-Agent': UA})
        if token:
            req.add_header('Authorization', 'Bearer ' + token)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.load(r)
    return retry(go, tries=tries, label=label or url[:60], quiet=quiet)


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def head_size(url, token=None, timeout=30):
    # Content-Length via HEAD. HuggingFace also exposes x-linked-size for LFS.
    try:
        req = urllib.request.Request(url, method='HEAD', headers={'User-Agent': UA})
        if token and 'huggingface.co' in url:
            req.add_header('Authorization', 'Bearer ' + token)
        with urllib.request.urlopen(req, timeout=timeout) as h:
            for k in ('x-linked-size', 'Content-Length'):
                v = h.headers.get(k)
                if v and v.isdigit() and int(v) > 0:
                    return int(v)
    except Exception:
        pass
    return 0


def url_exists(url, token=None, timeout=25):
    try:
        req = urllib.request.Request(url, method='HEAD', headers={'User-Agent': UA})
        if token and 'huggingface.co' in url:
            req.add_header('Authorization', 'Bearer ' + token)
        with urllib.request.urlopen(req, timeout=timeout) as h:
            return 200 <= h.status < 400
    except Exception:
        return False


def browser_headers(url, mode='navigate', site='cross-site', origin=None):
    # ComfyUI's default middleware rejects requests whose fetch metadata says
    # cross-site. urllib sends none of these, which is why a plain check passes
    # while a real browser gets a 403. Emulate the browser instead.
    u = urllib.parse.urlparse(url)
    h = {'User-Agent': UA,
         'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
         'Sec-Fetch-Site': site,
         'Sec-Fetch-Mode': mode,
         'Sec-Fetch-Dest': 'document' if mode == 'navigate' else 'script'}
    if origin is None and site != 'none':
        origin = '%s://%s' % (u.scheme, u.netloc)
    if origin:
        h['Origin'] = origin
    return h


def check_url(url, timeout=30, want_markers=(b'<script', b'assets/'), browser=True,
              site='cross-site'):
    # True only if this really is the ComfyUI page, not a proxy error page.
    try:
        hdrs = browser_headers(url, site=site) if browser else {'User-Agent': UA}
        req = urllib.request.Request(url, headers=hdrs)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = r.read(300000)
            if r.status != 200:
                return False, 'HTTP %d' % r.status
            if any(m in body for m in want_markers):
                return True, 'HTTP 200, %s of HTML' % human(len(body))
            return False, 'HTTP 200 but no script tags (%s) - not the ComfyUI page' % human(len(body))
    except urllib.error.HTTPError as e:
        try:
            snip = e.read(300).decode('utf8', 'ignore').replace('\n', ' ').strip()[:110]
        except Exception:
            snip = ''
        if e.code == 403:
            return False, ('HTTP 403 - ComfyUI refused the request. Its origin-only '
                           'middleware rejects cross-site navigation. Relaunch with '
                           '--enable-cors-header.')
        return False, ('HTTP %d %s' % (e.code, snip)).strip()
    except Exception as e:
        return False, str(e)[:120]


def check_origin_policy(base_url, timeout=20):
    # Reproduce the three ways a browser can arrive, because ComfyUI treats them
    # differently. Clicking a link from Colab is the one that 403s.
    results = {}
    for label, site in (('typed into the address bar', 'none'),
                        ('clicked from another page', 'cross-site'),
                        ('same-origin subresource', 'same-origin')):
        try:
            req = urllib.request.Request(base_url.rstrip('/') + '/',
                                         headers=browser_headers(base_url, site=site))
            with urllib.request.urlopen(req, timeout=timeout) as r:
                results[label] = (r.status, len(r.read(200000)))
        except urllib.error.HTTPError as e:
            results[label] = (e.code, 0)
        except Exception as e:
            results[label] = (0, str(e)[:60])
    blocked = [k for k, v in results.items() if v[0] == 403]
    return results, blocked


ASSET_RE = re.compile(r'(?:src|href)\s*=\s*["\']([^"\']+\.(?:js|mjs|css))["\']', re.I)


def _asset_is_required(ref):
    # Only JavaScript is load-bearing. A real ComfyUI index.html also references
    # user.css and api/userdata/user.css, which 404 on a fresh install and are
    # supposed to - the app runs fine without them. Flagging those as broken
    # would condemn a healthy server.
    low = ref.lower()
    if low.endswith(('.js', '.mjs')):
        return '/api/' not in low and not low.startswith('api/')
    return False


def check_assets(base_url, timeout=25, limit=8):
    # A blank page is usually index.html arriving and its JavaScript not.
    # Fetch what index.html actually references and separate must-have from
    # nice-to-have. Returns (rows, error); each row is
    # (ref, status, bytes, ok, note, required).
    base = base_url.rstrip('/') + '/'
    try:
        req = urllib.request.Request(base, headers=browser_headers(base, site='none'))
        with urllib.request.urlopen(req, timeout=timeout) as r:
            html = r.read(400000).decode('utf8', 'ignore')
    except urllib.error.HTTPError as e:
        if e.code == 403:
            return [], ('index.html returned 403 - ComfyUI is refusing the request. '
                        'Relaunch with --enable-cors-header.')
        return [], 'could not fetch index.html: HTTP %d' % e.code
    except Exception as e:
        return [], 'could not fetch index.html: %s' % str(e)[:90]

    refs, seen = [], set()
    for m in ASSET_RE.finditer(html):
        u = m.group(1).strip()
        if not u or u.startswith(('data:', '//')) or u in seen:
            continue
        if u.startswith('http') and not u.startswith(base_url.rstrip('/')):
            continue                       # third-party CDN, not ours to police
        seen.add(u)
        refs.append(u)
    if not any(_asset_is_required(u) for u in refs):
        return [], ('index.html references no javascript at all - the frontend '
                    'package is incomplete')

    # Required first, so a low limit never hides the thing that matters.
    refs.sort(key=lambda u: not _asset_is_required(u))
    out = []
    for u in refs[:limit]:
        full = urllib.parse.urljoin(base, u)
        required = _asset_is_required(u)
        try:
            rq = urllib.request.Request(full, headers=browser_headers(
                base_url, mode='cors', site='same-origin'))
            with urllib.request.urlopen(rq, timeout=timeout) as r:
                body = r.read(200000)
                ctype = (r.headers.get('Content-Type') or '').lower()
                wrong_type = required and 'html' in ctype
                out.append((u, r.status, len(body), not wrong_type,
                            'served as HTML, not JavaScript' if wrong_type else '',
                            required))
        except urllib.error.HTTPError as e:
            why = ('HTTP 403 - blocked by ComfyUI, relaunch with --enable-cors-header'
                   if e.code == 403 else 'HTTP %d' % e.code)
            if not required and e.code == 404:
                why = 'HTTP 404 - optional, absent on a fresh install'
            out.append((u, e.code, 0, False, why, required))
        except Exception as e:
            out.append((u, 0, 0, False, str(e)[:70], required))
    return out, None


# index.html names only a handful of scripts. The app is ~65 chunks pulled in by
# dynamic import, and ONE unreachable chunk kills the whole ES module graph:
# blank page, nothing in the server log. These names are the ones ad blockers,
# browser shields and DNS filters commonly match.
ADBLOCK_PRONE = ('datadog', 'sentry', 'telemetry', 'firebase', 'analytic',
                 'tracker', 'tracking', 'gtm', 'segment', 'mixpanel', 'amplitude')


def parse_module_graph(base_url, timeout=25):
    # (entry_url, [chunk filenames], error)
    base = base_url.rstrip('/') + '/'
    try:
        req = urllib.request.Request(base, headers=browser_headers(base, site='none'))
        html = urllib.request.urlopen(req, timeout=timeout).read().decode('utf8', 'ignore')
    except Exception as e:
        return None, [], 'could not fetch index.html: %s' % str(e)[:80]
    m = re.search(r'src="([^"]*assets/index-[^"]+\.js)"', html)
    if not m:
        return None, [], 'index.html names no entry script'
    entry = urllib.parse.urljoin(base, m.group(1))
    try:
        req = urllib.request.Request(entry, headers=browser_headers(
            base_url, mode='cors', site='same-origin'))
        js = urllib.request.urlopen(req, timeout=timeout).read(400000).decode('utf8', 'ignore')
    except Exception as e:
        return entry, [], 'could not fetch the entry script: %s' % str(e)[:80]
    dm = re.search(r'm\.f\s*=\s*\[(.*?)\]', js, re.S)
    if not dm:
        return entry, [], 'entry script has no dependency map (frontend layout changed)'
    return entry, re.findall(r'"\./([^"]+\.js)"', dm.group(1)), None


def check_module_graph(base_url, deps, timeout=20, deadline=None):
    # (unreachable, adblock_prone, checked). Ranged GET so it stays cheap.
    base = base_url.rstrip('/')
    bad, checked = [], 0
    for d in deps:
        if deadline and time.time() > deadline:
            break
        h = browser_headers(base_url, mode='cors', site='same-origin')
        h['Range'] = 'bytes=0-64'
        try:
            req = urllib.request.Request(base + '/assets/' + d, headers=h)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                if r.status not in (200, 206):
                    bad.append((d, 'HTTP %d' % r.status))
        except urllib.error.HTTPError as e:
            bad.append((d, 'HTTP %d' % e.code))
        except Exception as e:
            bad.append((d, str(e)[:50]))
        checked += 1
    prone = [d for d in deps if any(b in d.lower() for b in ADBLOCK_PRONE)]
    return bad, prone, checked


def check_extensions(base_url, timeout=20, limit=40):
    # Custom nodes inject frontend JS, served from /extensions/<node>/*.js and
    # listed by /api/extensions. The app loads every one of them while booting,
    # so one that 404s or throws can leave you stuck on the splash screen.
    base = base_url.rstrip('/')
    listing = None
    for path in ('/api/extensions', '/extensions'):
        d = get_json(base + path, tries=1, quiet=True, timeout=timeout)
        if isinstance(d, list):
            listing = d
            break
    if listing is None:
        return None, [], 'could not read the extension list'
    bad = []
    for ext in listing[:limit]:
        u = ext if str(ext).startswith('http') else base + '/' + str(ext).lstrip('/')
        try:
            h = browser_headers(base_url, mode='cors', site='same-origin')
            h['Range'] = 'bytes=0-64'
            req = urllib.request.Request(u, headers=h)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                if r.status not in (200, 206):
                    bad.append((str(ext), 'HTTP %d' % r.status))
        except urllib.error.HTTPError as e:
            bad.append((str(ext), 'HTTP %d' % e.code))
        except Exception as e:
            bad.append((str(ext), str(e)[:50]))
    return listing, bad, None


def assets_broken(rows):
    # Only a failed *required* asset means the app cannot start.
    return [r for r in rows if r[5] and not r[3]]


def check_ws(base_url, path='/ws?clientId=frizzy-selftest', timeout=20):
    # Raw websocket handshake. ComfyUI needs /ws to finish booting, and the
    # Colab kernel proxy silently drops the upgrade - this is what catches that.
    u = urllib.parse.urlparse(base_url)
    host = u.hostname
    if not host:
        return False, 'unparseable url'
    https = (u.scheme == 'https')
    port = u.port or (443 if https else 80)
    key = base64.b64encode(os.urandom(16)).decode()
    req = (
        'GET %s HTTP/1.1\r\n'
        'Host: %s\r\n'
        'Upgrade: websocket\r\n'
        'Connection: Upgrade\r\n'
        'Sec-WebSocket-Key: %s\r\n'
        'Sec-WebSocket-Version: 13\r\n'
        'User-Agent: %s\r\n'
        'Origin: %s://%s\r\n\r\n'
    ) % (path, u.netloc, key, UA, u.scheme, u.netloc)
    s = None
    try:
        s = socket.create_connection((host, port), timeout=timeout)
        if https:
            s = ssl.create_default_context().wrap_socket(s, server_hostname=host)
        s.settimeout(timeout)
        s.sendall(req.encode())
        buf = b''
        end = time.time() + timeout
        while b'\r\n\r\n' not in buf and time.time() < end:
            chunk = s.recv(4096)
            if not chunk:
                break
            buf += chunk
        head = buf.decode('utf8', 'ignore')
        first = head.split('\r\n')[0] if head else ''
        if ' 101' in first:
            return True, 'websocket upgraded (101)'
        return False, 'no upgrade: %s' % (first[:80] or 'no response')
    except Exception as e:
        return False, str(e)[:110]
    finally:
        try:
            if s:
                s.close()
        except Exception:
            pass


# ---------------------------------------------------------------- files
HTML_HEADS = (b'<!DO', b'<htm', b'<HTM', b'{\"er', b'{\"me', b'<?xm', b'<!do')


def verify(path, expect_size=0, min_bytes=1000000):
    p = Path(path)
    if not p.exists():
        return False, 'no file was written'
    n = p.stat().st_size
    if n < min_bytes:
        return False, 'only %s - that is an error page, not a model' % human(n)
    with open(p, 'rb') as f:
        head = f.read(8)
    if head[:4] in HTML_HEADS:
        return False, 'server returned an error page (gated repo or bad token)'
    if p.suffix.lower() == '.gguf' and head[:4] != b'GGUF':
        return False, 'not a GGUF file (magic bytes %r)' % head[:4]
    if expect_size and abs(n - expect_size) > 2 * 1024 * 1024:
        return False, 'truncated: got %s, expected %s - rerun to resume' % (human(n), human(expect_size))
    return True, human(n)


def download(url, dest_dir, fname, expect_size=0, token=None, min_bytes=1000000):
    # aria2c -> curl -> wget -> urllib. Stops early on auth failures, since
    # a different downloader will not fix a 401.
    d = Path(dest_dir)
    d.mkdir(parents=True, exist_ok=True)
    dest = d / fname
    ha = ('--header=\"Authorization: Bearer %s\"' % token) if token else ''
    hc = ('-H \"Authorization: Bearer %s\"' % token) if token else ''
    methods = [
        ('aria2c', 'aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M --summary-interval=15 '
                   '--allow-overwrite=true --max-tries=3 --retry-wait=5 %s -d \"%s\" -o \"%s\" \"%s\"'
                   % (ha, d, fname, url)),
        ('curl', 'curl -L -C - --retry 3 --retry-delay 5 --fail-with-body %s -o \"%s\" \"%s\"'
                 % (hc, dest, url)),
        ('wget', 'wget -c -t 3 --waitretry=5 %s -O \"%s\" \"%s\"' % (ha, dest, url)),
    ]
    tried = []
    for name, cmd in methods:
        if shutil.which(name) is None:
            continue
        tried.append(name)
        info('trying %s' % name)
        sh(cmd, check=False, label=name)
        good, msg = verify(dest, expect_size, min_bytes)
        if good:
            return True, '%s via %s' % (msg, name)
        warn('%s: %s' % (name, msg))
        if 'error page' in msg:
            return False, msg
    try:
        info('trying python urllib')
        tried.append('urllib')
        req = urllib.request.Request(url, headers={'User-Agent': UA})
        if token:
            req.add_header('Authorization', 'Bearer ' + token)
        with urllib.request.urlopen(req, timeout=180) as r, open(dest, 'wb') as f:
            shutil.copyfileobj(r, f, 1024 * 1024)
        return verify(dest, expect_size, min_bytes)
    except Exception as e:
        return False, 'all downloaders failed (%s): %s' % ('/'.join(tried), str(e)[:100])


# ---------------------------------------------------------------- misc
def port_open(port=PORT, host='127.0.0.1', timeout=1.0):
    with socket.socket() as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0


def free_gb(p='/content'):
    try:
        return shutil.disk_usage(p).free / 1024 ** 3
    except Exception:
        return 0.0


def pkg_version(name):
    try:
        import importlib.metadata as m
        return m.version(name)
    except Exception:
        return None


QUANT_RE = re.compile(r'\bQ\d_(?:K(?:_[SML])?|[01])\b')
QUANT_LADDER = ['Q8_0', 'Q6_K', 'Q5_K_M', 'Q5_K_S', 'Q4_K_M', 'Q4_K_S', 'Q3_K_M', 'Q3_K_S', 'Q2_K']


def swap_quant(name, url, target):
    # Rewrite the quant token in a filename and its URL. Returns
    # (new_name, new_url, old_token) or (name, url, None) if there is nothing to do.
    m = QUANT_RE.search(name)
    if not m or m.group(0) == target:
        return name, url, None
    return QUANT_RE.sub(target, name), QUANT_RE.sub(target, url), m.group(0)


def is_workflow_json(raw_bytes):
    try:
        d = json.loads(raw_bytes)
    except Exception:
        return False
    if isinstance(d, dict) and 'nodes' in d and 'links' in d:
        return True
    if isinstance(d, dict) and d:
        vals = [v for v in list(d.values())[:8] if isinstance(v, dict)]
        if vals and all('class_type' in v for v in vals):
            return True
    return False


# ---------------------------------------------------------------- comfy health
BOOT_ENDPOINTS = [
    ('/', 'index.html'),
    ('/api/system_stats', 'backend alive'),
    ('/api/settings', 'user settings'),
    ('/api/userdata?dir=workflows', 'workflow list (a Drive symlink stalls here)'),
    ('/api/object_info', 'node schemas (a slow custom node stalls here)'),
]


def comfy_health(base='http://127.0.0.1:%d' % PORT, timeout=25):
    # Time every endpoint the frontend hits while booting. Returns (rows, worst).
    # Each row is (seconds, status, bytes, path, label, timed_out) so callers
    # never have to guess a sentinel value. worst is (seconds, path, label, timed_out).
    big = float(timeout) * 100
    rows, worst = [], None
    for path, label in BOOT_ENDPOINTS:
        t = time.time()
        try:
            req = urllib.request.Request(base + path, headers={'User-Agent': UA})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                body = r.read(400000)
            rows.append((time.time() - t, r.status, len(body), path, label, False))
        except socket.timeout:
            rows.append((big, 0, 0, path, label, True))
        except urllib.error.HTTPError as e:
            rows.append((time.time() - t, e.code, 0, path, label, False))
        except Exception:
            rows.append((big, -1, 0, path, label, True))
        if worst is None or rows[-1][0] > worst[0]:
            worst = (rows[-1][0], path, label, rows[-1][5])
    return rows, worst


def health_verdict(worst, timeout=25):
    # Turn the slowest endpoint into an actionable sentence.
    if not worst:
        return None
    dt, path, label = worst[0], worst[1], worst[2]
    timed_out = worst[3] if len(worst) > 3 else (dt >= timeout)
    if not timed_out:
        if dt > 20:
            return ('slow', '%s took %.0fs. Not hung, just slow - give the browser a full minute.' % (path, dt))
        return None
    if 'userdata' in path or 'settings' in path:
        return ('hung', 'userdata/settings is hung. That is a Drive symlink on user/. '
                        'Cell 2 auto-repairs it; rerun cell 1 then cell 2.')
    if 'object_info' in path:
        return ('hung', 'object_info is hung. A custom node stalls on import or on a network '
                        'call and the frontend blocks on this. Remove the node named in the '
                        'IMPORT FAILED lines and rerun cell 2.')
    if path == '/':
        return ('hung', 'The server will not even serve index.html. The frontend pip package '
                        'is broken - rerun cell 1, then try LEGACY_FRONTEND.')
    return ('hung', '%s is hung (%s).' % (path, label))


# ---------------------------------------------------------------- transports
_TUNNEL_PROCS = {}


def _register(name, proc):
    if proc is not None:
        _TUNNEL_PROCS[name] = proc


def kill_tunnel(name):
    # Stop one tunnel without touching ComfyUI.
    p = _TUNNEL_PROCS.pop(name, None)
    if p is not None:
        try:
            p.kill()
        except Exception:
            pass
    pat = {'cloudflared': 'cloudflared tunnel', 'pinggy': 'a.pinggy.io',
           'ngrok': 'ngrok http', 'localhost.run': 'localhost.run'}.get(name)
    if pat:
        kill_matching(pat, grace=0.5)


def _useful_tail(body, limit=220):
    # Prefer the line that explains the failure over the last line printed.
    lines = [l.strip() for l in (body or '').splitlines() if l.strip()]
    hot = [l for l in lines
           if re.search(r'\bERR\b|error|failed|fatal|denied|refused|timeout', l, re.I)]
    pick = hot[-2:] if hot else lines[-2:]
    out = ' / '.join(pick)
    return out[:limit] if out else 'no output'


def tunnel_cloudflared(port=PORT, timeout=75):
    if shutil.which('cloudflared') is None:
        sh('wget -q -O /tmp/cf.deb https://github.com/cloudflare/cloudflared/releases/latest/'
           'download/cloudflared-linux-amd64.deb && dpkg -i /tmp/cf.deb',
           tries=2, check=False, label='install cloudflared')
    if shutil.which('cloudflared') is None:
        return None, 'cloudflared would not install'
    url, proc, body = spawn_capture(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:%d' % port],
        r'https://[-\w]+\.trycloudflare\.com', timeout=timeout,
        logfile=str(TUNLOG) + '.cf', shell=False)
    _register('cloudflared', proc)
    if url:
        return url, ''
    return None, _useful_tail(body)


def tunnel_pinggy(port=PORT, timeout=60):
    if shutil.which('ssh') is None:
        return None, 'ssh not available'
    cmd = ('ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null '
           '-o ServerAliveInterval=20 -o ServerAliveCountMax=3 -o TCPKeepAlive=yes '
           '-o ExitOnForwardFailure=yes '
           '-p 443 -R0:localhost:%d a.pinggy.io' % port)
    url, proc, body = spawn_capture(cmd, r'https://[-\w.]+\.pinggy\.link',
                                    timeout=timeout, logfile=str(TUNLOG) + '.pinggy')
    _register('pinggy', proc)
    if url:
        return url, ''
    return None, _useful_tail(body)


def tunnel_localhost_run(port=PORT, timeout=60):
    # Same ssh mechanism as pinggy, different operator. Free, no account.
    if shutil.which('ssh') is None:
        return None, 'ssh is not installed'
    cmd = ('ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null '
           '-o ServerAliveInterval=20 -o ServerAliveCountMax=3 -o TCPKeepAlive=yes '
           '-o ExitOnForwardFailure=yes '
           '-R 80:localhost:%d nokey@localhost.run' % port)
    url, proc, body = spawn_capture(cmd, r'https://[-\w.]+\.lhr\.life',
                                    timeout=timeout, logfile=str(TUNLOG) + '.lhr')
    _register('localhost.run', proc)
    if url:
        return url, ''
    return None, _useful_tail(body)


def tunnel_ngrok(port=PORT, token=None, timeout=45):
    if not token:
        return None, 'no NGROK_TOKEN secret set'
    if shutil.which('ngrok') is None:
        sh('curl -sL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz '
           '| tar xz -C /usr/local/bin', tries=2, check=False, label='install ngrok')
    if shutil.which('ngrok') is None:
        return None, 'ngrok would not install'
    sh('ngrok config add-authtoken %s' % token, check=False)
    _register('ngrok', subprocess.Popen(
        ['ngrok', 'http', str(port), '--log', 'stdout'],
        stdout=open(str(TUNLOG) + '.ngrok', 'w'), stderr=subprocess.STDOUT))
    end = time.time() + timeout
    while time.time() < end:
        time.sleep(2)
        d = get_json('http://127.0.0.1:4040/api/tunnels', tries=1, quiet=True)
        for t in (d or {}).get('tunnels', []):
            if str(t.get('public_url', '')).startswith('https'):
                return t['public_url'], ''
    return None, 'ngrok started but published no https tunnel'


def tunnel_colab(port=PORT, **kw):
    # Returns a URL but has NO websocket support - always flagged degraded.
    try:
        from google.colab.output import eval_js
        url = eval_js('google.colab.kernel.proxyPort(%d)' % port)
        return (url, '') if url else (None, 'proxyPort returned nothing')
    except Exception as e:
        return None, str(e)[:110]


def tunnel_ping(url, timeout=25):
    # (alive, reason). /system_stats is small, always present, and answers while a
    # generation is running - as long as the event loop is not blocked.
    if not url:
        return False, 'no url'
    try:
        req = urllib.request.Request(url.rstrip('/') + '/system_stats',
                                     headers={'User-Agent': UA})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return (r.status == 200), 'http %s' % r.status
    except socket.timeout:
        return False, 'timeout'
    except Exception as e:
        name = type(e).__name__
        if 'URLError' in name or 'timed out' in str(e).lower():
            return False, 'timeout' if 'timed out' in str(e).lower() else 'unreachable'
        return False, name


def local_ping(port=None, timeout=25):
    # The discriminator. If 127.0.0.1 is also slow, ComfyUI is busy loading a model
    # or running a graph - the tunnel is innocent and must not be touched.
    p = port or PORT
    try:
        req = urllib.request.Request('http://127.0.0.1:%d/system_stats' % p,
                                     headers={'User-Agent': UA})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


def queue_busy(port=None, timeout=8):
    # Rebuilding a tunnel mid-render is the worst possible moment for it.
    try:
        d = get_json('http://127.0.0.1:%d/queue' % (port or PORT), tries=1, quiet=True,
                     timeout=timeout)
        if not d:
            return False
        return bool(d.get('queue_running')) or bool(d.get('queue_pending'))
    except Exception:
        return False


def watch_tunnel(url, name, rebuild, interval=60, grace=3, log=None):
    # Quick tunnels do not last: pinggy stops at 60 minutes, cloudflared throttles
    # per IP, and both ssh transports drop when the operator feels like it. From
    # the browser that looks exactly like ComfyUI dying, so this polls and rebuilds.
    #
    # The hard part is not noticing a dead tunnel. It is NOT killing a live one.
    # ComfyUI is single threaded: while it loads a text encoder into VRAM nothing
    # answers, including a perfectly healthy tunnel. So every miss is checked
    # against 127.0.0.1 first, and a tunnel is only rebuilt when the server
    # answers locally and does not answer through the tunnel.
    say = log or (lambda m: print('  [tunnel] %s' % m, flush=True))
    state = {'url': url, 'name': name, 'restarts': 0, 'last_ok': time.time(),
             'skipped': 0, 'stop': threading.Event()}

    def loop():
        misses = 0
        while not state['stop'].wait(interval):
            alive, why = tunnel_ping(state['url'])
            if alive:
                misses = 0
                state['last_ok'] = time.time()
                continue
            if not local_ping():
                # server busy or down: the supervisor's problem, not ours
                state['skipped'] += 1
                misses = 0
                continue
            if queue_busy():
                state['skipped'] += 1
                continue
            misses += 1
            # a hard connection error means it is genuinely gone; a timeout with a
            # healthy local server might just be a slow hop, so give that more rope
            needed = 1 if why in ('unreachable', 'ConnectionResetError',
                                  'RemoteDisconnected') else grace
            if misses < needed:
                continue
            say('%s stopped answering (%s) while ComfyUI is fine locally - rebuilding'
                % (state['name'], why))
            try:
                new_url, new_name, err = rebuild()
            except Exception as e:
                new_url, new_name, err = None, None, str(e)[:120]
            if new_url and tunnel_ping(new_url)[0]:
                state['restarts'] += 1
                state['name'] = new_name or state['name']
                if new_url != state['url']:
                    state['url'] = new_url
                    say('NEW URL (%s): %s' % (state['name'], new_url))
                    say('the old one is dead, open this one instead')
                else:
                    say('back on the same URL')
                state['last_ok'] = time.time()
                misses = 0
            else:
                say('could not rebuild it: %s' % (err or 'no url'))
                misses = needed - 1

    t = threading.Thread(target=loop, daemon=True)
    t.start()
    state['thread'] = t
    return state


def stop_watch(state):
    try:
        state['stop'].set()
    except Exception:
        pass


def pick_transport(candidates, verify_http=True, verify_ws=True, log=True):
    # candidates: list of (name, callable -> (url, err), degraded_bool)
    # Returns (url, name, degraded, attempts) where attempts records every try.
    attempts = []
    for name, fn, degraded in candidates:
        if log:
            info('trying %s' % name)
        try:
            url, err = fn()
        except Exception as e:
            url, err = None, str(e)[:110]
        if not url:
            attempts.append((name, False, err))
            kill_tunnel(name)
            if log:
                warn('%s: %s' % (name, err))
            continue
        if degraded:
            attempts.append((name, True, 'degraded, not verified'))
            if log:
                warn('%s: no websocket support, UI may not finish loading' % name)
            return url, name, True, attempts
        http_ok, http_msg = (check_url(url, timeout=40) if verify_http else (True, 'skipped'))
        if not http_ok:
            attempts.append((name, False, http_msg))
            kill_tunnel(name)
            if log:
                fail('%s gave a URL but it does not serve: %s' % (name, http_msg))
                explain(http_msg, quiet=True)
            continue
        ws_ok, ws_msg = (check_ws(url) if verify_ws else (True, 'skipped'))
        if not ws_ok:
            attempts.append((name, False, 'http ok but ws failed: ' + ws_msg))
            kill_tunnel(name)
            if log:
                fail('%s serves HTTP but not websockets: %s' % (name, ws_msg))
                info('ComfyUI needs /ws to finish booting - moving on')
            continue
        attempts.append((name, True, '%s + %s' % (http_msg, ws_msg)))
        if log:
            ok('%s verified: %s, %s' % (name, http_msg, ws_msg))
        return url, name, False, attempts
    return None, None, False, attempts


# ---------------------------------------------------------------- arg builder
def build_comfy_args(help_text, port=PORT, cc=0, vram=0.0, vram_mode='auto',
                     attention='auto', cache='none', previews=True,
                     legacy_frontend=False, verbose=False, disable_api_nodes=False,
                     reserve_vram=None, cors=True, extra=''):
    # Assemble ComfyUI flags, then drop any this build does not advertise, so a
    # renamed or removed flag degrades instead of killing the launch.
    #
    # Checked against comfyanonymous/ComfyUI master: --normalvram no longer
    # exists. The VRAM group is gpu-only / highvram / lowvram / novram / cpu, and
    # passing nothing lets DynamicVRAM manage it, which is what you want on a T4.
    want = ['--listen 127.0.0.1', '--port %d' % port, '--disable-auto-launch']

    if vram_mode not in ('auto', 'dynamic'):
        want.append('--' + vram_mode)
    if reserve_vram is not None:
        want.append('--reserve-vram %s' % reserve_vram)

    amap = {'pytorch': '--use-pytorch-cross-attention',
            'split': '--use-split-cross-attention',
            'quad': '--use-quad-cross-attention',
            'comfy default': ''}
    if attention in amap:
        want.append(amap[attention])
    else:
        # No FlashAttention or SageAttention below compute 8.0, so do not offer them.
        want.append('--use-pytorch-cross-attention' if cc < 80 else '')

    cmap = {'none': '--cache-none', 'classic': '--cache-classic',
            'lru': '--cache-lru 1', 'auto': ''}
    want.append(cmap.get(cache, '--cache-none'))

    # Without this, ComfyUI's origin-only middleware 403s any cross-site
    # navigation, which is exactly what clicking a tunnel link is.
    if cors:
        want.append('--enable-cors-header')
    if previews:
        want.append('--preview-method auto')
    if disable_api_nodes:
        want.append('--disable-api-nodes')
    if verbose:
        want.append('--verbose DEBUG')
    if legacy_frontend:
        want.append('--front-end-version Comfy-Org/ComfyUI_legacy_frontend@latest')
    if extra and extra.strip():
        want += extra.split()

    keep, drop = [], []
    for a in want:
        if not a:
            continue
        if not a.startswith('--'):
            keep.append(a)
            continue
        (keep if a.split('=')[0].split()[0] in help_text else drop).append(a)
    return keep, drop


# Escalation ladder for an out-of-memory launch. --lowvram is a no-op when
# DynamicVRAM is active, so it is not a useful first step any more.
OOM_LADDER = ['novram', 'cpu']


# ---------------------------------------------------------------- manifest
def plan_manifest(manifest, quant_mode='fit my GPU', vram=0.0, include_optional=True,
                  size_fn=None, exists_fn=None, log=True, manual_lister=None):
    # Turn a models.json into a concrete download plan.
    # size_fn(url) -> bytes, exists_fn(url) -> bool are injected so this is testable.
    size_fn = size_fn or (lambda u: head_size(u))
    exists_fn = exists_fn or (lambda u: url_exists(u))
    plan, total, skipped, notes = [], 0, 0, []

    manual = []
    for f in manifest.get('files', []):
        name, url = f.get('name'), f.get('url')
        dest, need = f.get('dest', 'diffusion_models'), f.get('required', True)
        note, swapped = f.get('note', ''), None
        if not name:
            notes.append('manifest entry with no name was skipped')
            continue
        if not need and not include_optional:
            skipped += 1
            continue

        # Entries with a page instead of a url need a human, unless the page is
        # a concrete HF repo we can resolve ourselves.
        if not url:
            got, why = resolve_manual(f, quant=quant_mode, lister=manual_lister, vram=vram)
            if got:
                for rname, rurl, rsize in got:
                    notes.append('resolved %s -> %s' % (name, rname))
                    total += rsize
                    plan.append({'name': rname, 'url': rurl, 'dest': dest, 'size': rsize,
                                 'required': need, 'note': note, 'swapped': None,
                                 'auto_resolved': True})
                continue
            manual.append({'name': name, 'page': f.get('page'), 'dest': dest,
                           'required': need, 'note': note, 'why': why})
            continue

        if name.lower().endswith('.gguf') and quant_mode != 'keep manifest default':
            if quant_mode == 'fit my GPU':
                cur = QUANT_RE.search(name)
                if cur and vram:
                    start = (QUANT_LADDER.index(cur.group(0))
                             if cur.group(0) in QUANT_LADDER else 0)
                    fitted = False
                    for cand in QUANT_LADDER[start:]:
                        n2, u2, old = swap_quant(name, url, cand)
                        sz = size_fn(url if old is None else u2)
                        if not sz:
                            continue
                        if sz / 1024 ** 3 <= vram * 0.80:
                            fitted = True
                            if old is not None:
                                notes.append('%s -> %s (%s fits %.0f GB)'
                                             % (old, cand, human(sz), vram))
                                name, url, swapped = n2, u2, old
                            break
                    if not fitted:
                        notes.append('no published quant of %s fits %.0f GB - '
                                     'keeping the manifest default' % (name, vram))
            else:
                n2, u2, old = swap_quant(name, url, quant_mode)
                if old and exists_fn(u2):
                    notes.append('%s -> %s' % (old, quant_mode))
                    name, url, swapped = n2, u2, old
                elif old:
                    notes.append('%s is not published in that repo - keeping %s' % (n2, old))

        size = size_fn(url)
        if not size and not exists_fn(url):
            notes.append('%s: URL does not resolve - fix it in the manifest' % name)
        total += size
        plan.append({'name': name, 'url': url, 'dest': dest, 'size': size,
                     'required': need, 'note': note, 'swapped': swapped,
                     'auto_resolved': False})
    return plan, total, skipped, notes, manual


# ---------------------------------------------------------------- manual entries
MODEL_EXT = ('.gguf', '.safetensors', '.sft', '.ckpt', '.pt', '.pth', '.bin')
_HF_PAGE = re.compile(r'https?://huggingface\.co/([\w.-]+)/([\w.-]+)/?$')


def hf_repo_from_page(page):
    # Only a concrete model page resolves. A search URL does not.
    if not page:
        return None
    m = _HF_PAGE.match(page.strip().rstrip('/'))
    if not m:
        return None
    owner, repo = m.group(1), m.group(2)
    if owner in ('models', 'datasets', 'spaces'):
        return None
    return '%s/%s' % (owner, repo)


def hf_list_files(repo, token=None):
    d = get_json('https://huggingface.co/api/models/%s/tree/main?recursive=1' % repo,
                 token=token, tries=2, quiet=True)
    if not isinstance(d, list):
        return []
    return [{'path': x['path'], 'size': x.get('size', 0)}
            for x in d if x.get('type') == 'file'
            and x['path'].lower().endswith(MODEL_EXT)]


# Extensions and filler carry no signal, and matching on them picks wrong files.
STOPWORDS = {'the', 'from', 'this', 'repo', 'and', 'for', 'grab', 'both', 'file', 'files',
             'single', 'all', 'one', 'search', 'hf', 'split', 'safetensors', 'gguf', 'ckpt',
             'pt', 'pth', 'bin', 'sft', 'model', 'models', 'main', 'use', 'with', 'that',
             'your', 'into', 'same', 'here', 'them', 'its'}


def _tokens(text):
    return set(t for t in re.split(r'[^a-z0-9]+', (text or '').lower()) if len(t) > 1)


def resolve_manual(entry, quant=None, lister=None, vram=0.0):
    # Try to turn a manual (page-only) manifest entry into real download URLs.
    # Returns (list_of(name, url, size), reason_if_unresolved).
    lister = lister or hf_list_files
    repo = hf_repo_from_page(entry.get('page'))
    if not repo:
        return [], 'no direct repo in the page link'
    files = lister(repo)
    if not files:
        return [], 'could not list %s' % repo

    name = entry.get('name', '')
    base = os.path.basename(name)
    # 1. exact filename match
    for f in files:
        if os.path.basename(f['path']).lower() == base.lower():
            return [(os.path.basename(f['path']),
                     'https://huggingface.co/%s/resolve/main/%s' % (repo, f['path']),
                     f['size'])], None

    # If the manifest names an actual file and that file is not in the repo, stop.
    # Guessing here downloads the wrong model, which is worse than asking a human.
    if base.lower().endswith(MODEL_EXT):
        return [], 'no file called %s in %s (filename entries are matched exactly)' % (base, repo)

    # 2. token scoring, for descriptive names like 'gemma_3_12B_it text encoder'
    want = _tokens(name) | _tokens(entry.get('note', ''))
    want -= STOPWORDS
    scored = []
    for f in files:
        stem = os.path.basename(f['path'])
        sc = len(want & _tokens(stem))
        if sc:
            scored.append((sc, f))
    if not scored:
        return [], 'nothing in %s matched %r' % (repo, name)
    best = max(s for s, _ in scored)
    cands = [f for s, f in scored if s == best]

    # 3. paired high-noise / low-noise experts
    blob = (name + ' ' + entry.get('note', '')).lower()
    if 'high' in blob and 'low' in blob:
        picked = []
        for side in ('highnoise', 'lownoise'):
            side_files = [f for f in cands
                          if side in os.path.basename(f['path']).lower().replace('-', '').replace('_', '')]
            pick = _pick_quant(side_files, quant, vram)
            if pick:
                picked.append(pick)
        if len(picked) == 2:
            return [(os.path.basename(f['path']),
                     'https://huggingface.co/%s/resolve/main/%s' % (repo, f['path']),
                     f['size']) for f in picked], None

    pick = _pick_quant(cands, quant, vram)
    if not pick:
        return [], 'no usable file among %d candidates in %s' % (len(cands), repo)
    return [(os.path.basename(pick['path']),
             'https://huggingface.co/%s/resolve/main/%s' % (repo, pick['path']),
             pick['size'])], None


def _pick_quant(files, quant, vram=0.0):
    if not files:
        return None
    if len(files) == 1:
        return files[0]
    if quant and quant not in ('fit my GPU', 'keep manifest default'):
        for f in files:
            m = QUANT_RE.search(os.path.basename(f['path']))
            if m and m.group(0) == quant:
                return f
    if vram:
        budget = vram * 0.80 * 1024 ** 3
        fits = [f for f in files if f['size'] and f['size'] <= budget]
        if fits:
            return max(fits, key=lambda f: f['size'])
    return min(files, key=lambda f: f['size'] or 1 << 62)


# ---------------------------------------------------------------- session
SESSION_FILE = Path('/content/.session_start')
SESSION_CAP_H = 12.0


def session_start():
    # First cell-1 run stamps the clock; later cells read it.
    try:
        if SESSION_FILE.exists():
            return float(SESSION_FILE.read_text().strip())
    except Exception:
        pass
    t = time.time()
    try:
        SESSION_FILE.write_text(str(t))
    except Exception:
        pass
    return t


def session_age():
    # (hours_elapsed, hours_left) against the free-tier 12 hour cap.
    used = (time.time() - session_start()) / 3600.0
    return used, max(0.0, SESSION_CAP_H - used)


def inventory():
    # What is actually installed right now, for a pre-launch sanity line.
    inv = {'models': {}, 'model_bytes': 0, 'workflows': 0, 'custom_nodes': 0}
    mroot = ROOT / 'models'
    if mroot.is_dir():
        for d in sorted(mroot.iterdir()):
            if not d.is_dir():
                continue
            files = [f for f in d.rglob('*')
                     if f.is_file() and f.suffix.lower() in MODEL_EXT]
            if files:
                n = sum(f.stat().st_size for f in files)
                inv['models'][d.name] = (len(files), n)
                inv['model_bytes'] += n
    wf = ROOT / 'user' / 'default' / 'workflows'
    if wf.is_dir():
        inv['workflows'] = sum(1 for f in wf.rglob('*') if f.is_file())
    cn = ROOT / 'custom_nodes'
    if cn.is_dir():
        inv['custom_nodes'] = sum(1 for d in cn.iterdir() if (d / '__init__.py').exists()
                                  or (d / '.git').exists())
    return inv


# ---------------------------------------------------------------- wheelhouse
# Colab reinstalls everything each session. Caching the wheels on Drive makes the
# install offline, fast, and identical every time - which is what you want when a
# working setup has to survive until you finish filming.
def wheelhouse_install(req_file, wheels_dir, constraints=None, python=None):
    # Returns (ok, message). Never touches the network.
    python = python or sys.executable
    w = Path(wheels_dir)
    if not w.is_dir() or not any(w.glob('*.whl')):
        return False, 'no wheelhouse at %s' % w
    n = len(list(w.glob('*.whl')))
    cons = ('-c %s' % constraints) if constraints else ''
    rc, out = sh('%s -m pip install --no-index --find-links="%s" -r "%s" %s'
                 % (python, w, req_file, cons), tries=1, check=False,
                 label='offline install')
    if rc == 0:
        return True, 'installed from %d cached wheels, no network used' % n
    missing = re.findall(r'No matching distribution found for ([\w.-]+)', out)
    return False, ('wheelhouse incomplete%s'
                   % ((': missing ' + ', '.join(sorted(set(missing))[:5])) if missing else ''))


def wheelhouse_build(req_file, wheels_dir, constraints=None, python=None, exclude=()):
    # Download every wheel the requirements need, minus anything Colab already
    # ships that we must never replace (torch and friends).
    python = python or sys.executable
    w = Path(wheels_dir)
    w.mkdir(parents=True, exist_ok=True)
    lines = [l.strip() for l in Path(req_file).read_text().splitlines()]
    keep = []
    for l in lines:
        if not l or l.startswith('#'):
            continue
        name = re.split(r'[=<>!~\[ ]', l)[0].strip().lower()
        if name in {x.lower() for x in exclude}:
            continue
        keep.append(l)
    tmp_req = Path('/content/_wheelhouse_req.txt')
    tmp_req.write_text('\n'.join(keep) + '\n')
    cons = ('-c %s' % constraints) if constraints else ''
    rc, out = sh('%s -m pip download -q -d "%s" -r "%s" %s'
                 % (python, w, tmp_req, cons), tries=2, check=False, label='pip download')
    n = len(list(w.glob('*.whl')))
    size = sum(f.stat().st_size for f in w.glob('*'))
    if rc != 0 and n == 0:
        return False, 'pip download failed', 0, 0
    return True, ('%d wheels, %s' % (n, human(size))), n, size


# Directories core ComfyUI scans under models/. Derived from upstream
# folder_paths.py; tests/test_real_comfy re-derives it and fails on drift.
# Anything outside this list only works if a custom node registers it.
CORE_MODEL_DIRS = (
    'audio_encoders', 'background_removal', 'checkpoints', 'classifiers', 'clip',
    'clip_vision', 'configs', 'controlnet', 'detection', 'diffusers',
    'diffusion_models', 'embeddings', 'frame_interpolation', 'geometry_estimation',
    'gligen', 'hypernetworks', 'latent_upscale_models', 'loras', 'model_patches',
    'optical_flow', 'photomaker', 'style_models', 't2i_adapter', 'text_encoders',
    'unet', 'upscale_models', 'vae', 'vae_approx')


def check_model_dest(dest, node_names=()):
    # (level, message). A dest outside the core list is not necessarily wrong -
    # custom nodes register their own - but a typo lands models where nothing looks.
    if dest in CORE_MODEL_DIRS:
        return 'ok', ''
    close = [d for d in CORE_MODEL_DIRS
             if d.startswith(dest[:5]) or dest.startswith(d[:5])]
    if close:
        return 'warn', ('models/%s is not a core ComfyUI folder - did you mean %s?'
                        % (dest, ' or '.join(close[:3])))
    if node_names:
        return 'info', ('models/%s is not a core folder; it has to be registered by '
                        'one of this workflow\'s custom nodes' % dest)
    return 'warn', ('models/%s is not a core ComfyUI folder and no custom node in this '
                    'workflow would register it - files there will be invisible' % dest)


PROTECTED_PKGS = ('torch', 'torchvision', 'torchaudio', 'triton', 'xformers',
                  'nvidia-cudnn-cu12', 'numpy')


# ---------------------------------------------------------------- safari
# ComfyUI's frontend ships syntax older Safari cannot COMPILE: regex lookbehind
# and class static blocks (16.4+) and the regex v flag (17+). A compile error
# kills the whole ES module graph, so the page stays on the splash screen with a
# perfectly healthy server. esbuild downlevels all of it; --front-end-root then
# serves the rewritten copy.
SAFARI_TARGETS = ('safari14', 'safari15', 'safari16', 'safari17')
_LOOKBEHIND = re.compile(r'\(\?<[=!](?:[^()\\\\]|\\\\.)*\)')
_REGEX_CALL = re.compile(r'(?:new\s+)?RegExp\(\s*')


def _strip_lookbehind(text):
    # Only for targets below Safari 16.4, where even new RegExp("(?<=a)b") throws
    # at runtime. esbuild has already turned literals into RegExp() calls, so the
    # remaining forms are string arguments - and they show up single quoted,
    # double quoted AND as template literals, with or without `new`.
    changed, out, i = [], [], 0
    for m in _REGEX_CALL.finditer(text):
        q = text[m.end():m.end() + 1]
        if q not in ('"', "'", '`'):
            continue
        j, esc = m.end() + 1, False
        while j < len(text):
            c = text[j]
            if esc:
                esc = False
            elif c == '\\':
                esc = True
            elif c == q:
                break
            j += 1
        if j >= len(text):
            continue
        body = text[m.end() + 1:j]
        if '(?<=' not in body and '(?<!' not in body:
            continue
        fixed = _LOOKBEHIND.sub('', body)
        if fixed == body:
            continue
        out.append(text[i:m.end() + 1])
        out.append(fixed)
        changed.append(body[:70])
        i = j
    out.append(text[i:])
    return ''.join(out), changed


def remaining_lookbehind(root):
    # Anything we could not rewrite, so the notebook can say so honestly.
    left = []
    for f in sorted(Path(root).rglob('*.js')):
        try:
            t = f.read_text(encoding='utf8', errors='ignore')
        except Exception:
            continue
        if '(?<=' in t or '(?<!' in t:
            left.append(f.name)
    return left


def install_check_page(html, root=None):
    # Drop a diagnostic page into whatever frontend directory is being served, so
    # it can be opened as a normal URL. Safari hides its inspector by default and
    # asking someone to enable it just to read one error is a bad trade.
    d = Path(root) if root else frontend_static_dir()
    if d is None or not d.is_dir():
        return None
    try:
        f = d / 'check.html'
        f.write_text(html, encoding='utf8')
        return f
    except Exception:
        return None


def frontend_static_dir():
    try:
        import comfyui_frontend_package as p
        d = Path(p.__file__).parent / 'static'
        return d if (d / 'index.html').is_file() else None
    except Exception:
        return None


def build_safari_frontend(dest='/content/comfyui_frontend_compat', target='safari15',
                          strip_lookbehind=None, log=True):
    # Returns (dest_path_or_None, stats_dict).
    src = frontend_static_dir()
    stats = {'target': target, 'files': 0, 'transpiled': 0, 'failed': [], 'regex_fixed': []}
    if src is None:
        return None, dict(stats, error='comfyui_frontend_package is not importable')
    if shutil.which('esbuild') is None:
        if log:
            info('installing esbuild (one-off, ~10 MB)')
        sh('npm install -g esbuild --silent', tries=2, check=False, label='npm esbuild')
    if shutil.which('esbuild') is None:
        return None, dict(stats, error='esbuild would not install - is npm available?')

    d = Path(dest)
    if d.exists():
        shutil.rmtree(d, ignore_errors=True)
    shutil.copytree(src, d)

    if strip_lookbehind is None:
        strip_lookbehind = target in ('safari14', 'safari15')

    js = sorted(d.rglob('*.js'))
    stats['files'] = len(js)
    for f in js:
        tmp = f.with_suffix('.js.__tmp')
        rc, out = sh('esbuild "%s" --target=%s --format=esm --outfile="%s" --log-level=error'
                     % (f, target, tmp), check=False)
        if rc != 0 or not tmp.exists():
            stats['failed'].append(f.name)
            if tmp.exists():
                tmp.unlink()
            continue
        text = tmp.read_text(encoding='utf8', errors='ignore')
        tmp.unlink()
        if strip_lookbehind:
            text, changed = _strip_lookbehind(text)
            for c in changed:
                stats['regex_fixed'].append('%s: %s' % (f.name, c))
        f.write_text(text, encoding='utf8')
        stats['transpiled'] += 1
    if strip_lookbehind:
        stats['lookbehind_left'] = remaining_lookbehind(d)
    return (d if stats['transpiled'] else None), stats


# ---------------------------------------------------------------- payload
# The app boots by downloading ~65 chunks, about 12.6 MB uncompressed. aiohttp's
# static handler serves a .gz sibling automatically when the browser accepts
# gzip, so pre-compressing turns that into roughly 3.8 MB with no ComfyUI change.
COMPRESSIBLE = ('.js', '.mjs', '.css', '.json', '.svg', '.map', '.html')


def precompress_frontend(root=None, level=6, min_bytes=4096):
    import gzip as _gz
    d = Path(root) if root else frontend_static_dir()
    stats = {'files': 0, 'raw': 0, 'gz': 0, 'skipped': 0}
    if d is None or not d.is_dir():
        return None, dict(stats, error='no frontend directory')
    for f in d.rglob('*'):
        if not f.is_file() or f.suffix.lower() not in COMPRESSIBLE:
            continue
        if f.name.endswith('.gz'):
            continue
        raw = f.stat().st_size
        if raw < min_bytes:
            stats['skipped'] += 1
            continue
        target = f.with_name(f.name + '.gz')
        if target.exists() and target.stat().st_mtime >= f.stat().st_mtime:
            stats['files'] += 1
            stats['raw'] += raw
            stats['gz'] += target.stat().st_size
            continue
        try:
            data = f.read_bytes()
            target.write_bytes(_gz.compress(data, level))
            stats['files'] += 1
            stats['raw'] += raw
            stats['gz'] += target.stat().st_size
        except Exception:
            stats['skipped'] += 1
    return d, stats


def measure_throughput(base_url, deps=None, timeout=180):
    # Download the single largest boot chunk in full and time it. Range requests
    # measure latency, not bandwidth - this is the number that says whether
    # "loading forever" is actually "loading slowly".
    base = base_url.rstrip('/')
    if deps is None:
        _e, deps, _err = parse_module_graph(base_url)
    if not deps:
        return None
    biggest, size = None, 0
    for d in deps:
        try:
            h = browser_headers(base_url, mode='cors', site='same-origin')
            req = urllib.request.Request(base + '/assets/' + d, method='HEAD', headers=h)
            with urllib.request.urlopen(req, timeout=30) as r:
                n = int(r.headers.get('Content-Length') or 0)
            if n > size:
                biggest, size = d, n
        except Exception:
            continue
    if not biggest:
        return None
    t = time.time()
    got = 0
    try:
        h = browser_headers(base_url, mode='cors', site='same-origin')
        h['Accept-Encoding'] = 'gzip, deflate'
        req = urllib.request.Request(base + '/assets/' + biggest, headers=h)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            enc = r.headers.get('Content-Encoding') or 'none'
            while True:
                chunk = r.read(262144)
                if not chunk:
                    break
                got += len(chunk)
    except Exception as e:
        return {'chunk': biggest, 'error': str(e)[:80]}
    dt = max(0.001, time.time() - t)
    return {'chunk': biggest, 'declared': size, 'got': got, 'secs': dt,
            'encoding': enc, 'mbps': (got * 8 / dt) / 1e6}


# ---------------------------------------------------------------- pinning
# ComfyUI ships to master most days. "It worked yesterday" is the expected
# outcome of tracking a moving branch, not bad luck. A pin records the exact
# commit and frontend versions from a session that worked, so you can get back
# to it - which matters most on the day you are recording.
# Full 40-character SHAs on purpose. GitHub will serve `git fetch origin <full-sha>`
# directly, but refuses a short one - and a shallow clone only reaches back about
# five weeks on a repo as busy as ComfyUI, so a short sha silently stops resolving.
KNOWN_GOOD = {
    'comfyui': '95d755cd8107a72258d452b5d3657273d571f07d',
    'comfyui_version': 'v0.34.0-18',
    'packages': {'comfyui-frontend-package': '1.51.9',
                 'comfyui-workflow-templates': '0.11.50',
                 'comfyui-embedded-docs': '0.5.10'},
    'nodes': {
        'ComfyUI-Manager': {'url': 'https://github.com/ltdrdata/ComfyUI-Manager',
                            'commit': 'f39cbd56fecae0b27a446c0cd450cd591f3a8bea'},
        'ComfyUI-GGUF': {'url': 'https://github.com/city96/ComfyUI-GGUF',
                         'commit': '6ea2651e7df66d7585f6ffee804b20e92fb38b8a'},
    },
    'note': 'verified working on a free Colab T4, 2026-08-30',
}
PIN_PACKAGES = ('comfyui-frontend-package', 'comfyui-workflow-templates',
                'comfyui-embedded-docs')


def current_pin(root=None):
    r = Path(root) if root else ROOT
    sha = subprocess.run('git -C "%s" rev-parse HEAD' % r, shell=True,
                         capture_output=True, text=True).stdout.strip()
    ver = subprocess.run('git -C "%s" describe --tags --always' % r, shell=True,
                         capture_output=True, text=True).stdout.strip()
    nodes = {}
    cn = r / 'custom_nodes'
    if cn.is_dir():
        for d in sorted(cn.iterdir()):
            if not (d / '.git').is_dir():
                continue
            g = lambda c: subprocess.run('git -C "%s" %s' % (d, c), shell=True,
                                         capture_output=True, text=True).stdout.strip()
            url, commit = g('config --get remote.origin.url'), g('rev-parse HEAD')
            if url and commit:
                nodes[d.name] = {'url': url, 'commit': commit}
    return {'comfyui': sha, 'comfyui_version': ver,
            'packages': {p: pkg_version(p) for p in PIN_PACKAGES if pkg_version(p)},
            'nodes': nodes,
            'python': sys.version.split()[0],
            'saved': time.strftime('%Y-%m-%d %H:%M')}


def read_pin(path):
    try:
        d = json.loads(Path(path).read_text())
        return d if d.get('comfyui') else None
    except Exception:
        return None


def write_pin(path, data=None):
    d = data or current_pin()
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(d, indent=2))
    return d


def fetch_commit(repo_dir, sha, url=None, log=True):
    # A full sha can be fetched straight from GitHub. A short one cannot, so fall
    # back to widening the shallow clone, then to a full history as a last resort.
    r = Path(repo_dir)
    full = len(str(sha)) >= 40
    if full:
        rc, _o = sh('git -C "%s" fetch --depth 1 origin %s' % (r, sha), tries=2,
                    check=False, label='fetch pinned commit')
        if rc == 0:
            rc, _o = sh('git -C "%s" checkout %s' % (r, sha), check=False, label='checkout')
            if rc == 0:
                return True, 'fetched by sha'
    for depth in (200, 1000):
        sh('git -C "%s" fetch --depth %d origin' % (r, depth), check=False,
           label='widen to %d' % depth)
        rc, _o = sh('git -C "%s" checkout %s' % (r, sha), check=False, label='checkout')
        if rc == 0:
            return True, 'found within %d commits' % depth
    sh('git -C "%s" fetch --unshallow origin' % r, check=False, label='full history')
    rc, _o = sh('git -C "%s" checkout %s' % (r, sha), check=False, label='checkout')
    return (rc == 0), ('found in full history' if rc == 0 else 'commit %s not found' % sha)


def apply_pin(pin, root=None, log=True):
    # Check out the recorded commit and install the recorded frontend versions.
    r = Path(root) if root else ROOT
    okc, okp = False, []
    sha = pin.get('comfyui')
    if sha:
        okc, how = fetch_commit(r, sha)
        if log:
            (ok if okc else warn)('ComfyUI pinned to %s (%s)' % (str(sha)[:12], how))
    want = [(p, v) for p, v in (pin.get('packages') or {}).items() if v]
    if want:
        spec = ' '.join('"%s==%s"' % (p, v) for p, v in want)
        rc, _o = sh('%s -m pip install -q %s' % (sys.executable, spec), tries=2,
                    check=False, label='pin frontend packages')
        for p, v in want:
            got = pkg_version(p)
            okp.append((p, v, got))
            if log:
                (ok if got == v else warn)('%-28s pinned %s, have %s' % (p, v, got))
    # custom nodes drift too, and a node update is just as capable of breaking a
    # session as a core update
    for name, meta in (pin.get('nodes') or {}).items():
        d = r / 'custom_nodes' / name
        if not (d / '.git').is_dir():
            continue
        good, how = fetch_commit(d, meta.get('commit'), meta.get('url'))
        if log:
            (ok if good else warn)('%-24s %s (%s)'
                                   % (name, str(meta.get('commit'))[:12], how))
    return okc, okp


# ---------------------------------------------------------------- manager
# ComfyUI-Manager gates node and model installs behind config it reads ONCE at
# startup. Two things must both be true: the flag in config.ini, and ComfyUI
# listening on a loopback address. We already listen on 127.0.0.1, so writing the
# config is the only missing half. security_level 'normal-' is the tier Manager
# defines for exactly this case - loopback-only, so the extra power is not
# reachable from outside the machine.
MANAGER_CONFIG = {
    'security_level': 'normal-',
    'allow_pip_install': 'true',
    'allow_git_url_install': 'true',
    'network_mode': 'public',
}


def configure_manager(root=None, overrides=None, log=True):
    r = Path(root) if root else ROOT
    if not (r / 'custom_nodes' / 'ComfyUI-Manager').is_dir():
        return None, 'ComfyUI-Manager is not installed'
    want = dict(MANAGER_CONFIG)
    want.update(overrides or {})
    # Manager has used a couple of locations across versions; write both.
    targets = [r / 'user' / 'default' / 'ComfyUI-Manager' / 'config.ini',
               r / 'user' / '__manager' / 'config.ini']
    written = []
    for cfg in targets:
        try:
            cfg.parent.mkdir(parents=True, exist_ok=True)
            existing = {}
            if cfg.is_file():
                for line in cfg.read_text().splitlines():
                    if '=' in line and not line.strip().startswith(('#', '[')):
                        k, _sep, v = line.partition('=')
                        existing[k.strip()] = v.strip()
            existing.update(want)
            body = '[default]\n' + ''.join('%s = %s\n' % (k, v)
                                           for k, v in sorted(existing.items()))
            cfg.write_text(body)
            written.append(str(cfg))
        except Exception:
            continue
    if log and written:
        ok('ComfyUI-Manager configured for Colab')
        info('security_level=normal-, pip installs and git-url installs allowed')
        info('this only works because ComfyUI listens on 127.0.0.1')
    return written, None


# ---------------------------------------------------------------- supervisor
def supervise(build_argv, root=None, log_path=None, max_restarts=8, window=900,
              on_event=None, poll=1.0, should_stop=None):
    # Keep ComfyUI running. It dies for three ordinary reasons on Colab: an OOM
    # kill, a custom node blowing up, and ComfyUI-Manager restarting it after an
    # install. The tunnel points at 127.0.0.1, so it survives all three - only
    # the server needs bringing back.
    r = Path(root) if root else ROOT
    lp = Path(log_path or LOG)
    events, restarts = [], []

    def emit(kind, msg):
        events.append((time.time(), kind, msg))
        if on_event:
            on_event(kind, msg)

    proc = None
    pos = 0

    def _stop():
        try:
            return bool(should_stop and should_stop())
        except Exception:
            return False

    try:
        while not _stop():
            if proc is None or proc.poll() is not None:
                if proc is not None:
                    code = proc.poll()
                    now = time.time()
                    restarts[:] = [t for t in restarts if now - t < window]
                    if len(restarts) >= max_restarts:
                        emit('giveup', 'exited %s and has restarted %d times in %d minutes'
                             % (code, len(restarts), window // 60))
                        return events
                    restarts.append(now)
                    delay = min(2 ** len(restarts), 20)
                    emit('restart', 'exited with %s - restarting in %ds (%d/%d)'
                         % (code, delay, len(restarts), max_restarts))
                    time.sleep(delay)
                # The previous process can still hold the listening socket for a
                # moment. Starting into an occupied port makes the new one die
                # immediately, which looks exactly like a crash loop.
                if port_open(PORT):
                    freed = wait_for(lambda: not port_open(PORT), timeout=20, interval=0.5)
                    if not freed:
                        emit('port', 'port %d still held - forcing it free' % PORT)
                        kill_matching('main.py --listen', grace=1.0)
                        try:
                            subprocess.run(['fuser', '-k', '%d/tcp' % PORT],
                                           capture_output=True, timeout=15)
                        except Exception:
                            pass
                        wait_for(lambda: not port_open(PORT), timeout=15, interval=0.5)
                fh = open(lp, 'a' if proc is not None else 'w')
                argv = build_argv()
                proc = subprocess.Popen('%s main.py %s' % (sys.executable, ' '.join(argv)),
                                        shell=True, cwd=str(r), stdout=fh,
                                        stderr=subprocess.STDOUT)
                emit('start', 'pid %s' % proc.pid)
                up = wait_for(lambda: port_open(PORT), timeout=420, interval=2,
                              on_dead=lambda: proc.poll() is not None)
                if up is None:
                    emit('died', 'exited during startup - see the log')
                elif up is False:
                    emit('failed', 'never opened port %d' % PORT)
                else:
                    emit('ready', 'listening on %d' % PORT)
            try:
                with open(lp, 'r', errors='ignore') as f:
                    f.seek(pos)
                    chunk = f.read()
                    pos = f.tell()
                if chunk:
                    sys.stdout.write(chunk)
                    sys.stdout.flush()
            except Exception:
                pass
            time.sleep(poll)
        emit('stopped', 'asked to stop')
        if proc and proc.poll() is None:
            proc.kill()
        return events
    except KeyboardInterrupt:
        emit('stopped', 'stopped from the notebook')
        if proc and proc.poll() is None:
            proc.kill()
        return events


# ---------------------------------------------------------------- install cache
# A fresh session re-clones ComfyUI and every custom node. The tree is only about
# 45 MB, 12 MB compressed, so caching it turns minutes of cloning into seconds of
# extraction. Models are excluded on purpose - they live on the session disk and
# would not fit in a free Drive anyway.
CACHE_EXCLUDE = ('models', 'output', 'input', 'temp')
CACHE_JUNK = ('__pycache__', '.venv', '.pytest_cache')

# A cached tree that cannot import is worse than no cache, because it looks
# installed. These are paths every usable checkout has.
CACHE_REQUIRED = ('main.py', 'execution.py', 'nodes.py', 'comfy/sd.py',
                  'comfy/ldm/models/autoencoder.py', 'comfy_extras', 'app')


def verify_install(root=None):
    r = Path(root) if root else ROOT
    missing = [p for p in CACHE_REQUIRED if not (r / p).exists()]
    return (not missing), missing


def repair_install(root=None):
    # Everything ComfyUI ships is tracked, so git puts back whatever went
    # missing without re-cloning.
    r = Path(root) if root else ROOT
    if not (r / '.git').exists():
        return False, 'not a git checkout'
    sh('git -C %s checkout -- .' % r, check=False, label='restore missing files')
    good, missing = verify_install(r)
    return good, ('repaired from git' if good
                  else 'still missing: %s' % ', '.join(missing[:4]))


def cache_install(dest, root=None, log=True):
    import tarfile
    r = Path(root) if root else ROOT
    d = Path(dest)
    d.parent.mkdir(parents=True, exist_ok=True)
    tmp = d.with_suffix('.tmp')

    def keep(ti):
        parts = ti.name.split('/')
        # TOP LEVEL ONLY. Matching any component also drops comfy/ldm/models,
        # which is where ComfyUI imports its VAE from, and the restored tree
        # then dies on import while looking perfectly installed.
        if len(parts) > 1 and parts[1] in CACHE_EXCLUDE:
            return None
        if any(p in CACHE_JUNK for p in parts) or ti.name.endswith('.pyc'):
            return None
        return ti

    t0 = time.time()
    try:
        with tarfile.open(tmp, 'w:gz', compresslevel=6) as tf:
            tf.add(str(r), arcname='ComfyUI', filter=keep)
        os.replace(tmp, d)
    except Exception as e:
        try:
            tmp.unlink()
        except Exception:
            pass
        return None, {'error': str(e)[:120]}
    st = {'bytes': d.stat().st_size, 'secs': time.time() - t0}
    if log:
        ok('install cached: %s in %.0fs' % (human(st['bytes']), st['secs']))
    return d, st


def restore_install(src, root=None, log=True):
    import tarfile
    s_ = Path(src)
    r = Path(root) if root else ROOT
    if not s_.is_file():
        return False, 'no cached install at %s' % s_
    t0 = time.time()
    try:
        with tarfile.open(s_, 'r:gz') as tf:
            tf.extractall(str(r.parent))
    except Exception as e:
        return False, 'could not extract: %s' % str(e)[:100]
    if not (r / 'main.py').is_file():
        return False, 'the cache extracted but main.py is missing'
    if log:
        ok('install restored from cache in %.0fs (no cloning needed)' % (time.time() - t0))
    return True, 'restored in %.0fs' % (time.time() - t0)
'''
Path("/content/frizzy_lib.py").write_text(LIB)

import sys, importlib
sys.path.insert(0, "/content")
import frizzy_lib
importlib.reload(frizzy_lib)
from frizzy_lib import *

import os, time, json, shutil, platform, subprocess
import importlib.metadata as pkgmeta

t0, rows = time.time(), []
session_start()
rule("SETUP")

# ---------------------------------------------------------------- hardware
step("Hardware")
gpu_name, cc, vram = None, 0, 0.0
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        gpu_name, cc, vram = p.name, p.major * 10 + p.minor, p.total_memory / 1024 ** 3
except Exception:
    pass

if not gpu_name:
    fail("no GPU attached")
    info("Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    info("Greyed out means Google has no free capacity right now.")
    info("Peak is 09:00-18:00 Pacific. Do not continue on CPU.")
    rows.append(("FAIL", "gpu", "none - stop here"))
else:
    ok("%s   %.1f GB VRAM   compute %.1f   torch %s"
       % (gpu_name, vram, cc / 10.0, pkg_version("torch")))
    y = lambda b: "yes" if b else "NO"
    info("bf16 %s | fp8 %s | FlashAttn2 %s | SageAttention %s"
         % (y(cc >= 80), y(cc >= 89), y(cc >= 80), y(cc >= 80)))
    if cc < 80:
        info("Pre-Ampere: prefer GGUF Q4_K_M / Q5_K_S. Q8 fits but crawls.")
    rows.append(("OK", "gpu", "%s %.0f GB" % (gpu_name, vram)))

try:
    import psutil
    ram = psutil.virtual_memory().total / 1024 ** 3
except Exception:
    ram = 0.0
disk = free_gb()
info("RAM %.1f GB | free disk %.1f GB | python %s" % (ram, disk, platform.python_version()))
if disk < 20:
    warn("under 20 GB free - run cell 4 with DEEP_CLEAN before downloading models")
rows.append(("WARN" if disk < 20 else "OK", "disk", "%.0f GB free" % disk))

# ---------------------------------------------------------------- tools
if shutil.which("aria2c") is None:
    step("Installing aria2 (4-6x faster model downloads)")
    sh("apt-get -qq install -y aria2", tries=2, check=False, label="apt aria2")
    (ok if shutil.which("aria2c") else warn)(
        "aria2c ready" if shutil.which("aria2c") else "aria2 unavailable, curl will be used")

# ---------------------------------------------------------------- drive
DRIVE = None
if USE_DRIVE:
    step("Google Drive")
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
                bk, n = "/content/drive_localbackup", 1
                while os.path.exists(bk):
                    bk = "/content/drive_localbackup_%d" % n
                    n += 1
                shutil.move("/content/drive", bk)
                warn("stray local files at /content/drive moved to " + bk)
            drive.mount("/content/drive")
        DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
        for s in ("user", "output", "input", "snapshots"):
            (DRIVE / s).mkdir(parents=True, exist_ok=True)
        ok(str(DRIVE))
        rows.append(("OK", "drive", DRIVE_FOLDER))
    except Exception as e:
        fail("Drive would not mount")
        explain(e)
        USE_DRIVE, DRIVE = False, None
        rows.append(("WARN", "drive", "not mounted, nothing will persist"))
else:
    rows.append(("WARN", "drive", "off, nothing will persist"))

# ---------------------------------------------------------------- pin
PIN = None
if VERSION_PIN == "last known good":
    saved = read_pin(DRIVE / "pin.json") if DRIVE else None
    PIN = saved or KNOWN_GOOD
    where = ("your last working session (%s)" % saved.get("saved", "?")) if saved \
        else "the built-in verified set (%s)" % KNOWN_GOOD.get("note", "")
    ok("pinning to %s" % where)
    info("ComfyUI %s  %s" % (PIN.get("comfyui"), PIN.get("comfyui_version", "")))
    COMFY_REF = PIN["comfyui"]
elif VERSION_PIN == "custom":
    COMFY_REF = (COMFY_REF or "master").strip()
    info("using %s" % COMFY_REF)
else:
    COMFY_REF = "master"
    warn("tracking master - it moves daily. Switch to `last known good` before recording.")

# ---------------------------------------------------------------- comfyui
if USE_INSTALL_CACHE and DRIVE and not ROOT.exists():
    step("Restoring the cached install")
    good, msg = restore_install(DRIVE / "install-cache-v2.tar.gz", root=ROOT)
    if not good:
        info(msg + " - cloning instead")
    else:
        whole, missing = verify_install(ROOT)
        if whole:
            rows.append(("OK", "install cache", "restored, no cloning"))
        else:
            warn("the cached tree is incomplete: %s" % ", ".join(missing[:4]))
            fixed_ok, how = repair_install(ROOT)
            if fixed_ok:
                ok(how + ", no re-clone needed")
                rows.append(("OK", "install cache", "restored and repaired"))
            else:
                warn("could not repair it (%s) - starting clean" % how)
                shutil.rmtree(ROOT, ignore_errors=True)
                rows.append(("WARN", "install cache", "discarded, cloning fresh"))

step("ComfyUI source")
if ROOT.exists():
    _whole, _missing = verify_install(ROOT)
    if not _whole:
        warn("this checkout is missing %d file(s): %s"
             % (len(_missing), ", ".join(_missing[:4])))
        info("that is the ModuleNotFoundError on comfy.ldm.models - repairing")
        _rok, _rhow = repair_install(ROOT)
        (ok if _rok else warn)(_rhow)
        if not _rok:
            shutil.rmtree(ROOT, ignore_errors=True)
            warn("could not repair it, cloning fresh")
if not ROOT.exists():
    src = None
    for m in ("https://github.com/comfyanonymous/ComfyUI",
              "https://github.com/Comfy-Org/ComfyUI"):
        rc, out = sh("git clone %s %s" % (m, ROOT), tries=2, check=False, label="clone")
        if rc == 0 and ROOT.exists():
            src = m
            break
        warn("mirror failed, trying the next one")
    if not src:
        fail("could not clone ComfyUI from any mirror")
        explain(out)
        raise SystemExit("Cannot continue without the source.")
    ok("cloned from " + src)

if PIN:
    # a pinned SHA may be older than a shallow clone reaches
    sh("git fetch --depth 200 origin", cwd=ROOT, tries=2, check=False, label="fetch")
    rc, _o = sh("git checkout %s" % COMFY_REF, cwd=ROOT, check=False, label="checkout pin")
    if rc != 0:
        sh("git fetch --unshallow origin", cwd=ROOT, check=False, label="unshallow")
        rc, _o = sh("git checkout %s" % COMFY_REF, cwd=ROOT, check=False, label="checkout pin")
    if rc != 0:
        warn("could not check out %s - falling back to master" % COMFY_REF)
        sh("git checkout master && git pull --ff-only origin master", cwd=ROOT, check=False)
        PIN = None
else:
    sh("git fetch origin %s" % COMFY_REF, cwd=ROOT, tries=2, check=False, label="fetch")
    sh("git checkout %s 2>/dev/null || git checkout FETCH_HEAD" % COMFY_REF,
       cwd=ROOT, check=False)
    sh("git pull --ff-only origin %s" % COMFY_REF, cwd=ROOT, check=False)
sha = subprocess.run("git rev-parse --short HEAD", shell=True, cwd=ROOT,
                     capture_output=True, text=True).stdout.strip()
ok("at " + sha)
rows.append(("OK", "comfyui", sha))

# ---------------------------------------------------------------- pip
step("Pinning Colab's torch so pip cannot replace it")
pins = ["%s==%s" % (n, pkg_version(n)) for n in
        ("torch", "torchvision", "torchaudio", "numpy", "triton", "xformers")
        if pkg_version(n)]
CONS.write_text("\n".join(pins) + "\n")
info(" ".join(pins))

step("Requirements")
pip_mode, last = None, ""

if USE_WHEELHOUSE and DRIVE:
    wok, wmsg = wheelhouse_install(ROOT / "requirements.txt", DRIVE / "wheels", CONS)
    if wok:
        ok(wmsg)
        pip_mode = "wheelhouse (offline)"
    else:
        info(wmsg + " - falling back to the network")

PASSES = (
    ("constrained", "%s -m pip install -r requirements.txt -c %s" % (sys.executable, CONS)),
    ("unconstrained", "%s -m pip install -r requirements.txt" % sys.executable),
    ("no-deps", "%s -m pip install -r requirements.txt -c %s --no-deps" % (sys.executable, CONS)),
)
for name, cmd in (PASSES if pip_mode is None else ()):
    rc, last = sh(cmd, cwd=ROOT, tries=2, check=False, label="pip " + name)
    if rc == 0:
        pip_mode = name
        ok("clean on the %s pass" % name)
        break
    warn("%s pass failed, escalating" % name)
if pip_mode is None:
    fail("all three pip passes failed")
    explain(last)
    rows.append(("FAIL", "requirements", "all passes failed"))
else:
    rows.append(("OK" if pip_mode in ("constrained", "wheelhouse (offline)") else "WARN",
                 "requirements", pip_mode))

try:
    importlib.reload(pkgmeta)
    import torch
    if torch.cuda.is_available():
        ok("torch %s, CUDA still available" % pkg_version("torch"))
    else:
        fail("torch is present but CUDA is gone")
        explain("Torch not compiled with CUDA")
        rows.append(("FAIL", "torch", "restart the runtime"))
except Exception as e:
    fail("torch will not import")
    explain(e)
    rows.append(("FAIL", "torch", "restart the runtime"))

# ---------------------------------------------------------------- frontend
step("Frontend package  (a blank or spinning UI is almost always this)")
FRONT = ["comfyui-frontend-package", "comfyui-workflow-templates", "comfyui-embedded-docs"]
want = [l.strip() for l in (ROOT / "requirements.txt").read_text().splitlines()
        if l.strip() and re.split(r"[=<>!\[ ]", l.strip())[0].strip().lower() in FRONT]
if PIN and PIN.get("packages"):
    want = ["%s==%s" % (k, v) for k, v in PIN["packages"].items() if v]
    info("pinning the frontend: " + ", ".join(want))
sh("%s -m pip install -U %s" % (sys.executable,
   " ".join('"%s"' % w for w in want) if want else "comfyui-frontend-package"),
   tries=2, check=False, label="frontend")

fe = True
for name in FRONT:
    v = pkg_version(name)
    (ok if v else warn)("%-32s %s" % (name, v or "not installed"))
    if name == FRONT[0] and not v:
        fe = False
try:
    import comfyui_frontend_package as _cfp
    idx = Path(_cfp.__file__).parent / "static" / "index.html"
    if idx.is_file():
        ok("static/index.html present (%s)" % human(idx.stat().st_size))
    else:
        fail("static/index.html is missing")
        fe = False
except Exception as e:
    fail("cannot import comfyui_frontend_package: %s" % str(e)[:90])
    fe = False
if not fe:
    explain("comfyui-frontend-package is not installed")
rows.append(("OK" if fe else "FAIL", "frontend", "verified" if fe else "broken, UI will not load"))

rows.append(("OK" if PIN else "WARN", "version",
             ("pinned to %s" % PIN.get("comfyui")) if PIN
             else "tracking master (moves daily)"))

# ---------------------------------------------------------------- nodes
step("Base custom nodes")
CN = ROOT / "custom_nodes"
CN.mkdir(exist_ok=True)
BASE = {"ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager",
        "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF"}
for u in [x.strip() for x in EXTRA_NODES.split(",") if x.strip()]:
    BASE[u.rstrip("/").split("/")[-1].replace(".git", "")] = u

nok, nbad = [], []
for name, url in BASE.items():
    d = CN / name
    if d.exists():
        info("= " + name)
        nok.append(name)
        continue
    rc, out = sh("git clone --depth 1 %s '%s'" % (url, d), tries=3, check=False, label=name)
    if rc != 0:
        fail(name + " could not be cloned - skipping, ComfyUI still starts")
        explain(out, quiet=True)
        nbad.append(name)
        continue
    r = d / "requirements.txt"
    if r.exists():
        rc2, _ = sh("%s -m pip install -q -r '%s' -c %s" % (sys.executable, r, CONS),
                    tries=2, check=False, label=name + " deps")
        if rc2 != 0:
            warn(name + ": dependency install failed, the node may not load")
    ok("+ " + name)
    nok.append(name)

mcfg = ROOT / "user" / "default" / "ComfyUI-Manager" / "config.ini"
if MANAGER_OFFLINE and (CN / "ComfyUI-Manager").exists() and not mcfg.exists():
    mcfg.parent.mkdir(parents=True, exist_ok=True)
    mcfg.write_text("[default]\nnetwork_mode = offline\n")
    info("Manager set offline so it cannot stall the UI at boot")
elif mcfg.exists() and not MANAGER_OFFLINE:
    body = mcfg.read_text()
    if "network_mode = offline" in body:
        mcfg.write_text(body.replace("network_mode = offline", "network_mode = public"))
        info("Manager offline mode from an earlier run has been cleared")
rows.append(("OK" if not nbad else "WARN", "base nodes",
             "%d ok%s" % (len(nok), (", failed: " + ", ".join(nbad)) if nbad else "")))

# ---------------------------------------------------------------- manager
step("ComfyUI-Manager")
_w, _e = configure_manager()
if _e:
    info(_e)
else:
    rows.append(("OK", "manager", "node + model installs enabled"))

# ---------------------------------------------------------------- layout
# These have nothing to do with Drive. Creating them only inside the restore
# block meant a Drive-less session started without user/ or input/, and the
# frontend calls /api/userdata while booting.
step("Working directories")
for d in ("user", "user/default", "user/default/workflows", "input", "output",
          "models", "custom_nodes"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)
ok("user/, input/, output/, models/ ready (local disk, not Drive)")
info("models live here: %s  -  %.0f GB free" % (ROOT / "models", free_gb()))

# ---------------------------------------------------------------- restore
if USE_DRIVE and RESTORE_LAST_SESSION and DRIVE:
    step("Restoring last session")
    for s in ("user", "input"):
        (ROOT / s).mkdir(parents=True, exist_ok=True)
        sh('rsync -a "%s/" "%s/"' % (DRIVE / s, ROOT / s), check=False, label="rsync " + s)
        info("%s/  %d files" % (s, sum(1 for _ in (ROOT / s).rglob("*") if _.is_file())))
    snap = DRIVE / "snapshots" / "latest.json"
    if snap.exists():
        try:
            saved = json.loads(snap.read_text())
        except Exception:
            saved = {"nodes": {}}
        missing = {k: v for k, v in saved.get("nodes", {}).items() if not (CN / k).exists()}
        for name, meta in missing.items():
            rc, out = sh("git clone %s '%s'" % (meta["url"], CN / name), tries=2,
                         check=False, label=name)
            if rc != 0:
                warn(name + " could not be restored")
                continue
            if meta.get("commit"):
                sh("git checkout %s" % meta["commit"], cwd=CN / name, check=False)
            r = (CN / name) / "requirements.txt"
            if r.exists():
                sh("%s -m pip install -q -r '%s' -c %s" % (sys.executable, r, CONS),
                   tries=2, check=False)
            ok("+ " + name)
        ok("snapshot from %s: %d nodes, %d needed cloning"
           % (saved.get("when", "?"), len(saved.get("nodes", {})), len(missing)))
        rows.append(("OK", "restored", "%d nodes" % len(saved.get("nodes", {}))))
    else:
        info("no snapshot to restore yet - cell 3 writes the first one")

used, left = session_age()
rows.append(("OK", "elapsed", "%.0fs setup, %.1fh into a %.0fh session"
             % (time.time() - t0, used, SESSION_CAP_H)))
rows.append(("OK", "next", "cell 2 to launch"))
summary(rows)


## 2 - Launch

Four things happen automatically before ComfyUI starts.

**The Models panel and the cache layer get installed.** ComfyUI sends
`Cache-Control: no-store` on every `.js` and `.css`, so the browser is *forbidden* from
caching and refetches the whole 12.6 MB app on every page load. The asset filenames are
content-hashed, so this pins them for a year and leaves `index.html` revalidating: repeat
loads go from 72 requests to about 7. API responses are deliberately left uncompressed;
compressing them here broke `/api/settings` through localhost.run, and a tunnel that
re-encodes what it forwards will gzip them for you anyway.

**Your browser is asked what it can compile.** Colab's output runs in your browser, so the
cell probes it directly for regex lookbehind, class static blocks and the regex `v` flag
rather than guessing from a user-agent string. If anything is missing - any Safari below 17,
anything on iOS - the frontend is rewritten with esbuild for the right target and served
from there. Force it with `os.environ["FRIZZY_COMPAT"] = "safari15"` if you open the URL in
a different browser than the one you are reading this in.

**Everything is pre-compressed.** ~65 chunks, 12.6 MB, gzipped once in about three seconds;
aiohttp then serves the `.gz` automatically. 12.6 MB becomes 3.8 MB on the wire.

**The tunnel gets a watchdog.** Quick tunnels do not last: pinggy stops at 60 minutes,
cloudflared throttles per IP, and both ssh transports drop when the operator feels like it.
From the browser that looks exactly like ComfyUI dying, when in fact the server is untouched
and only the URL is gone. The watchdog pings through the tunnel every 45 seconds, which also
stops an idle one being reaped, and rebuilds it after two misses. It tries the transport
that was working first, then the others, verifies HTTP and websockets, and prints the new
address in the log you are already watching. The ssh transports now also run with
`ServerAliveInterval=20` so the connection is held open rather than waiting to be dropped.

### The blank page

ComfyUI's default middleware in `server.py` returns **403 to any request whose
`Sec-Fetch-Site` header says `cross-site`**, and to any request where `Host` and `Origin`
disagree while `Host` is loopback. Clicking a tunnel link printed in Colab's output *is* a
cross-site navigation, so the browser gets a 403 and renders nothing. `CORS_HEADER` is on by
default and swaps that middleware for a permissive CORS one.

### Four checks run before you are given a URL

1. **index.html** arrives with real markup.
2. **The JavaScript it references actually loads.** A white screen is index.html plus a 404
   on `/assets/*.js`, and only this check sees it.
3. **A websocket upgrade succeeds on `/ws`.** ComfyUI blocks on `/ws` while booting.
   Google's port proxy passes HTTP and drops websockets, which leaves you on the logo.
4. **All three ways a browser can arrive are accepted** - typed, clicked, and as a
   subresource, each sent with real browser fetch-metadata headers.

**White screen? Turn on `SAFE_MODE`.** It starts ComfyUI with every custom node disabled,
mine included. If the UI appears, a node's JavaScript is what breaks it. If it stays white,
the frontend package is at fault and cell 1 will repair it.

Flags are checked against this build's own `--help` before launch. `--normalvram` was
removed from ComfyUI, and `--lowvram` does nothing while DynamicVRAM is active, so `auto`
passes no VRAM flag at all and an out-of-memory start escalates to `--novram` instead.


In [ ]:
import sys, os
if not os.path.exists("/content/frizzy_lib.py"):
    raise SystemExit("Run cell 1 (Setup) first.")
sys.path.insert(0, "/content")
import importlib, frizzy_lib
importlib.reload(frizzy_lib)
from frizzy_lib import *
#@title 3. Launch ComfyUI { display-mode: "form" }

VRAM_MODE = "auto"  #@param ["auto", "lowvram", "novram", "highvram", "gpu-only"]
#@markdown `auto` passes no VRAM flag and lets DynamicVRAM manage it. That is the right
#@markdown answer on a T4. `--normalvram` was removed from ComfyUI, and `--lowvram` is a
#@markdown no-op while DynamicVRAM is on, so `novram` is the real escalation.
RESERVE_VRAM_GB = 0.4  #@param {type:"number"}
#@markdown Colab is headless, so almost nothing else needs VRAM. Lower than a desktop default. 0 to skip.
ATTENTION = "auto"  #@param ["auto", "pytorch", "split", "quad", "comfy default"]
#@markdown No FlashAttention or SageAttention below compute 8.0, so a T4 gets pytorch.
CACHE = "none"  #@param ["none", "auto", "lru", "classic"]
#@markdown `none` re-executes every node each run but survives 12.7 GB of host RAM.
#@markdown `auto` is ComfyUI's RAM-pressure default: faster, more likely to get the VM killed.
#@markdown Compression and asset caching are always on, and the browser is asked directly
#@markdown whether it can compile the stock frontend. If it cannot, the rewritten one is
#@markdown built for it automatically - no toggle to remember, no Safari version to guess.
CORS_HEADER = True  #@param {type:"boolean"}
#@markdown Leave this ON. ComfyUI's default middleware returns 403 to any cross-site
#@markdown navigation, and clicking a tunnel link from this output IS one. That 403 is
#@markdown the blank page. This flag replaces that middleware.
SAFE_MODE = False  #@param {type:"boolean"}
#@markdown Starts with EVERY custom node disabled. If the UI is a white screen, turn this on:
#@markdown it loads, a custom node's JavaScript is the cause. It stays white, the frontend is.
DISABLE_API_NODES = False  #@param {type:"boolean"}
#@markdown Stops api.comfy.org calls at boot. It also cuts the frontend off from the internet,
#@markdown so try it as a fix, not as a default.
PREVIEWS = True  #@param {type:"boolean"}
LEGACY_FRONTEND = False  #@param {type:"boolean"}
#@markdown Last resort if the new frontend still will not load.
ACCESS = "auto"  #@param ["auto", "cloudflared", "pinggy", "localhost.run", "ngrok", "colab tab", "colab iframe"]
#@markdown The two colab options skip tunnels entirely and use Google's own port proxy.
#@markdown It does not forward websockets, so previews and progress will not work and the
#@markdown app may not finish loading. Useful for proving the server is alive.
AUTO_RESTART = True  #@param {type:"boolean"}
#@markdown Keeps ComfyUI alive. It dies for three ordinary reasons here: an OOM kill, a
#@markdown custom node crashing, and ComfyUI-Manager restarting it after you install
#@markdown something. The tunnel points at localhost, so it survives all three - only the
#@markdown server needs bringing back, and your URL keeps working.
AUTO_RETRY_ON_OOM = True  #@param {type:"boolean"}
STARTUP_TIMEOUT = 480  #@param {type:"integer"}
VERBOSE = False  #@param {type:"boolean"}
EXTRA_ARGS = ""  #@param {type:"string"}

# ---------------------------------------------------------------- payload
# The custom node that makes models download HERE and the UI cache properly.
# Kept as text so one cell writes it, and cell 4 can repair it from a copy.
DOWNLOADER_PY = r'''
"""Server-side model downloader for ComfyUI.

Everything here runs on the machine ComfyUI runs on, which on Colab is the VM.
That is the whole point: the browser never touches the bytes.

Folders come from ComfyUI's own folder_paths registry, so a file always lands
where ComfyUI actually looks for it. When a folder resolves to more than one
path - which is what extra_model_paths.yaml does - the local VM disk wins over
a Drive mount, because a 12 GB write through the Drive FUSE layer is slow
enough to outlive the session.

aria2c does the transfer when it is present: 16 connections instead of 1, which
is 4-6x on HuggingFace. urllib is kept as the fallback so this still works on a
box without aria2.
"""
import json
import os
import queue
import re
import shutil
import subprocess
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

UA = "ComfyUI-FrizzyDownloader/2.0"
CHUNK = 1024 * 1024
HTML_HEADS = (b"<!DO", b"<htm", b"<HTM", b'{"er', b'{"me', b"<?xm", b"<!do")
ARIA2 = shutil.which("aria2c")
DRIVE_PREFIXES = ("/content/drive", "/gdrive", "/content/gdrive")

# filename or url hint -> the folder ComfyUI scans
ROUTING = [
    (r"vae[-_.]?approx", "vae_approx"),
    (r"\bvae\b|[-_]vae[-_.]", "vae"),
    (r"clip[-_]?vision", "clip_vision"),
    (r"text[-_]?encoder|umt5|t5xxl|clip[-_]?l\b|clip[-_]?g\b|gemma|mistral|qwen.*encoder",
     "text_encoders"),
    (r"controlnet|control[-_]?v\d|control[-_]lora|control[-_]sd\d|t2i[-_]?adapter",
     "controlnet"),
    (r"\blora\b|[-_]lora[-_.]|lightx2v|distill.*rank|relight", "loras"),
    (r"upscal|esrgan|realesr|swinir", "upscale_models"),
    (r"embedding|textual[-_]inversion", "embeddings"),
    (r"style[-_]model", "style_models"),
    (r"audio[-_]?encoder|wav2vec", "audio_encoders"),
    (r"ipadapter|ip[-_]adapter", "controlnet"),
    (r"\.gguf$", "diffusion_models"),
    (r"unet|diffusion[-_]model|transformer", "diffusion_models"),
]


def comfy_root():
    try:
        import folder_paths
        return os.path.dirname(os.path.abspath(folder_paths.__file__))
    except Exception:
        return os.getcwd()


def on_drive(path):
    p = os.path.abspath(path or "")
    return any(p.startswith(x) for x in DRIVE_PREFIXES)


def _free(path):
    probe = path
    for _ in range(8):
        if os.path.isdir(probe):
            try:
                return shutil.disk_usage(probe).free
            except Exception:
                return 0
        nxt = os.path.dirname(probe)
        if not nxt or nxt == probe:
            return 0
        probe = nxt
    return 0


def split_folder(folder):
    """'loras/wan' -> ('loras', 'wan'). Subfolders are kept, not flattened."""
    folder = (folder or "").strip().strip("/")
    if "/" in folder:
        name, sub = folder.split("/", 1)
        return name, sub.strip("/")
    return folder, ""


def folder_search_paths(name):
    try:
        import folder_paths
        paths = [p for p in folder_paths.get_folder_paths(name) if p]
    except Exception:
        paths = []
    if not paths:
        paths = [os.path.join(comfy_root(), "models", name)]
    return paths


def pick_path(name):
    """The path a new file should be written to, and why.

    Order: an existing writable local directory with the most free space, then
    any local candidate, then whatever is left. Drive paths are last on purpose.
    """
    paths = folder_search_paths(name)
    local = [p for p in paths if not on_drive(p)]
    pool = local or paths

    existing = [p for p in pool if os.path.isdir(p) and os.access(p, os.W_OK)]
    if existing:
        chosen = max(existing, key=_free)
    else:
        chosen = pool[0]

    note = ""
    if on_drive(chosen) and not local:
        note = "only a Drive path is registered for this folder - writes will be slow"
    elif local and any(on_drive(p) for p in paths):
        note = "a Drive path exists for this folder, using the VM disk instead"
    return chosen, note


def registered_folders():
    """Every folder ComfyUI scans, with the path a download would go to."""
    try:
        import folder_paths
        names = sorted(folder_paths.folder_names_and_paths.keys())
    except Exception:
        return []
    out = []
    for name in names:
        paths = folder_search_paths(name)
        if not paths:
            continue
        target, note = pick_path(name)
        try:
            n = len(folder_paths.get_filename_list(name))
        except Exception:
            n = 0
        subs = []
        if os.path.isdir(target):
            try:
                subs = sorted(d for d in os.listdir(target)
                              if os.path.isdir(os.path.join(target, d)))[:40]
            except Exception:
                subs = []
        out.append({"name": name, "path": target, "paths": paths, "files": n,
                    "free": _free(target), "drive": on_drive(target),
                    "subfolders": subs, "note": note})
    return out


# .safetensors says nothing about what a file IS - almost every model ships as
# one - so these are the last resort, not the first rule.
EXT_FALLBACK = [(r"\.gguf$", "diffusion_models"), (r"\.(ckpt|pt|pth)$", "checkpoints"),
                (r"\.safetensors$", "checkpoints")]

# what a folder is called in a URL vs what ComfyUI calls it
PATH_ALIASES = {"unet": "diffusion_models", "unet_models": "diffusion_models",
                "diffusion_models": "diffusion_models", "transformers": "diffusion_models",
                "text_encoder": "text_encoders", "text_encoders": "text_encoders",
                "clip": "text_encoders", "clip_vision": "clip_vision", "vae": "vae",
                "vae_approx": "vae_approx", "loras": "loras", "lora": "loras",
                "controlnet": "controlnet", "controlnets": "controlnet",
                "embeddings": "embeddings", "upscale_models": "upscale_models",
                "upscale": "upscale_models", "checkpoints": "checkpoints",
                "style_models": "style_models", "audio_encoders": "audio_encoders",
                "ipadapter": "controlnet", "photomaker": "photomaker",
                "model_patches": "model_patches", "ultralytics": "ultralytics"}


def known_folders():
    try:
        import folder_paths
        return set(folder_paths.folder_names_and_paths.keys())
    except Exception:
        return set(PATH_ALIASES.values())


def guess_folder_conf(url, filename=""):
    """(folder, confidence). confidence is high when the URL itself names a
    folder, medium when the filename says what the file is, low when all we had
    to go on was the extension - which is why everything used to land in
    checkpoints."""
    known = known_folders()
    segs = [urllib.parse.unquote(x).lower()
            for x in urllib.parse.urlparse(url).path.split("/") if x]

    # 1. the repo already sorted it: .../split_files/text_encoders/umt5.safetensors
    for seg in reversed(segs[:-1] if segs else []):
        if seg in known:
            return seg, "high"
        mapped = PATH_ALIASES.get(seg)
        if mapped:
            return (mapped if mapped in known or not known else mapped), "high"

    # 2. the name says what it is
    name = (filename or "").lower()
    from_url = segs[-1] if segs else ""
    rest = urllib.parse.unquote(url).lower()
    for hay in (name, from_url, rest):
        if not hay:
            continue
        for pat, folder in ROUTING:
            if re.search(pat, hay):
                return folder, "medium"

    # 3. nothing but the extension
    for hay in (name, from_url):
        for pat, folder in EXT_FALLBACK:
            if hay and re.search(pat, hay):
                return folder, "low"
    return "checkpoints", "low"


def guess_folder(url, filename=""):
    return guess_folder_conf(url, filename)[0]


def move_model(src, folder, filename=None):
    """Put a file that is already on disk into a different folder. Used for the
    ones the guess got wrong, after the fact."""
    src = os.path.abspath(src)
    if not os.path.isfile(src):
        raise ValueError("no such file: %s" % src)
    name, sub = split_folder(folder)
    base, _note = pick_path(name)
    dest_dir = os.path.join(base, sub) if sub else base
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, filename or os.path.basename(src))
    if os.path.abspath(dest) == src:
        return dest, "already there"
    if os.path.exists(dest):
        raise ValueError("%s already exists" % dest)
    shutil.move(src, dest)
    try:
        import folder_paths
        cache = getattr(folder_paths, "filename_list_cache", None)
        if isinstance(cache, dict):
            cache.clear()
    except Exception:
        pass
    return dest, ""


def list_models(limit=400):
    """Everything under models/, newest first - which is the order you want when
    you are fixing where the last download went."""
    root = os.path.join(comfy_root(), "models")
    out = []
    if not os.path.isdir(root):
        return out
    for dirpath, _dirs, files in os.walk(root):
        for f in files:
            if f.endswith((".part", ".aria2", ".download")) or f.startswith("."):
                continue
            full = os.path.join(dirpath, f)
            try:
                st = os.stat(full)
            except Exception:
                continue
            rel = os.path.relpath(dirpath, root)
            out.append({"path": full, "name": f, "folder": "" if rel == "." else rel,
                        "size": st.st_size, "mtime": st.st_mtime})
    out.sort(key=lambda x: -x["mtime"])
    return out[:limit]


def filename_from(url, headers=None):
    if headers:
        cd = headers.get("Content-Disposition") or ""
        m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?', cd)
        if m:
            return urllib.parse.unquote(m.group(1)).strip()
    path = urllib.parse.urlparse(url).path
    name = urllib.parse.unquote(os.path.basename(path))
    return name or "download.bin"


def validate(path, expect=0):
    """Cheap checks that catch the ways a 'finished' download is really an
    error page. Size alone is not one of them: a config json is 2 kB and a
    small LoRA is under a megabyte."""
    if not os.path.exists(path):
        return False, "nothing was written"
    n = os.path.getsize(path)
    if n < 4096 and not path.lower().endswith((".json", ".yaml", ".yml", ".txt")):
        return False, "only %d bytes - that is an error page, not a model" % n
    with open(path, "rb") as f:
        head = f.read(8)
    if head[:4] in HTML_HEADS and not path.lower().endswith((".json", ".yaml", ".yml")):
        return False, "the server sent an error page (gated repo, or a bad token)"
    if path.lower().endswith(".gguf") and head[:4] != b"GGUF":
        return False, "not a GGUF file"
    if path.lower().endswith(".safetensors") and n > 1_000_000:
        # the first 8 bytes are the little-endian length of the JSON header
        hlen = int.from_bytes(head, "little")
        if hlen <= 0 or hlen > n:
            return False, "not a safetensors file - the header length is nonsense"
    if expect and abs(n - expect) > 2 * 1024 * 1024:
        return False, "incomplete: %d of %d bytes" % (n, expect)
    if n > 1024 ** 3:
        size = "%.2f GB" % (n / 1024 ** 3)
    elif n > 1024 ** 2:
        size = "%.1f MB" % (n / 1024 ** 2)
    else:
        size = "%.0f KB" % (n / 1024)
    return True, size


def auth_for(url, tokens):
    if ("huggingface.co" in url or "hf.co" in url) and tokens.get("hf"):
        return {"Authorization": "Bearer " + tokens["hf"]}
    return {}


def with_civitai(url, tokens):
    if "civitai.com" in url and tokens.get("civitai"):
        sep = "&" if "?" in url else "?"
        return url + sep + "token=" + tokens["civitai"]
    return url


class Job:
    _next = 1
    _lock = threading.Lock()

    def __init__(self, url, folder=None, filename=None):
        with Job._lock:
            self.id = Job._next
            Job._next += 1
        self.url = url.strip()
        self.folder = folder
        self.folder_explicit = bool(folder)
        self.confidence = "high" if folder else ""
        self.filename = filename
        self.state = "queued"      # queued running done failed cancelled
        self.done_bytes = 0
        self.total = 0
        self.speed = 0.0
        self.via = ""
        self.message = ""
        self.dest = ""
        self.started = None
        self.finished = None
        self._cancel = threading.Event()
        self._proc = None

    def as_dict(self):
        pct = (self.done_bytes / self.total * 100) if self.total else 0
        eta = ((self.total - self.done_bytes) / self.speed) if (self.speed and self.total) else 0
        return {"id": self.id, "url": self.url, "folder": self.folder,
                "confidence": self.confidence,
                "filename": self.filename, "state": self.state,
                "done": self.done_bytes, "total": self.total, "pct": round(pct, 1),
                "speed": round(self.speed / 1e6, 2), "eta": int(eta),
                "via": self.via, "message": self.message, "dest": self.dest}


class Downloader:
    """One worker, one download at a time. Parallel jobs on a Colab VM just split
    the same pipe and make progress unreadable - aria2 already parallelises the
    connections within a single file, which is where the speed actually is."""

    def __init__(self):
        self.jobs = []
        self.q = queue.Queue()
        self.tokens = {"hf": os.environ.get("HF_TOKEN", ""),
                       "civitai": os.environ.get("CIVITAI_TOKEN", "")}
        self._lock = threading.Lock()
        self._worker = None

    def set_tokens(self, hf=None, civitai=None):
        if hf is not None:
            self.tokens["hf"] = hf.strip()
        if civitai is not None:
            self.tokens["civitai"] = civitai.strip()

    def _ensure_worker(self):
        if self._worker is None or not self._worker.is_alive():
            self._worker = threading.Thread(target=self._run, daemon=True)
            self._worker.start()

    def add(self, url, folder=None, filename=None):
        j = Job(url, folder, filename)
        if not j.folder:
            j.folder, j.confidence = guess_folder_conf(j.url, j.filename or "")
        with self._lock:
            self.jobs.append(j)
        self.q.put(j)
        self._ensure_worker()
        return j

    def cancel(self, job_id):
        with self._lock:
            for j in self.jobs:
                if j.id == job_id and j.state in ("queued", "running"):
                    j._cancel.set()
                    if j._proc is not None:
                        try:
                            j._proc.terminate()
                        except Exception:
                            pass
                    if j.state == "queued":
                        j.state = "cancelled"
                        j.message = "cancelled before it started"
                    return True
        return False

    def clear_finished(self):
        with self._lock:
            self.jobs = [j for j in self.jobs if j.state in ("queued", "running")]

    def status(self):
        with self._lock:
            return {"jobs": [j.as_dict() for j in self.jobs],
                    "busy": any(j.state in ("running", "queued") for j in self.jobs),
                    "aria2": bool(ARIA2),
                    "tokens": {"hf": bool(self.tokens["hf"]),
                               "civitai": bool(self.tokens["civitai"])}}

    # ---------------------------------------------------------------- worker
    def _run(self):
        while True:
            try:
                job = self.q.get(timeout=3)
            except queue.Empty:
                return
            if job.state == "cancelled":
                continue
            try:
                self._do(job)
            except Exception as e:
                job.state = "failed"
                job.message = "%s: %s" % (type(e).__name__, str(e)[:140])

    def _target_dir(self, folder):
        name, sub = split_folder(folder)
        base, note = pick_path(name)
        return (os.path.join(base, sub) if sub else base), note

    def _do(self, job):
        job.state = "running"
        job.started = time.time()
        url = with_civitai(job.url, self.tokens)
        headers = {"User-Agent": UA}
        headers.update(auth_for(url, self.tokens))

        try:
            req = urllib.request.Request(url, headers=headers, method="HEAD")
            with urllib.request.urlopen(req, timeout=45) as r:
                job.total = int(r.headers.get("Content-Length") or 0)
                if not job.filename:
                    job.filename = filename_from(url, r.headers)
        except Exception:
            if not job.filename:
                job.filename = filename_from(url)

        if not job.folder_explicit:
            # The URL alone is often useless - a CivitAI download link has no
            # filename in it at all. Re-decide now that HEAD has told us one.
            job.folder, job.confidence = guess_folder_conf(url, job.filename)

        d, note = self._target_dir(job.folder)
        os.makedirs(d, exist_ok=True)
        job.dest = os.path.join(d, job.filename)
        if note:
            job.message = note

        good, msg = validate(job.dest, job.total)
        if good:
            job.state, job.message = "done", "already on disk (%s)" % msg
            job.done_bytes = job.total or os.path.getsize(job.dest)
            job.finished = time.time()
            return

        free = shutil.disk_usage(d).free
        if job.total and job.total > free - 2 * 1024 ** 3:
            job.state = "failed"
            job.message = ("needs %.1f GB, only %.1f GB free on %s"
                           % (job.total / 1024 ** 3, free / 1024 ** 3, d))
            return

        used_aria = False
        if ARIA2:
            used_aria = self._aria2(job, url, headers)
        if not used_aria and not job._cancel.is_set():
            job.via = "urllib"
            self._stream(job, url, headers)

        if job._cancel.is_set():
            job.state, job.message = "cancelled", "cancelled"
            for p in (job.dest, job.dest + ".part", job.dest + ".part.aria2"):
                try:
                    os.remove(p)
                except Exception:
                    pass
            return

        good, msg = validate(job.dest, job.total)
        job.finished = time.time()
        if good:
            secs = max(job.finished - job.started, 0.1)
            job.state = "done"
            job.message = "%s in %ds via %s" % (msg, int(secs), job.via or "urllib")
            self._invalidate(job.folder)
        else:
            job.state, job.message = "failed", msg
            if "incomplete" not in msg:
                try:
                    os.remove(job.dest)
                except Exception:
                    pass

    # ---------------------------------------------------------------- aria2
    def _aria2(self, job, url, headers):
        """16 connections, resumable. Progress comes from stat() on the part
        file rather than from parsing aria2's console output, which changes
        format between versions."""
        d = os.path.dirname(job.dest)
        part = os.path.basename(job.dest) + ".part"
        cmd = [ARIA2, "--console-log-level=warn", "--summary-interval=0",
               "--continue=true", "--max-connection-per-server=16", "--split=16",
               "--min-split-size=1M", "--max-tries=3", "--retry-wait=5",
               "--allow-overwrite=true", "--auto-file-renaming=false",
               "--file-allocation=none", "--user-agent=" + UA]
        for k, v in headers.items():
            if k.lower() != "user-agent":
                cmd.append("--header=%s: %s" % (k, v))
        cmd += ["-d", d, "-o", part, url]

        try:
            job._proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                         stderr=subprocess.STDOUT, text=True)
        except Exception as e:
            job.message = "aria2 would not start (%s), falling back" % str(e)[:60]
            return False

        job.via = "aria2c x16"
        full = os.path.join(d, part)
        last_t, last_b = time.time(), 0
        tail = []
        while job._proc.poll() is None:
            time.sleep(0.5)
            try:
                n = os.path.getsize(full)
            except Exception:
                n = job.done_bytes
            now = time.time()
            if now - last_t >= 0.7:
                job.speed = max((n - last_b) / (now - last_t), 0)
                last_t, last_b = now, n
            job.done_bytes = n
            if job._cancel.is_set():
                try:
                    job._proc.terminate()
                except Exception:
                    pass
                break
        try:
            out = job._proc.stdout.read() or ""
        except Exception:
            out = ""
        rc = job._proc.returncode
        job._proc = None
        tail = [x for x in out.splitlines() if x.strip()][-3:]

        if job._cancel.is_set():
            return True
        if rc == 0 and os.path.exists(full):
            try:
                os.replace(full, job.dest)
            except Exception:
                pass
            for junk in (full + ".aria2",):
                try:
                    os.remove(junk)
                except Exception:
                    pass
            return True

        job.message = "aria2 exit %s%s - retrying with urllib" % (
            rc, (": " + tail[-1][:90]) if tail else "")
        return False

    # ---------------------------------------------------------------- urllib
    def _stream(self, job, url, headers):
        part = job.dest + ".part"
        have = os.path.getsize(part) if os.path.exists(part) else 0
        h = dict(headers)
        if have:
            h["Range"] = "bytes=%d-" % have
        req = urllib.request.Request(url, headers=h)
        with urllib.request.urlopen(req, timeout=90) as r:
            if r.status == 206 and have:
                mode = "ab"
                job.done_bytes = have
                if not job.total:
                    cr = r.headers.get("Content-Range") or ""
                    m = re.search(r"/(\d+)$", cr)
                    if m:
                        job.total = int(m.group(1))
            else:
                mode, have = "wb", 0
                job.done_bytes = 0
                if not job.total:
                    job.total = int(r.headers.get("Content-Length") or 0)
            last_t, last_b = time.time(), job.done_bytes
            with open(part, mode) as f:
                while True:
                    if job._cancel.is_set():
                        break
                    buf = r.read(CHUNK)
                    if not buf:
                        break
                    f.write(buf)
                    job.done_bytes += len(buf)
                    now = time.time()
                    if now - last_t >= 0.7:
                        job.speed = (job.done_bytes - last_b) / (now - last_t)
                        last_t, last_b = now, job.done_bytes
        if not job._cancel.is_set():
            os.replace(part, job.dest)

    def _invalidate(self, folder):
        # Make the new file visible without restarting ComfyUI.
        name, _sub = split_folder(folder)
        try:
            import folder_paths
            cache = getattr(folder_paths, "filename_list_cache", None)
            if isinstance(cache, dict):
                cache.pop(name, None)
                cache.clear()
        except Exception:
            pass


DOWNLOADER = Downloader()
'''

SPEEDUP_PY = r'''
"""Make the ComfyUI UI usable over a tunnel.

ComfyUI ships `Cache-Control: no-store` on every .js and .css. That is correct
for localhost - you always want the newest frontend after an update, and the
re-download is free. Over a tunnel from a datacenter it means the browser is
forbidden from caching, so all ~12.6 MB of the app is fetched again on every
single page load.

The asset filenames are content-hashed (index-BAp94dQ0.js). A hashed name can be
cached forever: if the content changes, the name changes. So we cache those hard
and leave everything else alone.
"""
import os
import re

# Set FRIZZY_API_GZIP=1 before launching to compress API responses as well. Off
# by default: see the note in the middleware.
API_GZIP = bool(os.environ.get("FRIZZY_API_GZIP"))

# vite/rolldown style content hash: name-<hash>.js
HASHED = re.compile(r"-[A-Za-z0-9_-]{6,}\.(?:js|mjs|css)$")
YEAR = 31536000

STATS = {"immutable": 0, "revalidate": 0, "untouched": 0, "bytes_saved_est": 0}


def _rule(path, allow_immutable=True):
    """(header, tag) or (None, None) to leave the response alone."""
    p = path.split("?", 1)[0]
    low = p.lower()

    # index.html must never be pinned - it points at the hashed names.
    if p == "/" or low.endswith("/index.html"):
        return "no-cache", "revalidate"

    if low.endswith((".js", ".mjs", ".css")):
        # Only the bundler's own output is content-hashed. A custom node script
        # like comfyui-manager.js merely looks hashed, and pinning it for a year
        # would stop node updates from ever reaching the browser.
        if allow_immutable and p.startswith("/assets/") and HASHED.search(p):
            return "public, max-age=%d, immutable" % YEAR, "immutable"
        # extension scripts and user.css change without changing name:
        # allow a cheap 304 instead of a full re-download
        return "public, max-age=300, must-revalidate", "revalidate"

    # fonts and icon sets are versioned by path and never change in place
    if low.endswith((".woff", ".woff2", ".ttf", ".eot", ".otf")):
        if allow_immutable and p.startswith("/assets/"):
            return "public, max-age=%d, immutable" % YEAR, "immutable"
        return "public, max-age=86400", "revalidate"

    return None, None


def install(app, log=print, allow_immutable=True):
    """Append a middleware that overrides no-store for cacheable static assets.

    Must run while the app is still mutable. ComfyUI imports custom nodes after
    creating PromptServer and before starting the server, so import time works.

    allow_immutable=False when a REWRITTEN frontend is being served. The Safari
    rewrite keeps every filename but changes the bytes, so a year-long cache
    would pin the old build forever. Content hashing only makes a name safe to
    cache when nothing else can change what that name returns.
    """
    try:
        from aiohttp import web
    except Exception:
        return False, "aiohttp unavailable"

    @web.middleware
    async def frizzy_cache(request, handler):
        # A websocket upgrade must pass through untouched: the response is already
        # prepared by the time it comes back, so anything set on it here raises.
        if (request.headers.get("Upgrade") or "").lower() == "websocket":
            return await handler(request)
        response = await handler(request)
        try:
            if request.method not in ("GET", "HEAD"):
                return response
            if request.path.startswith("/api/"):
                # Compressing API responses here is OFF by default. It looked like
                # free speed - /api/object_info is megabytes of JSON fetched on
                # every page load - but a tunnel that re-encodes what it forwards
                # (localhost.run does) can then deliver a body the frontend cannot
                # parse. A failed /api/settings is a settings dialog that never
                # opens, with generation still working perfectly, which is a
                # miserable thing to debug. Let the tunnel do its own gzip.
                if not API_GZIP:
                    return response
                try:
                    accepts = (request.headers.get("Accept-Encoding") or "").lower()
                    body = getattr(response, "body", None)
                    if ("gzip" in accepts
                            and not response.headers.get("Content-Encoding")
                            and body is not None and len(body) > 8192):
                        response.enable_compression()
                        STATS["compressed"] = STATS.get("compressed", 0) + 1
                except Exception:
                    pass
                return response
            header, tag = _rule(request.path, allow_immutable)
            if not header:
                STATS["untouched"] += 1
                return response
            # assign, do not setdefault: ComfyUI's own middleware runs outside
            # this one and uses setdefault, so an explicit value here wins.
            response.headers["Cache-Control"] = header
            STATS[tag] += 1
        except Exception:
            pass
        return response

    try:
        app.middlewares.append(frizzy_cache)
    except Exception as e:
        return False, "middlewares already frozen: %s" % str(e)[:60]
    log("[FrizzyDownloader] asset caching enabled%s, API compression %s "
        % ("" if allow_immutable else " (revalidating: a rewritten frontend is being served)",
           "on" if API_GZIP else "off")
        + "(ComfyUI ships no-store on .js/.css, which forces a full "
        "re-download of the whole app on every page load)")
    return True, "installed"
'''

INIT_PY = r'''
"""Frizzy model downloader - paste links, pick a folder, it downloads on the VM."""
import os
import shutil
import urllib.parse
import urllib.request

from .downloader import (DOWNLOADER, registered_folders, guess_folder_conf, pick_path,
                         split_folder, filename_from, on_drive, comfy_root,
                         move_model, list_models, UA)

import asyncio
import time

WEB_DIRECTORY = "./web"

# Walking models/ takes long enough to be felt. Doing it inside an async handler
# blocks ComfyUI's only event loop, which is why saving a workflow and opening a
# panel started taking forever. Everything blocking below runs in a thread, and
# the two expensive listings are cached for a few seconds.
_CACHE = {}


async def _threaded(key, fn, ttl=0):
    now = time.time()
    if ttl:
        hit = _CACHE.get(key)
        if hit and now - hit[0] < ttl:
            return hit[1]
    try:
        value = await asyncio.to_thread(fn)
    except AttributeError:                      # python < 3.9
        loop = asyncio.get_event_loop()
        value = await loop.run_in_executor(None, fn)
    if ttl:
        _CACHE[key] = (now, value)
    return value
NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS", "WEB_DIRECTORY"]

try:
    from aiohttp import web
    from server import PromptServer
    routes = PromptServer.instance.routes
except Exception:                       # imported outside ComfyUI, e.g. in tests
    routes = None

# ComfyUI sends Cache-Control: no-store on every .js and .css. On localhost that
# costs nothing. Over a tunnel it forces a fresh download of the whole ~12.6 MB
# app on every page load, which is most of why the UI feels slow remotely.
try:
    from . import speedup
    if routes is not None:
        # A rewritten frontend reuses the original filenames, so immutable
        # caching would serve its predecessor forever.
        rewritten = bool(os.environ.get("FRIZZY_FRONTEND_REWRITTEN"))
        speedup.install(PromptServer.instance.app, allow_immutable=not rewritten)
except Exception as _e:
    print("[FrizzyDownloader] could not enable asset caching: %s" % _e)


def _existing_anywhere(filename):
    """Same filename already under models/, wherever it is. Cheap duplicate check."""
    root = os.path.join(comfy_root(), "models")
    hits = []
    if not filename or not os.path.isdir(root):
        return hits
    for dirpath, _dirnames, files in os.walk(root):
        if filename in files:
            full = os.path.join(dirpath, filename)
            try:
                hits.append({"path": full, "size": os.path.getsize(full)})
            except Exception:
                pass
        if len(hits) > 12:
            break
    return hits


def _remote_size(url):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": UA}, method="HEAD")
        hf = DOWNLOADER.tokens.get("hf")
        if hf and ("huggingface.co" in url or "hf.co" in url):
            req.add_header("Authorization", "Bearer " + hf)
        with urllib.request.urlopen(req, timeout=25) as r:
            return int(r.headers.get("Content-Length") or 0), filename_from(url, r.headers)
    except Exception:
        return 0, filename_from(url)


def _register():
    @routes.get("/frizzy/dl/folders")
    async def _folders(request):
        folders = await _threaded("folders", registered_folders, ttl=20)
        return web.json_response({"folders": folders,
                                  "root": comfy_root(),
                                  "aria2": bool(DOWNLOADER.status().get("aria2"))})

    @routes.post("/frizzy/dl/probe")
    async def _probe(request):
        """What would happen if I queued these - before anything is downloaded."""
        body = await request.json()
        return await _threaded("probe", lambda: _probe_sync(body.get("items") or []))

    def _probe_sync(items):
        out, total = [], 0
        for it in items:
            url = (it.get("url") or "").strip()
            if not url:
                continue
            size, guessed_name = _remote_size(url)
            name = (it.get("filename") or "").strip() or guessed_name
            folder = (it.get("folder") or "").strip()
            if folder:
                conf = "high"
            else:
                folder, conf = guess_folder_conf(url, name)
            base, note = pick_path(split_folder(folder)[0])
            sub = split_folder(folder)[1]
            dest = os.path.join(base, sub, name) if sub else os.path.join(base, name)
            total += size
            out.append({"url": url, "filename": name, "folder": folder, "dest": dest,
                        "size": size, "drive": on_drive(dest), "note": note,
                        "confidence": conf, "exists": os.path.exists(dest),
                        "duplicates": _existing_anywhere(name)})
        try:
            free = shutil.disk_usage(os.path.join(comfy_root(), "models")).free
        except Exception:
            free = 0
        return web.json_response({"items": out, "total": total, "free": free})

    _register_rest()


def _register_rest():

    @routes.post("/frizzy/dl/queue")
    async def _queue(request):
        body = await request.json()
        items = body.get("items") or []
        if isinstance(items, str):
            items = [{"url": u} for u in items.split() if u.strip()]
        added = []
        for it in items:
            url = (it.get("url") or "").strip()
            if not url:
                continue
            j = DOWNLOADER.add(url, it.get("folder") or None, it.get("filename") or None)
            added.append(j.as_dict())
        return web.json_response({"added": added})

    @routes.get("/frizzy/dl/status")
    async def _status(request):
        return web.json_response(DOWNLOADER.status())

    @routes.post("/frizzy/dl/cancel")
    async def _cancel(request):
        body = await request.json()
        return web.json_response({"ok": DOWNLOADER.cancel(int(body.get("id", 0)))})

    @routes.post("/frizzy/dl/clear")
    async def _clear(request):
        DOWNLOADER.clear_finished()
        return web.json_response({"ok": True})

    @routes.post("/frizzy/dl/tokens")
    async def _tokens(request):
        body = await request.json()
        DOWNLOADER.set_tokens(body.get("hf"), body.get("civitai"))
        return web.json_response({"ok": True})

    @routes.get("/frizzy/dl/files")
    async def _files(request):
        files = await _threaded("files", list_models, ttl=10)
        return web.json_response({"files": files,
                                  "root": os.path.join(comfy_root(), "models")})

    @routes.post("/frizzy/dl/move")
    async def _move(request):
        """Change where a file already on disk lives. The guess is wrong often
        enough that fixing it afterwards has to be one click, not a shell."""
        body = await request.json()
        src = (body.get("src") or "").strip()
        job_id = body.get("id")
        if not src and job_id:
            for j in DOWNLOADER.jobs:
                if j.id == int(job_id):
                    src = j.dest
                    break
        try:
            dest, note = move_model(src, body.get("folder") or "",
                                    (body.get("filename") or "").strip() or None)
        except Exception as e:
            return web.json_response({"ok": False, "error": str(e)[:200]}, status=400)
        _CACHE.clear()
        if job_id:
            for j in DOWNLOADER.jobs:
                if j.id == int(job_id):
                    j.dest = dest
                    j.folder = body.get("folder") or j.folder
                    j.message = "moved to " + (body.get("folder") or "")
        return web.json_response({"ok": True, "dest": dest, "note": note})

    @routes.post("/frizzy/dl/guess")
    async def _guess(request):
        body = await request.json()
        urls = body.get("urls") or []
        guesses = []
        for u in urls:
            folder, conf = guess_folder_conf(
                u, os.path.basename(urllib.parse.urlparse(u).path))
            guesses.append({"url": u, "folder": folder, "confidence": conf})
        return web.json_response({"guesses": guesses})


if routes is not None:
    _register()
    print("[FrizzyDownloader] model downloader ready (aria2: %s)"
          % ("yes" if DOWNLOADER.status().get("aria2") else "no"))
'''

WEB_JS = r'''
import { app } from "../../scripts/app.js";

const API = {
  async folders() { return await (await fetch("/frizzy/dl/folders")).json(); },
  async status()  { return await (await fetch("/frizzy/dl/status")).json(); },
  async files()   { return await (await fetch("/frizzy/dl/files")).json(); },
  async post(path, body) {
    const r = await fetch(path, {method:"POST", headers:{"Content-Type":"application/json"},
      body: JSON.stringify(body || {})});
    return await r.json();
  },
  probe(items) { return API.post("/frizzy/dl/probe", {items}); },
  queue(items) { return API.post("/frizzy/dl/queue", {items}); },
  cancel(id)   { return API.post("/frizzy/dl/cancel", {id}); },
  clear()      { return API.post("/frizzy/dl/clear", {}); },
  move(body)   { return API.post("/frizzy/dl/move", body); },
  tokens(hf, civitai) { return API.post("/frizzy/dl/tokens", {hf, civitai}); },
};

const gb = (n) => n >= 1073741824 ? (n / 1073741824).toFixed(2) + " GB"
                : n >= 1048576 ? (n / 1048576).toFixed(0) + " MB"
                : (n / 1024).toFixed(0) + " KB";

const CSS = `
.fzdl-btn{position:fixed;right:16px;bottom:16px;z-index:1200;background:#2b6cb0;color:#fff;
  border:0;border-radius:8px;padding:10px 14px;font:13px system-ui;cursor:pointer;
  box-shadow:0 2px 10px rgba(0,0,0,.4)}
.fzdl-wrap{position:fixed;inset:0;z-index:1300;background:rgba(0,0,0,.55);display:flex;
  align-items:center;justify-content:center}
.fzdl{background:#1e1e1e;color:#e8e8e8;border:1px solid #3a3a3a;border-radius:10px;
  width:min(1000px,96vw);max-height:90vh;overflow:auto;padding:18px;font:13px/1.5 system-ui}
.fzdl h2{margin:0 0 2px;font-size:16px}
.fzdl h3{margin:18px 0 4px;font-size:13px;color:#bbb;font-weight:600}
.fzdl .sub{color:#999;margin-bottom:14px}
.fzdl textarea{width:100%;min-height:84px;background:#151515;color:#eee;border:1px solid #3a3a3a;
  border-radius:6px;padding:9px;font:12px ui-monospace,monospace;box-sizing:border-box}
.fzdl select,.fzdl input{background:#151515;color:#eee;border:1px solid #3a3a3a;border-radius:6px;
  padding:6px;font:12px system-ui;max-width:100%}
.fzdl button{background:#2b6cb0;color:#fff;border:0;border-radius:6px;padding:8px 13px;
  font:13px system-ui;cursor:pointer;margin-right:7px}
.fzdl button.ghost{background:#333}
.fzdl button.small{padding:4px 9px;font-size:12px;margin:0}
.fzdl .row{display:flex;gap:9px;align-items:center;flex-wrap:wrap;margin:11px 0}
.fzdl table{width:100%;border-collapse:collapse;margin-top:8px;font-size:12px}
.fzdl th,.fzdl td{text-align:left;padding:6px 7px;border-bottom:1px solid #2f2f2f;
  vertical-align:top}
.fzdl th{color:#999;font-weight:500}
.fzdl .bar{height:5px;background:#333;border-radius:3px;overflow:hidden;margin-top:4px}
.fzdl .bar i{display:block;height:100%;background:#4299e1}
.fzdl .done{color:#68d391}.fzdl .failed{color:#fc8181}.fzdl .running{color:#f6ad55}
.fzdl .muted{color:#888}.fzdl .flag{color:#f6ad55}
.fzdl select.low{border-color:#a06520;background:#241a0d}
`;

function el(tag, attrs = {}, ...kids) {
  const n = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs)) {
    if (k === "class") n.className = v;
    else if (k === "style") n.style.cssText = v;
    else if (k.startsWith("on")) n.addEventListener(k.slice(2), v);
    else if (v !== null && v !== undefined) n.setAttribute(k, v);
  }
  for (const c of kids) n.append(c?.nodeType ? c : document.createTextNode(c ?? ""));
  return n;
}

let panel = null, timer = null, meta = null, staged = [];

function toast(msg, isError) {
  const n = el("div", {}, msg);
  n.style.cssText = "position:fixed;right:16px;bottom:70px;z-index:1400;background:" +
    (isError ? "#7f1d1d" : "#1e3a5f") + ";color:#fff;padding:10px 14px;border-radius:8px;" +
    "font:13px system-ui;box-shadow:0 2px 10px rgba(0,0,0,.4);max-width:min(460px,80vw)";
  document.body.append(n);
  setTimeout(() => n.remove(), 6000);
}

function folderSelect(current, onchange, cls) {
  const s = el("select", {class: cls || ""});
  const names = (meta?.folders || []).map(f => f.name);
  const base = (current || "").split("/")[0];
  if (base && !names.includes(base)) names.unshift(base);
  for (const n of names) {
    const o = el("option", {value: n}, n);
    if (n === base) o.selected = true;
    s.append(o);
  }
  if (onchange) s.addEventListener("change", () => onchange(s.value));
  return s;
}

/* -------------------------------------------------------------- interception
 * ComfyUI's missing-models dialog hands the URL to the browser: it renders an
 * anchor and lets you click it. On a local install that is correct, because the
 * browser's download folder IS the ComfyUI machine. Here it is a laptop, and
 * the model ends up nowhere near the GPU that needs it.
 *
 * A real mouse click never goes through HTMLAnchorElement.prototype.click, so
 * patching that alone misses every actual click. The capture-phase listener is
 * the one that fires.
 */
const MODEL_EXT = /\.(safetensors|gguf|ckpt|pt|pth|bin|sft|onnx)(\?|$)/i;
const MODEL_HOST = /(^|\.)(huggingface\.co|hf\.co|civitai\.com)$/i;
let intercepting = true;

export function shouldIntercept(href, hasDownloadAttr, pageOrigin) {
  if (!href) return false;
  let u;
  try { u = new URL(href, pageOrigin); } catch (e) { return false; }
  if (!/^https?:$/.test(u.protocol)) return false;
  // generated outputs are served by ComfyUI itself - leave those alone
  if (u.origin === pageOrigin && /^\/(api\/)?(view|viewvideo)\b/.test(u.pathname)) return false;
  if (MODEL_EXT.test(u.pathname)) return true;
  // a bare repo link is a page you want to READ, so only take it when the
  // dialog has explicitly marked it as a download
  return hasDownloadAttr && MODEL_HOST.test(u.hostname);
}

function queueFromBrowser(url, name) {
  API.queue([{ url, filename: name || null }]).then((r) => {
    const j = (r.added || [])[0];
    toast("Downloading on the VM: " + (j?.filename || name || url.split("/").pop()) +
          (j ? "  ->  " + j.folder + (j.confidence === "low" ? "  (guessed - check it)" : "") : ""));
    if (!panel) openPanel();
  }).catch(() => toast("Could not queue that on the server", true));
}

function installIntercept() {
  if (window.__frizzyIntercept) return;
  window.__frizzyIntercept = true;

  document.addEventListener("click", (ev) => {
    if (!intercepting) return;
    const a = ev.target?.closest?.("a[href]");
    if (!a) return;
    if (!shouldIntercept(a.href, a.hasAttribute("download"), location.origin)) return;
    ev.preventDefault();
    ev.stopPropagation();
    queueFromBrowser(a.href, a.getAttribute("download") || "");
  }, true);

  const anchorClick = HTMLAnchorElement.prototype.click;
  HTMLAnchorElement.prototype.click = function () {
    try {
      if (intercepting && shouldIntercept(this.href, this.hasAttribute("download"),
                                          location.origin)) {
        queueFromBrowser(this.href, this.getAttribute("download") || "");
        return;
      }
    } catch (e) { /* never block a real download because of a bug here */ }
    return anchorClick.apply(this, arguments);
  };

  const nativeOpen = window.open;
  window.open = function (url, ...rest) {
    try {
      if (intercepting && typeof url === "string" &&
          shouldIntercept(url, false, location.origin)) {
        queueFromBrowser(url, "");
        return null;
      }
    } catch (e) {}
    return nativeOpen.call(window, url, ...rest);
  };
}

/* ------------------------------------------------------------------ staging */
function renderStaging(host, tbody) {
  host.replaceChildren();
  if (!staged.length) return;

  const rows = el("tbody");
  staged.forEach((it, i) => {
    const sel = folderSelect(it.base, (v) => {
      it.base = v;
      it.folder = v + (it.sub ? "/" + it.sub : "");
    }, it.confidence === "low" ? "low" : "");
    const sub = el("input", {placeholder:"subfolder", size:"12", value: it.sub || ""});
    sub.addEventListener("input", () => {
      it.sub = sub.value.trim().replace(/^\/+|\/+$/g, "");
      it.folder = it.base + (it.sub ? "/" + it.sub : "");
    });
    const nm = el("input", {size:"24", value: it.filename || ""});
    nm.addEventListener("input", () => { it.filename = nm.value.trim(); });

    const flags = [];
    if (it.confidence === "low") flags.push("only the extension said so - check this");
    else if (it.confidence === "medium") flags.push("from the filename");
    if (it.exists) flags.push("already at that path, will be skipped");
    else if (it.duplicates?.length) flags.push("same name in " + it.duplicates.map(d =>
      d.path.replace(/.*\/models\//, "")).join(", "));
    if (it.drive) flags.push("that path is on Drive, not the VM disk");

    rows.append(el("tr", {},
      el("td", {}, nm, el("div", {class:"muted"}, it.size ? gb(it.size) : "size unknown")),
      el("td", {}, sel, sub),
      el("td", {class: it.confidence === "low" ? "flag" : "muted"}, flags.join("; ")),
      el("td", {}, el("button", {class:"ghost small", onclick: () => {
        staged.splice(i, 1); renderStaging(host, tbody);
      }}, "drop"))));
  });

  const total = staged.reduce((a, b) => a + (b.size || 0), 0);
  host.append(
    el("h3", {}, "Staged - fix the folder before it starts"),
    el("table", {}, el("thead", {}, el("tr", {},
      el("th", {}, "file"), el("th", {}, "folder"), el("th", {}, "note"), el("th", {}, ""))), rows),
    el("div", {class:"row"},
      el("button", {onclick: async () => {
        const items = staged.map(it => ({url: it.url, folder: it.folder || it.base,
                                         filename: it.filename || null}));
        staged = [];
        renderStaging(host, tbody);
        await API.queue(items);
        refreshTable(tbody);
      }}, "Start " + staged.length + " download(s)"),
      el("button", {class:"ghost", onclick: () => { staged = []; renderStaging(host, tbody); }},
        "Clear"),
      el("span", {class:"muted"}, "total " + gb(total))));
}

/* -------------------------------------------------------------------- queue */
async function refreshTable(tbody) {
  let s;
  try { s = await API.status(); } catch (e) { return; }
  tbody.replaceChildren();
  if (!s.jobs.length) {
    tbody.append(el("tr", {}, el("td", {class:"muted", colspan:"5"}, "nothing queued")));
    return;
  }
  let active = false;
  for (const j of s.jobs) {
    if (j.state === "running" || j.state === "queued") active = true;
    const name = el("td", {}, el("div", {}, j.filename || j.url.slice(0, 60)),
                    el("div", {class:"muted"}, j.dest || j.url.slice(0, 80)));

    // a finished file can still be in the wrong place: move it from here
    const where = el("td", {});
    if (j.state === "done" && j.dest) {
      where.append(folderSelect(j.folder, async (v) => {
        const r = await API.move({id: j.id, folder: v});
        toast(r.ok ? "Moved to " + v : "Move failed: " + (r.error || ""), !r.ok);
        try { await app.refreshComboInNodes(); } catch (e) {}
        refreshTable(tbody);
      }, j.confidence === "low" ? "low" : ""));
    } else {
      where.append(el("div", {}, j.folder || ""));
      if (j.confidence === "low") where.append(el("div", {class:"flag"}, "guessed"));
    }

    const bar = el("div", {class:"bar"}, el("i", {style:`width:${j.pct}%`}));
    const prog = el("td", {},
      j.state === "running"
        ? el("div", {}, `${j.pct}%  ${gb(j.done)} / ${j.total ? gb(j.total) : "?"}  ` +
                        `${j.speed} MB/s  ${j.via || ""}`)
        : el("div", {class:j.state}, j.state),
      j.state === "running" ? bar : "");
    const act = el("td", {});
    if (j.state === "queued" || j.state === "running")
      act.append(el("button", {class:"ghost small", onclick: () => API.cancel(j.id)}, "cancel"));
    tbody.append(el("tr", {}, name, where, prog,
                    el("td", {class:"muted"}, j.message || ""), act));
  }
  if (!active && timer) {
    try { await app.refreshComboInNodes(); } catch (e) {}
  }
}

/* ----------------------------------------------------------- files on disk */
async function renderFiles(host, filter) {
  host.replaceChildren(el("div", {class:"muted"},
    "reading models/ ... (first time in a session takes a moment)"));
  let data;
  try {
    data = await API.files();
  } catch (e) {
    host.replaceChildren(el("div", {class:"muted"}, "could not read the model folder"));
    return;
  }
  const q = (filter || "").toLowerCase();
  const list = data.files.filter(f => !q || f.name.toLowerCase().includes(q) ||
                                      f.folder.toLowerCase().includes(q)).slice(0, 60);
  const rows = el("tbody");
  for (const f of list) {
    rows.append(el("tr", {},
      el("td", {}, f.name, el("div", {class:"muted"}, gb(f.size))),
      el("td", {}, folderSelect(f.folder, async (v) => {
        const r = await API.move({src: f.path, folder: v});
        toast(r.ok ? f.name + " -> " + v : "Move failed: " + (r.error || ""), !r.ok);
        try { await app.refreshComboInNodes(); } catch (e) {}
        renderFiles(host, filter);
      })),
      el("td", {class:"muted"}, f.folder || "(models root)")));
  }
  host.replaceChildren(el("table", {}, el("thead", {}, el("tr", {},
    el("th", {}, "file"), el("th", {}, "move to"), el("th", {}, "currently"))), rows));
  if (!list.length) host.append(el("div", {class:"muted"}, "nothing matches"));
}

/* ------------------------------------------------------------------- panel */
async function openPanel() {
  if (panel) { panel.remove(); panel = null; clearInterval(timer); timer = null; return; }
  // Do NOT await anything before the panel is in the DOM. Listing the model
  // folders means walking them, and a panel that renders after that walk looks
  // like a button that does nothing.
  if (!meta) meta = {folders: [], aria2: true, pending: true};

  const ta = el("textarea", {placeholder:
    "One model URL per line.\nHuggingFace resolve or blob links, CivitAI download links, direct URLs."});
  const defSel = el("select", {});
  defSel.append(el("option", {value:""}, "work the folder out per link"));
  for (const f of meta.folders || [])
    defSel.append(el("option", {value:f.name},
      `${f.name}  (${f.files} files, ${gb(f.free)} free${f.drive ? ", ON DRIVE" : ""})`));
  const hf = el("input", {type:"password", placeholder:"HF token (gated repos)", size:"22"});
  const cv = el("input", {type:"password", placeholder:"CivitAI token", size:"18"});
  const stageHost = el("div", {});
  const tbody = el("tbody");
  const filesHost = el("div", {});
  const filter = el("input", {placeholder:"filter by name or folder", size:"24"});
  filter.addEventListener("input", () => renderFiles(filesHost, filter.value));

  const dl = el("button", {onclick: async () => {
    const urls = ta.value.split("\n").map(s => s.trim()).filter(Boolean);
    if (!urls.length) return;
    if (meta.pending) {                       // opened before the list arrived
      try { meta = await API.folders(); } catch (e) { meta = {folders: []}; }
    }
    if (hf.value || cv.value) await API.tokens(hf.value, cv.value);
    stageHost.replaceChildren(el("div", {class:"muted"},
      "checking " + urls.length + " link(s)..."));
    let r;
    try {
      r = await API.probe(urls.map(u => ({url:u, folder: defSel.value || null})));
    } catch (e) {
      stageHost.replaceChildren(el("div", {class:"flag"}, "check failed"));
      return;
    }
    staged = r.items.map(it => Object.assign(it, {
      base: (it.folder || "").split("/")[0],
      sub: (it.folder || "").split("/").slice(1).join("/")}));
    ta.value = "";
    renderStaging(stageHost, tbody);
    if (r.total > r.free)
      toast("That is " + gb(r.total) + " and only " + gb(r.free) + " is free", true);
  }}, "Download");

  const body = el("div", {class:"fzdl"},
    el("h2", {}, "Models"),
    el("div", {class:"sub"},
      "Everything downloads on the machine ComfyUI runs on" +
      (meta.aria2 ? ", with aria2 on 16 connections. " : ". ") +
      "Nothing starts until you have seen where it is going."),
    ta,
    el("div", {class:"row"}, "Folder:", defSel, hf, cv),
    el("div", {class:"row"}, dl,
      el("button", {class:"ghost", onclick: async () => { await API.clear(); refreshTable(tbody); }},
        "Clear finished"),
      el("button", {class:"ghost", onclick: async () => { await app.refreshComboInNodes(); }},
        "Refresh model lists"),
      el("button", {class:"ghost", onclick: () => openPanel()}, "Close")),
    stageHost,
    el("h3", {}, "Queue"),
    el("table", {}, el("thead", {}, el("tr", {},
        el("th", {}, "file"), el("th", {}, "folder"), el("th", {}, "progress"),
        el("th", {}, "note"), el("th", {}, ""))), tbody),
    el("h3", {}, "Already on disk, newest first - change any of these too"),
    el("div", {class:"row"}, filter,
      el("button", {class:"ghost small", onclick: () => renderFiles(filesHost, filter.value)},
        "reload")),
    filesHost);

  panel = el("div", {class:"fzdl-wrap",
                     onclick: (e) => { if (e.target === panel) openPanel(); }}, body);
  document.body.append(panel);
  refreshTable(tbody);
  timer = setInterval(() => refreshTable(tbody), 1000);

  if (meta.pending) {
    try {
      meta = await API.folders();
    } catch (e) {
      meta = {folders: []};
      defSel.append(el("option", {value:""}, "could not read the folder list"));
    }
    for (const f of meta.folders || [])
      defSel.append(el("option", {value:f.name},
        `${f.name}  (${f.files} files, ${gb(f.free)} free${f.drive ? ", ON DRIVE" : ""})`));
  }
  renderFiles(filesHost, "");
}

app.registerExtension({
  name: "frizzy.ModelDownloader",
  async setup() {
    document.head.append(el("style", {}, CSS));
    installIntercept();
    try {
      if (app.extensionManager?.registerSidebarTab) {
        app.extensionManager.registerSidebarTab({
          id: "frizzy-downloader", icon: "pi pi-download", title: "Models",
          tooltip: "Download models onto the VM", type: "custom",
          render: (e) => { e.append(el("div", {style:"padding:12px"},
            el("button", {class:"fzdl-btn", style:"position:static", onclick: openPanel},
              "Open model downloader"))); },
        });
      }
    } catch (e) {}
    document.body.append(el("button", {class:"fzdl-btn", onclick: openPanel}, "Models"));
    // escape hatch if you genuinely want a file on your own machine
    window.frizzyDownloadIntercept = (on) => {
      intercepting = on !== false;
      toast("Server-side model downloads " + (intercepting ? "on" : "off"));
    };
  },
});
'''

import os, time, shutil, subprocess
from pathlib import Path

rows = []
if not ROOT.exists():
    raise SystemExit("Run cell 1 first.")
rule("LAUNCH")

step("What is installed")
inv = inventory()
if inv["models"]:
    for folder, (n, b) in inv["models"].items():
        info("%-18s %d file(s), %s" % (folder + "/", n, human(b)))
    ok("%s of models, %d workflow(s), %d custom node(s)"
       % (human(inv["model_bytes"]), inv["workflows"], inv["custom_nodes"]))
else:
    warn("no model files found under models/ - ComfyUI will start but cannot generate")
    info("drag a workflow JSON onto the canvas, then use the Models button")
used, left = session_age()
info("session: %.1fh used, about %.1fh left before Colab cuts it" % (used, left))
if left < 1:
    warn("under an hour left - save with cell 3 before starting anything long")

step("Clearing anything already running")
kill_all(PORT)

# a Drive-symlinked user/ makes /api/userdata hang, which hangs the whole UI
u = ROOT / "user"
if u.is_symlink():
    warn("user/ is a symlink -> %s" % os.readlink(u))
    info("that stalls /api/userdata and the UI never finishes loading")
    tmp = Path("/content/_user_local")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    sh('rsync -a "%s/" "%s/"' % (u, tmp), check=False)
    u.unlink()
    shutil.move(str(tmp), str(u))
    ok("converted to a real local directory")

import json, re

# ---------------------------------------------------------------- speed layer
# The Models panel and the cache layer are not optional any more: every session
# wants them, and forgetting to install them is what makes the UI feel broken.
FRIZZY_EXT = {"downloader.py": DOWNLOADER_PY,
              "speedup.py": SPEEDUP_PY,
              "__init__.py": INIT_PY,
              "web/frizzyDownloader.js": WEB_JS}


def install_frizzy_extension(root=None, quiet=False):
    """Write the custom node that downloads models ON THIS VM and fixes caching.
    Returns (path, files_changed). Rewrites only what actually differs, so a
    rerun costs nothing."""
    import ast as _ast
    node = Path(root or ROOT) / "custom_nodes" / "ComfyUI-FrizzyDownloader"
    (node / "web").mkdir(parents=True, exist_ok=True)
    changed = 0
    for name, body in FRIZZY_EXT.items():
        f = node / name
        old = f.read_text(encoding="utf8") if f.exists() else None
        if old != body:
            f.write_text(body, encoding="utf8")
            changed += 1
    for name in ("downloader.py", "speedup.py", "__init__.py"):
        try:
            _ast.parse((node / name).read_text(encoding="utf8"))
        except SyntaxError as e:
            raise SystemExit("%s did not survive being written out: line %s: %s"
                             % (name, e.lineno, e.msg))
    # cell 4 repairs this without needing the payload again
    try:
        Path("/content/frizzy_ext.json").write_text(json.dumps(FRIZZY_EXT), encoding="utf8")
    except Exception:
        pass
    return node, changed


step("Model downloader and cache layer")
_node, _changed = install_frizzy_extension()
ok("%s (%d file(s) written)" % (_node.name, _changed))
info("models you add from inside ComfyUI download here, on the VM, with aria2")
info("also cancels ComfyUI's Cache-Control: no-store, so the 12.6 MB app is")
info("fetched once instead of on every page load")
for _k in ("HF_TOKEN", "CIVITAI_TOKEN"):
    _v = secret(_k)
    if _v:
        os.environ[_k] = _v
        info("%s passed through to the downloader" % _k)
rows.append(("OK", "downloader", "installed, downloads land on the VM"))


# ---------------------------------------------------------------- browser
# Which frontend to serve is a question about YOUR browser, so ask it. Colab
# output runs in that browser, so eval_js is a direct capability probe - not a
# user-agent guess, which is wrong for every Safari-engine browser on iOS.
def probe_browser(timeout=12):
    try:
        from google.colab.output import eval_js
    except Exception:
        return None
    js = ("(function(){var r={ua:navigator.userAgent,lb:0,sb:0,vf:0};"
          "try{new RegExp('(?<=a)b');r.lb=1}catch(e){}"
          "try{new Function('class X{static{}}');r.sb=1}catch(e){}"
          "try{new RegExp('[\\\\p{ASCII}]','v');r.vf=1}catch(e){}"
          "return JSON.stringify(r);})()")
    try:
        raw = eval_js(js, timeout_sec=timeout)
        return json.loads(raw) if raw else None
    except Exception:
        return None


def browser_name(ua):
    if re.search(r"Edg/", ua):
        return "Edge"
    if re.search(r"Firefox|FxiOS", ua):
        return "Firefox"
    if re.search(r"CriOS", ua):
        return "Chrome on iOS (Safari engine)"
    if re.search(r"Chrome|Chromium", ua):
        return "Chrome"
    if re.search(r"Safari", ua):
        return "Safari"
    return "unknown"


step("Checking what your browser can compile")
SAFARI_TARGET = (os.environ.get("FRIZZY_COMPAT") or "").strip() or None
_probe = probe_browser()
if _probe is None:
    info("could not reach the browser from here - assuming a modern engine")
    info('force it with  os.environ["FRIZZY_COMPAT"] = "safari15"  and rerun')
else:
    _ua = _probe.get("ua", "")
    _v = re.search(r"Version/(\d+)", _ua)
    info("%s%s" % (browser_name(_ua), ("  " + _v.group(0)) if _v else ""))
    _missing = [n for n, k in (("regex lookbehind", "lb"), ("class static blocks", "sb"),
                               ("regex v flag", "vf")) if not _probe.get(k)]
    if not _missing:
        ok("it can compile the stock frontend - no rewrite needed")
    elif SAFARI_TARGET:
        warn("cannot compile: %s" % ", ".join(_missing))
        info("using the target you forced: %s" % SAFARI_TARGET)
    else:
        warn("cannot compile: %s" % ", ".join(_missing))
        if not _probe.get("lb") or not _probe.get("sb"):
            SAFARI_TARGET = "safari14" if (_v and int(_v.group(1)) < 15) else "safari15"
        else:
            SAFARI_TARGET = "safari16"
        info("rewriting the frontend for %s automatically" % SAFARI_TARGET)


# ---------------------------------------------------------------- flags
FRONT_ROOT = None
if SAFARI_TARGET:
    step("Rewriting the frontend so your browser can compile it")
    cached = Path("/content/comfyui_frontend_compat")
    marker = cached / ".target"
    if cached.is_dir() and marker.is_file() and marker.read_text().strip() == SAFARI_TARGET:
        ok("reusing the copy built earlier this session (%s)" % SAFARI_TARGET)
        FRONT_ROOT = cached
    else:
        FRONT_ROOT, st = build_safari_frontend(target=SAFARI_TARGET)
        if FRONT_ROOT is None:
            fail(st.get("error", "could not build it"))
            warn("carrying on with the stock frontend - Safari will still be blank")
        else:
            ok("%d files rewritten for %s" % (st["transpiled"], SAFARI_TARGET))
            if st["failed"]:
                warn("%d file(s) could not be rewritten: %s"
                     % (len(st["failed"]), ", ".join(st["failed"][:3])))
            for r in st["regex_fixed"][:6]:
                info("lookbehind removed  %s" % r)
            left = st.get("lookbehind_left") or []
            if left:
                warn("lookbehind still present in: %s" % ", ".join(left[:4]))
            marker.write_text(SAFARI_TARGET)
    if FRONT_ROOT:
        # the rewrite keeps every filename, so tell the cache layer not to pin them
        os.environ["FRIZZY_FRONTEND_REWRITTEN"] = "1"
        info("assets will be revalidated rather than pinned, because the rewrite")
        info("reuses the original filenames. Hard-reload once after switching.")
    else:
        os.environ.pop("FRIZZY_FRONTEND_REWRITTEN", None)
    rows.append(("OK" if FRONT_ROOT else "WARN", "safari compat",
                 SAFARI_TARGET if FRONT_ROOT else "failed, using stock frontend"))

# A page you can just open. Safari hides its inspector behind a settings toggle,
# and nobody should have to enable developer tools to read one error message.
CHECK_HTML = r'''
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ComfyUI browser check</title>
<style>
  :root { color-scheme: dark; }
  body { margin:0; padding:18px; background:#141414; color:#e8e8e8;
         font:14px/1.5 ui-monospace,SFMono-Regular,Menlo,monospace; }
  h1 { font-size:17px; margin:0 0 4px; }
  .sub { color:#888; margin-bottom:14px; font-size:13px; }
  #bar { position:sticky; top:0; background:#141414; padding-bottom:10px; }
  button { background:#2b6cb0; color:#fff; border:0; padding:9px 15px;
           border-radius:6px; font:inherit; cursor:pointer; margin-right:8px; }
  button:disabled { background:#444; cursor:default; }
  pre { white-space:pre-wrap; word-break:break-word; background:#1d1d1d;
        border:1px solid #333; border-radius:6px; padding:12px; margin:0; }
  .ok{color:#68d391} .bad{color:#fc8181} .warn{color:#f6ad55} .dim{color:#888}
</style>
</head>
<body>
<h1>ComfyUI browser check</h1>
<div class="sub">Runs in this browser, against this server. No developer tools needed.</div>
<div id="bar">
  <button id="run">Run the check</button>
  <button id="copy" disabled>Copy result</button>
  <span id="status" class="dim"></span>
</div>
<pre id="out">Press "Run the check".</pre>
<script>
const OUT = document.getElementById('out');
const STATUS = document.getElementById('status');
let LINES = [];
function say(s, cls) {
  LINES.push(s);
  const span = document.createElement('span');
  span.textContent = s + "\n";
  if (cls) span.className = cls;
  OUT.appendChild(span);
}
function head(s) { say(""); say("=== " + s + " ==="); }

async function run() {
  OUT.textContent = ""; LINES = [];
  document.getElementById('run').disabled = true;
  const t0 = performance.now();
  const origin = location.origin;

  say("=== comfyui-browser-check/3 ===");
  say("server     " + origin);
  say("agent      " + navigator.userAgent);
  say("time       " + new Date().toISOString());
  const ua = navigator.userAgent;
  const isSafari = /^((?!chrome|android|crios|fxios|edg).)*safari/i.test(ua);
  const vm = ua.match(/Version\/(\d+)\.(\d+)/);
  say("browser    " + (isSafari ? "Safari" : /Firefox/.test(ua) ? "Firefox" :
      /Edg\//.test(ua) ? "Edge" : /Chrome/.test(ua) ? "Chrome" : "other") +
      (vm ? " " + vm[1] + "." + vm[2] : ""));
  say("cookies    " + navigator.cookieEnabled + "   private-ish: " +
      (await isProbablyPrivate()));

  head("can this engine compile the bundle");
  const feat = (name, need, fn) => {
    let ok = false, err = "";
    try { ok = !!fn(); } catch (e) { err = e.name + ": " + e.message.slice(0, 70); }
    say((ok ? "  OK   " : "  FAIL ") + name.padEnd(24) + "needs " + need +
        (err ? "  " + err : ""), ok ? "ok" : "bad");
    return ok;
  };
  const hard = [];
  if (!feat("regex lookbehind", "Safari 16.4+", () => new RegExp("(?<=a)b"))) hard.push("lookbehind");
  if (!feat("class static block", "Safari 16.4+", () => new Function("class X{static{}}"))) hard.push("static blocks");
  if (!feat("regex v flag", "Safari 17+", () => new RegExp("[\\p{ASCII}]", "v"))) hard.push("v flag");
  feat("Array.toSorted", "Safari 16.4+", () => typeof [].toSorted === "function");
  feat("structuredClone", "Safari 15.4+", () => typeof structuredClone === "function");
  feat("Object.groupBy", "Safari 17.4+", () => typeof Object.groupBy === "function");
  feat("localStorage", "any", () => { localStorage.setItem("_t","1"); localStorage.removeItem("_t"); return true; });
  feat("indexedDB", "any", () => !!window.indexedDB);
  feat("WebSocket", "any", () => !!window.WebSocket);
  if (hard.length) say("  >>> this engine cannot compile: " + hard.join(", ") +
                       " - rerun cell 2, it rebuilds the frontend for you", "bad");

  head("app javascript");
  let deps = [], entry = null;
  try {
    const html = await (await fetch("/", {cache:"no-store"})).text();
    const m = html.match(/src="([^"]*assets\/index-[^"]+\.js)"/);
    entry = m ? m[1] : null;
    say("  entry      " + entry);
    if (entry) {
      const js = await (await fetch(entry, {cache:"no-store"})).text();
      const a = js.match(/m\.f\s*=\s*\[(.*?)\]/s);
      if (a) deps = [...a[1].matchAll(/"\.\/([^"]+\.js)"/g)].map(x => x[1]);
    }
  } catch (e) { say("  could not read the entry: " + e.message, "bad"); }
  say("  chunks     " + deps.length);
  const blocked = [], notok = [];
  for (const d of deps) {
    STATUS.textContent = "checking chunks… " + (blocked.length + notok.length);
    try {
      const r = await fetch("./assets/" + d, {cache:"no-store", headers:{Range:"bytes=0-64"}});
      if (!r.ok && r.status !== 206) notok.push(d + "  HTTP " + r.status);
    } catch (e) { blocked.push(d + "  " + e.message); }
  }
  say("  non-2xx    " + notok.length, notok.length ? "bad" : "ok");
  notok.slice(0,10).forEach(x => say("    " + x, "bad"));
  say("  BLOCKED    " + blocked.length, blocked.length ? "bad" : "ok");
  blocked.slice(0,15).forEach(x => say("    " + x, "bad"));
  if (blocked.length) {
    say("  >>> These never reached the server. A content blocker, Safari's", "bad");
    say("  >>> tracking prevention, or DNS filtering is stopping them. One", "bad");
    say("  >>> blocked module breaks the whole app.", "bad");
  }

  head("download speed from THIS browser");
  try {
    let big = null, bigN = 0;
    for (const d of deps) {
      try {
        const r = await fetch("./assets/" + d, {method:"HEAD", cache:"no-store"});
        const n = +(r.headers.get("Content-Length") || 0);
        if (n > bigN) { bigN = n; big = d; }
      } catch (e) {}
    }
    if (big) {
      STATUS.textContent = "measuring speed…";
      const t1 = performance.now();
      const r = await fetch("./assets/" + big, {cache:"no-store"});
      const buf = await r.arrayBuffer();
      const dt = (performance.now() - t1) / 1000;
      const mbps = (buf.byteLength * 8 / dt) / 1e6;
      say("  chunk      " + big);
      say("  got        " + (buf.byteLength/1e6).toFixed(2) + " MB in " + dt.toFixed(1) + "s");
      say("  speed      " + mbps.toFixed(1) + " Mbit/s");
      const gz = r.headers.get("Content-Encoding") === "gzip" || buf.byteLength < bigN * 0.6;
      const payload = gz ? 3.8e6 : 12.6e6;
      const est = payload / (buf.byteLength / dt);
      say("  the app must fetch about " + (payload/1e6).toFixed(1) +
          " MB -> roughly " + est.toFixed(0) + "s before anything appears",
          est > 45 ? "warn" : "ok");
      if (est > 45) say("  >>> That is slow enough to look like a hang. Wait it out, or " +
                        "rerun cell 2 for a different tunnel.", "warn");
    }
  } catch (e) { say("  could not measure: " + e.message, "warn"); }

  head("custom node scripts");
  let exts = null;
  for (const p of ["/api/extensions","/extensions"]) {
    try { const r = await fetch(p,{cache:"no-store"}); if (r.ok) { exts = await r.json(); break; } }
    catch(e){}
  }
  if (!exts) say("  could not read the list", "warn");
  else {
    say("  listed     " + exts.length);
    const ebad = [];
    for (const e of exts.slice(0,60)) {
      try { const r = await fetch(e,{cache:"no-store",headers:{Range:"bytes=0-64"}});
            if (!r.ok && r.status !== 206) ebad.push(e + "  HTTP " + r.status); }
      catch(err){ ebad.push(e + "  " + err.message); }
    }
    say("  failing    " + ebad.length, ebad.length ? "bad" : "ok");
    ebad.slice(0,12).forEach(x => say("    " + x, "bad"));
    if (ebad.length) say("  >>> Set SAFE_MODE = True in cell 2 to start without them.", "warn");
  }

  head("api endpoints");
  for (const p of ["/api/system_stats","/api/settings","/api/userdata?dir=workflows",
                   "/api/object_info","/api/extensions","/api/queue"]) {
    const t = performance.now();
    try {
      const r = await fetch(p,{cache:"no-store"});
      const b = await r.arrayBuffer();
      const dt = ((performance.now()-t)/1000).toFixed(2);
      say("  " + dt.padStart(6) + "s  " + String(r.status).padEnd(4) +
          String(b.byteLength).padStart(9) + " b  " + p, r.ok ? "" : "bad");
    } catch (e) {
      say("  ERROR   " + p + "  " + e.message, "bad");
    }
  }

  head("websocket");
  await new Promise(res => {
    const url = origin.replace(/^http/, "ws") + "/ws?clientId=browsercheck";
    let done = false;
    const to = setTimeout(() => { if (!done) { done = true; say("  TIMEOUT after 12s - the app waits on this", "bad"); res(); } }, 12000);
    try {
      const ws = new WebSocket(url);
      ws.onopen = () => { if(!done){done=true; clearTimeout(to); say("  connected", "ok"); ws.close(); res();} };
      ws.onerror = () => { if(!done){done=true; clearTimeout(to); say("  ERROR - Safari refused the websocket", "bad"); res();} };
      ws.onclose = (e) => { if(!done){done=true; clearTimeout(to); say("  closed before opening, code " + e.code, "bad"); res();} };
    } catch (e) { done = true; clearTimeout(to); say("  threw: " + e.message, "bad"); res(); }
  });

  head("does the app actually mount");
  await new Promise(res => {
    const f = document.createElement("iframe");
    f.style.cssText = "width:1px;height:1px;opacity:0;position:absolute;left:-9999px";
    const errs = [];
    let settled = false;
    const finish = (verdict, cls) => {
      if (settled) return; settled = true;
      say("  " + verdict, cls);
      errs.slice(0,12).forEach(e => say("    " + e, "bad"));
      try { f.remove(); } catch(_) {}
      res();
    };
    f.onload = () => {
      try {
        const w = f.contentWindow;
        w.addEventListener("error", ev => errs.push(
          (ev.target && ev.target.src) ? ("resource " + ev.target.src) :
          (ev.message + " @ " + (ev.filename||"") + ":" + (ev.lineno||""))), true);
        w.addEventListener("unhandledrejection", ev => errs.push(
          "promise: " + (ev.reason && ev.reason.message ? ev.reason.message : ev.reason)));
      } catch (e) { /* cross-origin, unlikely here */ }
      setTimeout(() => {
        let mounted = false, splash = false;
        try {
          const d = f.contentDocument;
          const app = d.querySelector("#vue-app");
          mounted = !!(app && app.children.length);
          splash = !!d.querySelector("#splash-loader");
        } catch (e) { errs.push("cannot inspect the frame: " + e.message); }
        if (mounted) finish("the app MOUNTED in a background frame - the server and this browser are both fine", "ok");
        else finish("the app did NOT mount after 20s" + (splash ? " (still on the splash screen)" : "") +
                    " - errors captured below", "bad");
      }, 20000);
    };
    f.src = "/?browsercheck=" + Date.now();
    document.body.appendChild(f);
    STATUS.textContent = "loading the app in a hidden frame, 20s…";
  });

  say("");
  say("elapsed " + ((performance.now()-t0)/1000).toFixed(1) + "s");
  say("=== end ===");
  STATUS.textContent = "done";
  document.getElementById('run').disabled = false;
  document.getElementById('copy').disabled = false;
}

async function isProbablyPrivate() {
  try {
    const q = await navigator.storage.estimate();
    return q && q.quota && q.quota < 120000000 ? "likely" : "no";
  } catch (e) { return "unknown"; }
}

document.getElementById('run').onclick = run;
document.getElementById('copy').onclick = async () => {
  const text = LINES.join("\n");
  try { await navigator.clipboard.writeText(text); STATUS.textContent = "copied"; }
  catch (e) {
    const ta = document.createElement("textarea");
    ta.value = text; document.body.appendChild(ta); ta.select();
    document.execCommand("copy"); ta.remove(); STATUS.textContent = "copied";
  }
};
</script>
</body>
</html>
'''


# ---------------------------------------------------------------- boot retry
# The app is ~65 separate requests. A quick tunnel drops one now and then, and a
# single missing chunk is a broken module graph: canvas, no UI, or nothing at
# all. Reloading fixes it because the rest is already cached and only the missing
# piece is refetched. That is a machine's job, not yours.
BOOT_RETRY_JS = r'''<script>
/* frizzy-boot-retry */
(function () {
  var KEY = "frizzyBootTries";
  var MAX = 3;
  var tries = parseInt(sessionStorage.getItem(KEY) || "0", 10);
  var failed = [];
  var done = false;

  function isAsset(u) {
    return u && /\/assets\/|\.js(\?|$)|\.css(\?|$)/.test(u);
  }

  function retry(what) {
    if (done) return;
    if (failed.indexOf(what) < 0) failed.push(what);
    if (tries >= MAX) return giveUp();
    done = true;
    sessionStorage.setItem(KEY, String(tries + 1));
    banner("An asset did not arrive. Reloading (" + (tries + 1) + "/" + MAX + ")...");
    setTimeout(function () { location.reload(); }, 600);
  }

  function banner(text) {
    var d = document.createElement("div");
    d.style.cssText = "position:fixed;left:0;right:0;top:0;z-index:99999;padding:10px 14px;" +
      "background:#1e3a5f;color:#fff;font:13px system-ui;text-align:center";
    d.textContent = text;
    (document.body || document.documentElement).appendChild(d);
  }

  function giveUp() {
    done = true;
    var d = document.createElement("div");
    d.style.cssText = "position:fixed;inset:0;z-index:99999;background:#161616;color:#eee;" +
      "font:13px/1.6 system-ui;padding:40px;overflow:auto";
    d.innerHTML = "<h2 style='margin:0 0 8px'>The app could not finish loading</h2>" +
      "<p style='color:#aaa;max-width:640px'>" + MAX + " attempts, and at least one file " +
      "never arrived. The server is almost certainly fine: this is the tunnel dropping " +
      "requests. Reload once more, or get a fresh tunnel by rerunning the launch cell in Colab.</p>" +
      "<pre style='color:#f6ad55;white-space:pre-wrap'>" +
      failed.map(function (f) { return f.replace(location.origin, ""); }).join("\n") +
      "</pre><button id='fzr' style='background:#2b6cb0;color:#fff;border:0;border-radius:6px;" +
      "padding:9px 14px;font:13px system-ui;cursor:pointer'>Try again</button>";
    (document.body || document.documentElement).appendChild(d);
    var b = document.getElementById("fzr");
    if (b) b.onclick = function () { sessionStorage.removeItem(KEY); location.reload(); };
  }

  // a <script> or <link> that 404s or resets: no exception, only this event
  window.addEventListener("error", function (e) {
    var t = e && e.target;
    if (!t || !t.tagName) return;
    var u = t.src || t.href;
    if (isAsset(u)) retry(u);
  }, true);

  // Vite's lazy chunks fail as a rejected promise instead
  window.addEventListener("unhandledrejection", function (e) {
    var m = e && e.reason && (e.reason.message || String(e.reason));
    if (m && /dynamically imported module|Importing a module script failed|Failed to fetch/i.test(m))
      retry(m.slice(0, 160));
  });

  // it mounted, so forget the counter and let a later reload start clean
  window.addEventListener("load", function () {
    setTimeout(function () {
      if (!done) sessionStorage.removeItem(KEY);
    }, 8000);
  });
})();
</script>
'''


def inject_boot_retry(root=None):
    """Put the retry script in index.html. Idempotent, and it keeps a backup so a
    second run patches the original rather than a patched copy."""
    d = Path(root) if root else frontend_static_dir()
    if d is None or not Path(d).is_dir():
        return False, "no frontend directory"
    idx = Path(d) / "index.html"
    if not idx.is_file():
        return False, "no index.html"
    body = idx.read_text(encoding="utf8", errors="ignore")
    if "frizzy-boot-retry" in body:
        return True, "already there"
    backup = idx.with_suffix(".html.orig")
    if not backup.exists():
        backup.write_text(body, encoding="utf8")
    if "</head>" in body:
        body = body.replace("</head>", BOOT_RETRY_JS + "</head>", 1)
    else:
        body = BOOT_RETRY_JS + body
    idx.write_text(body, encoding="utf8")
    # the .gz next to it is now the old html
    gz = idx.with_name(idx.name + ".gz")
    try:
        gz.unlink()
    except Exception:
        pass
    return True, "injected"


step("Making the first load survive a dropped request")
_bok, _bmsg = inject_boot_retry(FRONT_ROOT)
if _bok:
    ok("index.html retries a missing chunk by itself (%s)" % _bmsg)
    info("a dropped asset reloads once instead of leaving you on a dead page")
else:
    warn("could not add the boot retry: %s" % _bmsg)
rows.append(("OK" if _bok else "WARN", "boot retry", _bmsg))


step("Pre-compressing the frontend")
if True:
    _d, cst = precompress_frontend(root=FRONT_ROOT)
    if _d is None:
        warn(cst.get("error", "could not pre-compress"))
    else:
        ok("%d files: %s -> %s on the wire (%.1fx less to download)"
           % (cst["files"], human(cst["raw"]), human(cst["gz"]),
              cst["raw"] / max(1, cst["gz"])))
        rows.append(("OK", "precompressed",
                     "%.1fx smaller" % (cst["raw"] / max(1, cst["gz"]))))

step("Building flags")
rc, help_txt = sh("%s main.py --help" % sys.executable, cwd=ROOT, check=False,
                  label="comfy --help", timeout=300)
if rc != 0 or not help_txt.strip():
    fail("main.py --help failed - the install is broken")
    explain(help_txt)
    raise SystemExit("Rerun cell 1.")
supported = lambda f: f.split("=")[0].split()[0] in help_txt

try:
    import torch
    p = torch.cuda.get_device_properties(0)
    cc, vram = p.major * 10 + p.minor, p.total_memory / 1024 ** 3
except Exception:
    cc, vram = 0, 0.0


def build_args(vram_mode):
    return build_comfy_args(help_txt, port=PORT, cc=cc, vram=vram, vram_mode=vram_mode,
                            attention=ATTENTION, cache=CACHE, previews=PREVIEWS,
                            legacy_frontend=LEGACY_FRONTEND, verbose=VERBOSE,
                            disable_api_nodes=DISABLE_API_NODES, cors=CORS_HEADER,
                            reserve_vram=(RESERVE_VRAM_GB if RESERVE_VRAM_GB else None),
                            extra=" ".join(filter(None, [
                                EXTRA_ARGS,
                                "--disable-all-custom-nodes" if SAFE_MODE else "",
                                ('--front-end-root "%s"' % FRONT_ROOT) if FRONT_ROOT else ""])))


args, dropped = build_args(VRAM_MODE)
ok(" ".join(args))
if dropped:
    warn("dropped, not in this ComfyUI build: " + " ".join(dropped))
if SAFE_MODE:
    warn("SAFE MODE: every custom node is disabled. Your workflows will not run,")
    info("this is only to find out whether a node is what is breaking the UI.")
if not CORS_HEADER:
    warn("CORS_HEADER is off. Clicking the URL from this output will return 403")
    info("and show a blank page. Paste the URL into the address bar instead.")


def start(argv):
    fh = open(LOG, "w")
    pr = subprocess.Popen("%s main.py %s" % (sys.executable, " ".join(argv)),
                          shell=True, cwd=ROOT, stdout=fh, stderr=subprocess.STDOUT)
    print("  waiting for the server", end="", flush=True)
    got = wait_for(lambda: port_open(PORT), timeout=STARTUP_TIMEOUT, interval=3,
                   dots=True, on_dead=lambda: pr.poll() is not None)
    return pr, got


proc, up = start(args)

if up is None or up is False:
    body = LOG.read_text(errors="ignore")
    print(" " + ("DIED" if up is None else "TIMEOUT"))
    hit = explain(body)
    if AUTO_RETRY_ON_OOM and hit == "GPU ran out of VRAM" and VRAM_MODE == "auto":
        for mode in OOM_LADDER:
            warn("out of VRAM - retrying with --%s" % mode)
            kill_all(PORT)
            args, _ = build_args(mode)
            info(" ".join(args))
            proc, up = start(args)
            if up is True:
                ok("came up on --%s" % mode)
                break
            body = LOG.read_text(errors="ignore")
    if up is None or up is False:
        print("\n  --- last 30 log lines ---")
        for l in body.splitlines()[-30:]:
            print("   ", l[:150])
        raise SystemExit("ComfyUI did not come up.")
ok("listening on %d" % PORT)

# ---------------------------------------------------------------- health
step("Checking it actually serves the UI")
hrows, worst = comfy_health(timeout=25)
for dt, status, size, path, label, timed_out in hrows:
    if timed_out:
        print("     TIMEOUT    %s   <-- %s" % (path, label))
    else:
        print("   %7.2fs  %3s  %9s  %s" % (dt, status or "-", human(size), path))

verdict = health_verdict(worst, timeout=25)
local_ok, local_msg = check_url("http://127.0.0.1:%d/" % PORT)
(ok if local_ok else fail)("index.html: " + local_msg)
if not local_ok:
    explain("comfyui-frontend-package is not installed")

# index.html arriving is not the same as the app loading. A white screen is
# index.html plus JavaScript that 404s, so fetch what it actually references.
assets_ok = True
if local_ok:
    step("Fetching the JavaScript index.html asks for")
    arows, aerr = check_assets("http://127.0.0.1:%d" % PORT)
    if aerr:
        fail(aerr)
        explain(aerr)
        assets_ok = False
    else:
        for u, status, size, good_a, why, required in arows:
            tag = "js " if required else "opt"
            line = "%s %-40s %s %s%s" % (tag, u[-40:], status, human(size),
                                         ("  " + why) if why else "")
            if required:
                (ok if good_a else fail)(line)
            else:
                info(line)
        broken = assets_broken(arows)
        assets_ok = not broken
        if broken:
            explain("assets/app.js 404 Not Found")
        else:
            ok("every required script loads (the css 404s above are normal on a "
               "fresh install)")
rows.append(("OK" if assets_ok else "FAIL", "frontend assets",
             "all load" if assets_ok else "one or more failed - this is the white screen"))

# index.html names 3 scripts; the app is ~65 chunks loaded by dynamic import.
# One unreachable chunk kills the whole module graph and blanks the page.
graph_ok, prone, check_page = True, [], None
if local_ok:
    step("Checking the full app module graph")
    entry, deps, gerr = parse_module_graph("http://127.0.0.1:%d" % PORT)
    if gerr:
        warn(gerr)
    elif deps:
        bad, prone, checked = check_module_graph("http://127.0.0.1:%d" % PORT, deps)
        ok("%d chunks in the dependency map, %d checked" % (len(deps), checked))
        if bad:
            graph_ok = False
            for d, why in bad[:12]:
                fail("%-46s %s" % (d[:46], why))
            explain("assets/app.js 404 Not Found")
        else:
            ok("every chunk the app needs is served correctly")
    rows.append(("OK" if graph_ok else "FAIL", "module graph",
                 "all chunks served" if graph_ok else "chunks missing - reinstall the frontend"))

    step("Installing the in-browser check page")
    check_page = install_check_page(CHECK_HTML, root=FRONT_ROOT)
    (ok if check_page else warn)(
        "open <url>/check.html to test from inside the browser" if check_page
        else "could not install the check page (frontend dir not writable)")

    step("Checking custom-node frontend scripts")
    exts, ebad, eerr = check_extensions("http://127.0.0.1:%d" % PORT)
    if eerr:
        warn(eerr)
    else:
        ok("%d extension script(s) listed" % len(exts))
        for e, why in ebad[:10]:
            fail("%-52s %s" % (e[-52:], why))
        if ebad:
            warn("the app loads every one of these while booting - a failing one can "
                 "leave you stuck on the splash screen")
            info("turn SAFE_MODE on in this cell to start with all custom nodes off")
        rows.append(("OK" if not ebad else "WARN", "node scripts",
                     "%d ok" % len(exts) if not ebad else "%d of %d failed"
                     % (len(ebad), len(exts))))

step("How ComfyUI treats each way a browser can arrive")
opres, opblocked = check_origin_policy("http://127.0.0.1:%d" % PORT)
for label, (status, extra) in opres.items():
    (fail if status == 403 else ok)("%-30s %s" % (label, status))
if opblocked:
    fail("blocked: " + ", ".join(opblocked))
    explain("non matching host and origin")
    if CORS_HEADER:
        warn("CORS_HEADER is on but requests are still blocked - report this")
rows.append(("OK" if not opblocked else "FAIL", "origin policy",
             "all arrival modes accepted" if not opblocked
             else "403 on: " + ", ".join(opblocked)))
if verdict:
    (warn if verdict[0] == "slow" else fail)(verdict[1])
_sha = subprocess.run("git rev-parse --short HEAD", shell=True, cwd=ROOT,
                      capture_output=True, text=True).stdout.strip()
rows.append(("OK", "versions", "ComfyUI %s / frontend %s"
             % (_sha, pkg_version("comfyui-frontend-package") or "?")))
rows.append(("OK" if local_ok and not verdict else "WARN", "ui",
             local_msg if local_ok else "not serving"))

if proc.poll() is not None:
    fail("ComfyUI exited while we were checking it")
    explain(LOG.read_text(errors="ignore"))
    raise SystemExit("The server died after opening the port - see the log above.")

failed_nodes = [l for l in LOG.read_text(errors="ignore").splitlines() if "IMPORT FAILED" in l]
if failed_nodes:
    warn("custom nodes that failed to import:")
    for l in failed_nodes[:10]:
        info(l[:140])
    rows.append(("WARN", "custom nodes", "%d failed to import" % len(failed_nodes)))

if not local_ok or not assets_ok or opblocked or not graph_ok:
    fail("the server is up but the UI does not load locally - a tunnel cannot fix that")
    if opblocked:
        info("Next step: make sure CORS_HEADER is on, then run this cell again.")
    elif not SAFE_MODE:
        info("Next step: turn SAFE_MODE on in this cell and run it again.")
        info("If the UI appears, a custom node's JavaScript is the cause.")
        info("If it is still blank, rerun cell 1, then try LEGACY_FRONTEND.")
    else:
        info("Safe mode is already on, so no custom node is involved.")
        info("Rerun cell 1 to repair the frontend package, then try LEGACY_FRONTEND.")
    summary(rows + [("FAIL", "next", "SAFE_MODE, then cell 1, then LEGACY_FRONTEND")])
    raise SystemExit


# ComfyUI-Manager fetches six JSON caches at startup, and each one blocks the
# event loop it shares with everything else. Handing over a URL in the middle of
# that is what makes the first load feel broken.
step("Waiting for startup tasks to finish")
_deadline = time.time() + 90
_seen = False
while time.time() < _deadline:
    _log = LOG.read_text(errors="ignore")
    if "All startup tasks have been completed" in _log:
        _seen = True
        break
    if "ComfyUI-Manager" not in _log:
        break                                   # not installed, nothing to wait for
    time.sleep(2)
if _seen:
    ok("Manager finished its startup fetches")
else:
    info("carrying on - startup tasks are still running, the first load may be slower")


# ---------------------------------------------------------------- transport
step("Getting a URL (each one is verified for HTTP and websockets before you see it)")
NGROK = secret("NGROK_TOKEN")
ALL = {"cloudflared": lambda: tunnel_cloudflared(PORT),
       "pinggy": lambda: tunnel_pinggy(PORT),
       "localhost.run": lambda: tunnel_localhost_run(PORT),
       "ngrok": lambda: tunnel_ngrok(PORT, NGROK)}

if ACCESS in ("colab tab", "colab iframe"):
    cands = [(ACCESS, lambda: tunnel_colab(PORT), True)]
else:
    # ngrok first when a token exists: the only one that does not depend on this
    # Colab host's egress IP being in good standing.
    order = (["cloudflared", "pinggy", "localhost.run", "ngrok"] if ACCESS == "auto"
             else [ACCESS] + [k for k in ALL if k != ACCESS])
    if ACCESS == "auto" and NGROK:
        order = ["ngrok"] + [k for k in order if k != "ngrok"]
    cands = [(n, ALL[n], False) for n in order]

ALL = {name: fn for name, fn, _d in cands}
URL, VIA, DEGRADED, attempts = pick_transport(cands)

rule()
if URL:
    print("  OPEN THIS   (%s)" % VIA)
    print("  " + URL)
    if DEGRADED:
        print("\n  Degraded: Google's proxy does not forward websockets.")
        print("  Previews and progress will not work and the app may not finish loading.")
    print("")
    if FRONT_ROOT:
        print("  Serving the frontend rewritten for %s." % SAFARI_TARGET)
    print("  Models button, bottom right: downloads land on this VM, not your laptop.")
    if check_page:
        print("")
        print("  If that page will not load, open this one instead and press Run:")
        print("  " + URL.rstrip("/") + "/check.html")
        print("  It tests everything from inside your browser and shows the result")
        print("  on screen. No developer tools needed.")
    if prone:
        print("")
        print("  If that page is blank in Chrome too, it is not the server.")
        print("  The app is an ES module graph and one blocked module breaks all of it.")
        print("  These chunks have names ad blockers and DNS filters commonly match:")
        for d in prone[:6]:
            print("      %s" % d)
        print("  Test in a private window with extensions disabled. If it loads there,")
        print("  allowlist this host in your blocker.")
    if ACCESS == "colab iframe":
        try:
            from IPython.display import IFrame, display
            display(IFrame(URL, width="100%", height=900))
        except Exception as e:
            warn("could not render the iframe: %s" % str(e)[:80])
    if not DEGRADED:
        step("Re-checking through the tunnel (paths can break in transit)")
        arows, aerr = check_assets(URL)
        if aerr:
            warn(aerr)
        else:
            bad_through = assets_broken(arows)
            if bad_through:
                for u, status, size, _g, why, _r in bad_through:
                    fail("%s -> %s %s" % (u[-44:], status, why))
                warn("assets load locally but not through %s - that is the tunnel, "
                     "not ComfyUI. Rerun this cell for a different one." % VIA)
            else:
                ok("assets load through the tunnel too")
        step("Measuring real download speed through the tunnel")
        tp = measure_throughput(URL, deps=deps if deps else None)
        if not tp:
            info("could not measure it")
        elif tp.get("error"):
            warn("throughput probe failed: %s" % tp["error"])
        else:
            ok("%s of %s in %.1fs = %.1f Mbit/s (encoding: %s)"
               % (human(tp["got"]), tp["chunk"][:28], tp["secs"], tp["mbps"], tp["encoding"]))
            payload = 3.8e6 if tp["encoding"] == "gzip" else 12.6e6
            est = payload / max(1.0, tp["got"] / tp["secs"])
            info("the app has about %s of javascript to fetch -> roughly %.0fs to first paint"
                 % (human(payload), est))
            if est > 45:
                warn("that is slow enough to look like it is hanging. Give it %.0fs "
                     "before deciding it is broken, or rerun for a different tunnel." % est)
                rows.append(("WARN", "app load estimate", "~%.0fs" % est))
            else:
                rows.append(("OK", "app load estimate", "~%.0fs" % est))

        tres, tblocked = check_origin_policy(URL)
        if tblocked:
            fail("through the tunnel, ComfyUI 403s when: " + ", ".join(tblocked))
            explain("non matching host and origin")
            info("Paste the URL into the address bar instead of clicking it,")
            info("or relaunch with CORS_HEADER on.")
        else:
            ok("every arrival mode works through the tunnel - clicking the link is fine")
    rows.append(("WARN" if DEGRADED else "OK", "url", VIA))
else:
    fail("no transport produced a working URL")
    for name, good, why in attempts:
        info("%-14s %s" % (name, why))
    info("")
    info("Quick tunnels are assigned per run, so rerunning this cell often fixes it.")
    info("For something that does not depend on Colab's shared IP: free account at")
    info("ngrok.com, then add the authtoken as a Colab secret named NGROK_TOKEN.")
    rows.append(("FAIL", "url", "%d transports tried" % len(attempts)))
print("=" * 68)

rows.append(("OK", "installed", "%s models, %d workflows"
             % (human(inv["model_bytes"]), inv["workflows"])))
summary(rows)

# ---------------------------------------------------------------- keep alive
WATCH = None
if URL and not DEGRADED:
    _via = {"name": VIA}

    def _rebuild_tunnel():
        # Same transport first, since the URL sometimes comes back identical.
        kill_tunnel(_via["name"])
        fn = ALL.get(_via["name"])
        if fn is not None:
            u, e = fn()
            if u and tunnel_ping(u)[0]:
                return u, _via["name"], ""
            kill_tunnel(_via["name"])
        others = [(n, ALL[n], False) for n in ALL if n != _via["name"]]
        u2, v2, _d, _a = pick_transport(others, log=False)
        if u2:
            _via["name"] = v2
            return u2, v2, ""
        return None, None, "every transport failed"

    WATCH = watch_tunnel(URL, VIA, _rebuild_tunnel, interval=60, grace=3)
    print("  Watching the tunnel every 60s. A dead one is rebuilt and the new URL is")
    print("  printed here. A slow one is left alone: every miss is checked against")
    print("  127.0.0.1 first, so a busy ComfyUI never costs you a working tunnel.")

    # Colab disconnects an idle BROWSER after about 90 minutes even while the
    # kernel is busy. This keeps the tab looking active and clicks reconnect if
    # one appears. It does not beat the 12 hour cap, and closing the tab still
    # ends the run.
    try:
        from IPython.display import Javascript, display
        display(Javascript("""
        if (!window._frizzyKeepAlive) {
          window._frizzyKeepAlive = setInterval(() => {
            try {
              document.dispatchEvent(new MouseEvent("mousemove",
                {bubbles: true, clientX: 1 + (Date.now() % 3), clientY: 1}));
              const b = document.querySelector("colab-connect-button");
              const inner = b && b.shadowRoot &&
                            b.shadowRoot.querySelector("#connect");
              if (inner && /reconnect/i.test(inner.textContent || "")) inner.click();
            } catch (e) {}
          }, 60000);
          console.log("frizzy keepalive on");
        }
        """))
        print("  Browser keepalive on. Stop it with  clearInterval(window._frizzyKeepAlive)")
    except Exception:
        pass

if AUTO_RESTART:
    print("\n  Supervising. If ComfyUI exits it is restarted and this URL keeps working,")
    print("  so ComfyUI-Manager's restart button is safe to use. Stop this cell to stop it.\n")
    try:
        proc.kill()
    except Exception:
        pass
    time.sleep(1)
    supervise(lambda: build_args(VRAM_MODE)[0], root=ROOT, log_path=LOG,
              on_event=lambda k, m: print("  [supervisor] %-8s %s" % (k, m), flush=True))
else:
    print("\n  Log tail below. Keep this cell running - stopping it stops ComfyUI.\n")
    subprocess.run("tail -f -n 25 %s" % LOG, shell=True)


## 3 - Save state

Pairs with cell 1. Records every custom node with its exact commit, so next session rebuilds
the same environment - including the Models panel, which lives in `custom_nodes`. Flushes
Drive at the end, because Drive buffers writes and a dying session eats them.


In [ ]:
import sys, os
if not os.path.exists("/content/frizzy_lib.py"):
    raise SystemExit("Run cell 1 (Setup) first.")
sys.path.insert(0, "/content")
import importlib, frizzy_lib
importlib.reload(frizzy_lib)
from frizzy_lib import *
#@title 4. Save state to Drive { display-mode: "form" }

SAVE_OUTPUTS = True  #@param {type:"boolean"}
SAVE_INPUTS = False  #@param {type:"boolean"}
RECORD_PIN = True  #@param {type:"boolean"}
#@markdown Records the exact ComfyUI commit and frontend versions running right now, so
#@markdown cell 1 can put you back here. Only run this from a session that actually worked.
CACHE_INSTALL = False  #@param {type:"boolean"}
#@markdown EXPERIMENTAL, and only useful with USE_INSTALL_CACHE on in cell 1. Tars the
#@markdown ComfyUI tree to Drive, models excluded, so the next session skips cloning.
#@markdown Your workflows, settings and custom node list are saved below regardless: that
#@markdown part is not affected by this and is what actually restores your session.
BUILD_WHEELHOUSE = True  #@param {type:"boolean"}
#@markdown Caches every wheel this install needed to Drive, so the next session installs
#@markdown offline and identically. Roughly 150-400 MB. torch is deliberately excluded.
FLUSH = True  #@param {type:"boolean"}
#@markdown Drive buffers writes. Without the flush a dying session eats them.

import json, os, subprocess, time
from pathlib import Path

rows = []
rule("SAVE STATE")
DRIVE = Path("/content/drive/MyDrive/frizzy-comfy")
if not os.path.ismount("/content/drive"):
    fail("Drive is not mounted")
    explain("drive.mount")
    raise SystemExit("Rerun cell 1 with USE_DRIVE on.")

stamp = time.strftime("%Y%m%d-%H%M")
for name, want in (("user", True), ("output", SAVE_OUTPUTS), ("input", SAVE_INPUTS)):
    if not want:
        continue
    src, dst = ROOT / name, DRIVE / name
    if not src.exists():
        warn(name + "/ does not exist, skipping")
        continue
    dst.mkdir(parents=True, exist_ok=True)
    n = sum(1 for _ in src.rglob("*") if _.is_file())
    rc, out = sh('rsync -a "%s/" "%s/"' % (src, dst), tries=2, check=False, label="rsync " + name)
    (ok if rc == 0 else fail)("%s/  %d files" % (name, n))
    if rc != 0:
        explain(out, quiet=True)
    rows.append(("OK" if rc == 0 else "FAIL", name, "%d files" % n))

g = lambda c, d: subprocess.run(c, shell=True, cwd=d, capture_output=True, text=True).stdout.strip()
snap = {"comfyui": g("git rev-parse HEAD", ROOT), "when": stamp, "nodes": {},
        "frontend": {}}
for _pkg in ("comfyui-frontend-package", "comfyui-workflow-templates",
             "comfyui-embedded-docs"):
    _v = pkg_version(_pkg)
    if _v:
        snap["frontend"][_pkg] = _v
for d in sorted((ROOT / "custom_nodes").iterdir()):
    if not (d / ".git").exists():
        continue
    snap["nodes"][d.name] = {"url": g("git config --get remote.origin.url", d),
                             "commit": g("git rev-parse HEAD", d)}
(DRIVE / "snapshots").mkdir(parents=True, exist_ok=True)
for n in ("snapshot-%s.json" % stamp, "latest.json"):
    (DRIVE / "snapshots" / n).write_text(json.dumps(snap, indent=2))
ok("snapshot: %d custom nodes, ComfyUI %s, frontend %s"
   % (len(snap["nodes"]), snap["comfyui"][:8],
      snap["frontend"].get("comfyui-frontend-package", "?")))
info("cell 1 can pin to exactly this with PIN_TO_LAST_WORKING - use it before you record")
rows.append(("OK", "snapshot", "%d nodes" % len(snap["nodes"])))

if RECORD_PIN:
    step("Recording this session as known good")
    pin = write_pin(DRIVE / "pin.json")
    ok("ComfyUI %s (%s)" % (pin["comfyui"], pin.get("comfyui_version", "?")))
    for k, v in (pin.get("packages") or {}).items():
        info("%-30s %s" % (k, v))
    info("cell 1 with VERSION_PIN = last known good will restore exactly this")
    rows.append(("OK", "pin recorded", pin["comfyui"]))

if CACHE_INSTALL:
    step("Caching the install")
    _p, _st = cache_install(DRIVE / "install-cache-v2.tar.gz")
    if _p is None:
        warn(_st.get("error", "could not cache the install"))
    else:
        info("cell 1 will restore this instead of cloning")
    rows.append(("OK" if _p else "WARN", "install cache",
                 human(_st["bytes"]) if _p else "failed"))

if BUILD_WHEELHOUSE:
    step("Caching wheels to Drive")
    okw, msgw, nw, sizew = wheelhouse_build(
        ROOT / "requirements.txt", DRIVE / "wheels",
        constraints=CONS if CONS.exists() else None, exclude=PROTECTED_PKGS)
    (ok if okw else warn)(msgw)
    if okw:
        info("next session installs from these with no network and no version drift")
        info("torch and numpy are excluded on purpose - Colab's own build must win")
    rows.append(("OK" if okw else "WARN", "wheelhouse", msgw))

if FLUSH:
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        ok("Drive flushed and unmounted - the writes are safe")
        info("rerun cell 1 to remount if you are carrying on")
    except Exception as e:
        warn("flush failed: %s" % str(e)[:80])

rows.append(("OK", "next", "safe to lose the session - cell 1 restores all of this"))
summary(rows)


## 4 - Models and doctor

One cell for "get me a model" and "why is this broken".

**Staging.** Paste URLs, press Download, and nothing downloads yet. You get a row per file
with an editable name, a folder dropdown, a subfolder box, the size, and a note saying how
the folder was worked out. Amber means it only had the extension to go on. Press Start and
it queues. If ComfyUI is running the jobs are handed to it, so they appear in the Models
panel too and the model lists refresh themselves on completion. If it is not running, the
same aria2 settings fetch them straight to disk.

**Moving.** Under that is everything already in `models/`, newest first. Pick a file, pick a
folder, press Move. That is the fix for every model that landed in `checkpoints` before this
build, and it works whether ComfyUI is up or not.

**The doctor** checks hardware, install, the frontend package *against the version this
ComfyUI revision pins* (a mismatch there breaks dialogs and the execution UI while
generation keeps working, which is a miserable thing to diagnose from the outside),
ComfyUI-Manager's age, the speed layer, every model file's first eight bytes, Drive, the
running server's boot endpoints, whether `/api/settings` and `/api/object_info` come back as
parseable JSON both locally and *through the tunnel*, whether asset caching is answering,
and the log.

With `AUTO_FIX` on it repairs what it safely can: a frontend version mismatch, an outdated
Manager, a missing or stale extension, a symlinked `user/`, error pages saved as
`.safetensors`, a port held by a dead process, unfinished downloads when disk is tight. Then
it prints one block you can paste back to me.

It imports from cell 1 when cell 1 has run, and falls back to its own definitions when it
has not, so it still works on a session that never got that far.


In [ ]:
import sys, os
sys.path.insert(0, "/content")
_LIB = os.path.exists("/content/frizzy_lib.py")
if _LIB:
    import importlib, frizzy_lib
    importlib.reload(frizzy_lib)
    from frizzy_lib import *
#@title 4. Models, doctor and repair { display-mode: "form" }

AUTO_FIX = True  #@param {type:"boolean"}
#@markdown Repairs what it can: the missing extension, a symlinked `user/`, half-finished
#@markdown downloads, error pages saved as models, a port held by a dead process.
CHECK_SERVER = True  #@param {type:"boolean"}
#@markdown Times the endpoints the frontend waits on, and checks that the API answers
#@markdown parseable JSON. Needs ComfyUI running, so stop cell 2 first if it is still
#@markdown supervising.
DEEP_CLEAN = False  #@param {type:"boolean"}
#@markdown Also empties the pip and HuggingFace caches. Several GB, and nothing you need.
LOG_LINES = 25  #@param {type:"integer"}

import json, re, shutil, subprocess, threading, time, urllib.request, urllib.parse
from pathlib import Path

# ---------------------------------------------------------------- fallbacks
# This cell has to work when cell 1 never finished, because that is exactly when
# you need it. Everything below is only defined if the library is not there.
if not _LIB:
    ROOT = Path("/content/ComfyUI")
    LOG = Path("/content/comfy.log")
    PORT = 8188

    class C:
        G = "\033[92m"; Y = "\033[93m"; R = "\033[91m"; B = "\033[94m"; X = "\033[0m"

    def rule(t=""):
        print("\n" + "=" * 74)
        if t:
            print(t)
            print("=" * 74)

    def step(m): print("\n" + C.B + "> " + m + C.X)
    def ok(m): print("  " + C.G + "OK  " + C.X + m)
    def warn(m): print("  " + C.Y + "WARN" + C.X + " " + m)
    def fail(m): print("  " + C.R + "FAIL" + C.X + " " + m)
    def info(m): print("       " + m)

    def human(n):
        n = float(n or 0)
        for u in ("B", "KB", "MB", "GB", "TB"):
            if n < 1024:
                return "%.1f %s" % (n, u)
            n /= 1024
        return "%.1f PB" % n

    def free_gb(p="/content"):
        try:
            return shutil.disk_usage(p).free / 1024 ** 3
        except Exception:
            return 0.0

    def port_open(port=PORT, host="127.0.0.1", timeout=1.0):
        import socket
        with socket.socket() as s:
            s.settimeout(timeout)
            return s.connect_ex((host, port)) == 0

    def summary(rows):
        rule("SUMMARY")
        for st, k, v in rows:
            print("  %-5s %-16s %s" % (st, k, v))

    def explain(text, quiet=False, limit=2):
        return None

    def kill_all(port=PORT, extra=()):
        subprocess.run("pkill -f main.py; pkill -f cloudflared; pkill -f ngrok",
                       shell=True, capture_output=True)

    def secret(name):
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return os.environ.get(name, "")

BASE = "http://127.0.0.1:%d" % PORT
MODELS = ROOT / "models"
NODE = ROOT / "custom_nodes" / "ComfyUI-FrizzyDownloader"
REPORT = []


def say(line=""):
    REPORT.append(line)


def api(path, payload=None, timeout=20):
    """Talk to the running ComfyUI. Returns None when it is not up."""
    url = BASE + path
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(
        url, data=data, headers={"Content-Type": "application/json"} if data else {})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = r.read()
            return json.loads(body) if body else {}
    except Exception:
        return None


SERVER_UP = port_open(PORT) and api("/system_stats", timeout=8) is not None

# ============================================================================
# PART 1 - MODEL DOWNLOADER
# ============================================================================
# When ComfyUI is running the queue is handed to it, so the same downloads show
# up in the Models panel inside the UI and the model lists refresh themselves.
# When it is not, the same files are fetched here with the same aria2 settings.

STANDARD_FOLDERS = ["checkpoints", "diffusion_models", "unet", "clip", "text_encoders",
                    "vae", "vae_approx", "loras", "controlnet", "clip_vision", "embeddings",
                    "upscale_models", "style_models", "audio_encoders", "gligen",
                    "photomaker", "ipadapter", "ultralytics", "model_patches"]

ARIA2 = shutil.which("aria2c")
HTML_HEADS = (b"<!DO", b"<htm", b"<HTM", b'{"er', b'{"me', b"<?xm", b"<!do")

ROUTING = [
    (r"vae[-_.]?approx", "vae_approx"),
    (r"\bvae\b|[-_]vae[-_.]", "vae"),
    (r"clip[-_]?vision", "clip_vision"),
    (r"text[-_]?encoder|umt5|t5xxl|clip[-_]?l\b|clip[-_]?g\b|gemma|qwen.*encoder", "text_encoders"),
    (r"controlnet|control[-_]?v\d|t2i[-_]?adapter", "controlnet"),
    (r"\blora\b|[-_]lora[-_.]|lightx2v|distill.*rank", "loras"),
    (r"upscal|esrgan|realesr|swinir", "upscale_models"),
    (r"embedding|textual[-_]inversion", "embeddings"),
    (r"\.gguf$", "diffusion_models"),
    (r"unet|diffusion[-_]model|transformer", "diffusion_models"),
]


def normalize(url):
    """HuggingFace and GitHub file PAGES are not the file. Turn them into it."""
    url = (url or "").strip()
    m = re.match(r"https?://huggingface\.co/([^/]+)/([^/]+)/blob/([^/]+)/(.+)", url)
    if m:
        return "https://huggingface.co/%s/%s/resolve/%s/%s" % m.groups()
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/blob/(.+)", url)
    if m:
        return "https://raw.githubusercontent.com/%s/%s/%s" % m.groups()
    return url


def name_from(url, headers=None):
    if headers:
        cd = headers.get("Content-Disposition") or ""
        m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?', cd)
        if m:
            return urllib.parse.unquote(m.group(1)).strip()
    p = urllib.parse.urlparse(url).path
    return urllib.parse.unquote(os.path.basename(p)) or "download.bin"


PATH_ALIASES = {"unet": "diffusion_models", "unet_models": "diffusion_models",
                "diffusion_models": "diffusion_models", "transformers": "diffusion_models",
                "text_encoder": "text_encoders", "text_encoders": "text_encoders",
                "clip": "text_encoders", "clip_vision": "clip_vision", "vae": "vae",
                "vae_approx": "vae_approx", "loras": "loras", "lora": "loras",
                "controlnet": "controlnet", "controlnets": "controlnet",
                "embeddings": "embeddings", "upscale_models": "upscale_models",
                "upscale": "upscale_models", "checkpoints": "checkpoints",
                "style_models": "style_models", "audio_encoders": "audio_encoders",
                "ipadapter": "controlnet", "photomaker": "photomaker"}

EXT_FALLBACK = [(r"\.gguf$", "diffusion_models"), (r"\.(ckpt|pt|pth)$", "checkpoints"),
                (r"\.safetensors$", "checkpoints")]


def guess_folder_conf(url, filename=""):
    """(folder, confidence). A .safetensors extension says nothing about what a
    file IS, which is why everything used to land in checkpoints. The URL path
    usually does: most repos already sort into split_files/<folder>/."""
    segs = [urllib.parse.unquote(x).lower()
            for x in urllib.parse.urlparse(url).path.split("/") if x]
    for seg in reversed(segs[:-1] if segs else []):
        mapped = PATH_ALIASES.get(seg)
        if mapped:
            return mapped, "high"
    name = (filename or "").lower()
    from_url = segs[-1] if segs else ""
    rest = urllib.parse.unquote(url).lower()
    for hay in (name, from_url, rest):
        if not hay:
            continue
        for pat, folder in ROUTING:
            if re.search(pat, hay):
                return folder, "medium"
    for hay in (name, from_url):
        for pat, folder in EXT_FALLBACK:
            if hay and re.search(pat, hay):
                return folder, "low"
    return "checkpoints", "low"


def guess_folder(url, filename=""):
    return guess_folder_conf(url, filename)[0]


def move_local(src, folder, filename=None):
    src = Path(src)
    dest_dir = MODELS / folder
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / (filename or src.name)
    if dest.resolve() == src.resolve():
        return dest, "already there"
    if dest.exists():
        raise ValueError("%s already exists" % dest)
    shutil.move(str(src), str(dest))
    return dest, ""


def local_files(limit=300):
    out = []
    if not MODELS.is_dir():
        return out
    for p in MODELS.rglob("*"):
        if not p.is_file() or p.suffix in (".part", ".aria2", ".download"):
            continue
        try:
            st = p.stat()
        except Exception:
            continue
        out.append({"path": str(p), "name": p.name, "size": st.st_size,
                    "folder": str(p.parent.relative_to(MODELS)), "mtime": st.st_mtime})
    out.sort(key=lambda x: -x["mtime"])
    return out[:limit]


def head(url, token=""):
    req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "frizzy-colab"})
    if token and ("huggingface.co" in url or "hf.co" in url):
        req.add_header("Authorization", "Bearer " + token)
    try:
        with urllib.request.urlopen(req, timeout=25) as r:
            return int(r.headers.get("Content-Length") or 0), name_from(url, r.headers)
    except Exception:
        return 0, name_from(url)


def duplicates(filename):
    if not filename or not MODELS.is_dir():
        return []
    return [str(p) for p in MODELS.rglob(filename) if p.is_file()][:6]


def live_folders():
    r = api("/frizzy/dl/folders", timeout=10) if SERVER_UP else None
    if r and r.get("folders"):
        return [f["name"] for f in r["folders"]]
    ondisk = sorted(p.name for p in MODELS.glob("*") if p.is_dir()) if MODELS.is_dir() else []
    return sorted(set(STANDARD_FOLDERS) | set(ondisk))


def aria2_download(url, dest_dir, filename, token="", out=None, total=0):
    """Same flags the server uses: 16 connections, resumable, verified."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / filename
    part = filename + ".part"
    if dest.exists() and dest.stat().st_size > 4096:
        return True, "already on disk (%s)" % human(dest.stat().st_size)

    hdr = []
    if token and ("huggingface.co" in url or "hf.co" in url):
        hdr = ["--header=Authorization: Bearer " + token]

    if ARIA2:
        cmd = [ARIA2, "--console-log-level=warn", "--summary-interval=1", "--continue=true",
               "--max-connection-per-server=16", "--split=16", "--min-split-size=1M",
               "--max-tries=3", "--retry-wait=5", "--allow-overwrite=true",
               "--auto-file-renaming=false", "--file-allocation=none",
               "--user-agent=frizzy-colab"] + hdr + ["-d", str(dest_dir), "-o", part, url]
    else:
        cmd = ["curl", "-L", "-C", "-", "--retry", "3", "--fail-with-body",
               "-o", str(dest_dir / part), url]
        for h in hdr:
            cmd[1:1] = ["-H", h.replace("--header=", "")]

    # Progress comes from stat() on the part file. aria2's own progress line is a
    # carriage-return redraw whose format moves between versions, and curl has a
    # different one again.
    p = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    src = dest_dir / part
    while p.poll() is None:
        time.sleep(0.7)
        if out is not None:
            try:
                n = src.stat().st_size
            except Exception:
                n = 0
            out("%s\n   %s%s" % (filename, human(n),
                                 ("  of %s  (%d%%)" % (human(total), n * 100 // max(total, 1)))
                                 if total else ""))
    p.wait()
    if p.returncode != 0 or not src.exists():
        return False, "downloader exited %s" % p.returncode
    with open(src, "rb") as f:
        magic = f.read(8)
    if src.stat().st_size < 4096 or magic[:4] in HTML_HEADS:
        src.unlink(missing_ok=True)
        return False, "that was an error page, not a model (gated repo or bad token)"
    for junk in (str(src) + ".aria2",):
        Path(junk).unlink(missing_ok=True)
    src.replace(dest)
    return True, human(dest.stat().st_size)


def build_downloader_ui():
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
        import ipywidgets as widgets
        from IPython.display import display

    FOLDERS = live_folders()
    WIDE = widgets.Layout(width="100%")

    urls = widgets.Textarea(
        placeholder=("One model URL per line.\n"
                     "HuggingFace resolve or blob links, CivitAI download links, direct URLs."),
        layout=widgets.Layout(width="100%", height="110px"))
    default_folder = widgets.Dropdown(
        options=["(work it out per link)"] + FOLDERS, value="(work it out per link)",
        description="Folder:", layout=widgets.Layout(width="330px"))
    hf = widgets.Password(placeholder="HF token (gated repos)", description="HF:",
                          layout=widgets.Layout(width="290px"))
    stage_btn = widgets.Button(description="Download", button_style="success",
                               layout=widgets.Layout(width="150px"))
    stage_box = widgets.VBox([])
    start_btn = widgets.Button(description="Start downloads", button_style="success",
                               disabled=True, layout=widgets.Layout(width="180px"))
    drop_btn = widgets.Button(description="Clear staging", button_style="",
                              layout=widgets.Layout(width="150px"))
    out = widgets.Output()

    tok = secret("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
    if tok:
        hf.value = tok

    # append_stdout is safe from a worker thread; `with out:` is not, and the
    # staging and download work both run on one.
    def show(text, replace=True):
        if replace:
            out.outputs = ()
        out.append_stdout(str(text) + "\n")

    rows = []          # [{url, name_w, folder_w, sub_w, note_w, size}]

    def row_widget(item):
        name_w = widgets.Text(value=item["filename"], layout=widgets.Layout(width="300px"))
        folder_w = widgets.Dropdown(
            options=FOLDERS if item["folder"] in FOLDERS else FOLDERS + [item["folder"]],
            value=item["folder"], layout=widgets.Layout(width="190px"))
        sub_w = widgets.Text(value=item.get("sub", ""), placeholder="subfolder",
                             layout=widgets.Layout(width="140px"))
        bits = []
        if item["confidence"] == "low":
            bits.append("only the extension said so, check this")
        elif item["confidence"] == "medium":
            bits.append("from the filename")
        if item.get("exists"):
            bits.append("already there, will be skipped")
        elif item.get("dupes"):
            bits.append("same name in " + ", ".join(item["dupes"]))
        colour = "#f6ad55" if item["confidence"] == "low" else "#888"
        note_w = widgets.HTML("<span style='color:%s;font-size:12px'>%s%s</span>"
                              % (colour, human(item["size"]) if item["size"] else "size unknown",
                                 ("  -  " + "; ".join(bits)) if bits else ""))
        rows.append({"url": item["url"], "name_w": name_w, "folder_w": folder_w,
                     "sub_w": sub_w, "size": item["size"]})
        return widgets.VBox([widgets.HBox([name_w, folder_w, sub_w]), note_w])

    def stage(urls_in, token, forced):
        boxes, total = [], 0
        rows.clear()
        for u in urls_in:
            url = normalize(u)
            size, fname = head(url, token)
            if forced:
                folder, conf = forced, "high"
            else:
                folder, conf = guess_folder_conf(url, fname)
            dest = MODELS / folder / fname
            total += size
            boxes.append(row_widget({"url": url, "filename": fname, "folder": folder,
                                     "confidence": conf, "size": size,
                                     "exists": dest.exists(),
                                     "dupes": [d.replace(str(MODELS) + "/", "")
                                               for d in duplicates(fname)
                                               if d != str(dest)]}))
        stage_box.children = tuple(boxes)
        start_btn.disabled = not boxes
        start_btn.description = "Start %d download(s)" % len(boxes)
        show("staged %d file(s), %s total, %.1f GB free on the VM\n"
             "change any folder above, then press Start"
             % (len(boxes), human(total), free_gb()))

    def on_stage(_):
        items = [l.strip() for l in urls.value.splitlines() if l.strip()]
        if not items:
            show("Paste a URL first.")
            return
        show("checking %d link(s)..." % len(items))
        forced = None if default_folder.value.startswith("(") else default_folder.value
        threading.Thread(target=stage, args=(items, hf.value, forced), daemon=True).start()

    def collect():
        items = []
        for r in rows:
            folder = r["folder_w"].value
            sub = r["sub_w"].value.strip().strip("/")
            items.append({"url": r["url"], "folder": folder + ("/" + sub if sub else ""),
                          "filename": r["name_w"].value.strip() or None,
                          "size": r["size"]})
        return items

    def work(items):
        if SERVER_UP and api("/frizzy/dl/folders", timeout=5) is not None:
            api("/frizzy/dl/tokens",
                {"hf": hf.value, "civitai": os.environ.get("CIVITAI_TOKEN", "")})
            api("/frizzy/dl/queue", {"items": [{k: v for k, v in it.items() if k != "size"}
                                               for it in items]})
            while True:
                st = api("/frizzy/dl/status", timeout=10)
                if st is None:
                    show("lost contact with ComfyUI - it may have restarted", replace=False)
                    return
                lines = ["downloading on the VM. The Models panel in ComfyUI shows the same "
                         "queue, and can move anything afterwards."]
                for j in st["jobs"]:
                    lines.append("%-38s %-18s %5.1f%%  %-9s %s"
                                 % ((j["filename"] or j["url"])[-38:], j["folder"] or "",
                                    j["pct"], j["state"], (j["message"] or "")[:40]))
                show("\n".join(lines))
                if not st["busy"]:
                    refresh_files(None)
                    return
                time.sleep(1.5)
        else:
            done = []
            for it in items:
                fname = it["filename"] or head(it["url"], hf.value)[1]
                good, msg = aria2_download(it["url"], MODELS / it["folder"], fname,
                                           hf.value, show, it["size"])
                done.append("%s %s -> %s  (%s)"
                            % ("OK  " if good else "FAIL", fname, it["folder"], msg))
                show("\n".join(done))
            show("\n".join(done + ["", "done. ComfyUI picks these up on its next start."]))
            refresh_files(None)

    def on_start(_):
        items = collect()
        if not items:
            return
        stage_box.children = ()
        start_btn.disabled = True
        rows.clear()
        urls.value = ""
        show("queued %d file(s)" % len(items))
        threading.Thread(target=work, args=(items,), daemon=True).start()

    def on_drop(_):
        rows.clear()
        stage_box.children = ()
        start_btn.disabled = True
        show("staging cleared")

    stage_btn.on_click(on_stage)
    start_btn.on_click(on_start)
    drop_btn.on_click(on_drop)

    # ---------------------------------------------------------------- moving
    # The guess is wrong often enough that fixing it afterwards has to be two
    # clicks. This lists what is already on disk, newest first.
    pick = widgets.Dropdown(options=[], description="File:",
                            layout=widgets.Layout(width="640px"))
    move_to = widgets.Dropdown(options=FOLDERS, description="To:",
                               layout=widgets.Layout(width="260px"))
    move_sub = widgets.Text(placeholder="subfolder", layout=widgets.Layout(width="150px"))
    move_btn = widgets.Button(description="Move", button_style="warning",
                              layout=widgets.Layout(width="110px"))
    reload_btn = widgets.Button(description="Reload list",
                                layout=widgets.Layout(width="130px"))
    move_out = widgets.Output()

    def file_list():
        if SERVER_UP:
            r = api("/frizzy/dl/files", timeout=20)
            if r and r.get("files"):
                return r["files"]
        return local_files()

    def refresh_files(_):
        try:
            files = file_list()
        except Exception as e:
            move_out.outputs = ()
            move_out.append_stdout("could not list models: %s\n" % str(e)[:80])
            return
        opts = [("%-46s  %8s  %s" % (f["name"][-46:], human(f["size"]),
                                     f["folder"] or "(root)"), f["path"]) for f in files]
        pick.options = opts
        move_out.outputs = ()
        move_out.append_stdout("%d file(s) under %s, newest first\n" % (len(opts), MODELS))

    def on_move(_):
        src = pick.value
        if not src:
            return
        folder = move_to.value + ("/" + move_sub.value.strip().strip("/")
                                  if move_sub.value.strip() else "")
        move_out.outputs = ()
        try:
            if SERVER_UP and api("/frizzy/dl/folders", timeout=5) is not None:
                r = api("/frizzy/dl/move", {"src": src, "folder": folder})
                if not r or not r.get("ok"):
                    raise ValueError((r or {}).get("error", "the server refused it"))
                dest = r["dest"]
            else:
                dest, _n = move_local(src, folder)
            move_out.append_stdout("moved to %s\n" % dest)
            if SERVER_UP:
                move_out.append_stdout("ComfyUI's model lists were refreshed too\n")
        except Exception as e:
            move_out.append_stdout("could not move it: %s\n" % str(e)[:140])
        refresh_files(None)

    move_btn.on_click(on_move)
    reload_btn.on_click(refresh_files)
    refresh_files(None)

    where = ("ComfyUI is running, so these go through its own queue and show up in the "
             "Models panel" if SERVER_UP else
             "ComfyUI is not running, so these download straight to disk here")
    display(widgets.VBox([
        widgets.HTML("<h3 style='margin:6px 0 2px'>Models</h3>"
                     "<div style='color:#888;font-size:12px'>%s. aria2 on 16 connections: %s. "
                     "Everything lands on the VM under <code>%s</code>. Nothing starts until "
                     "you have seen where it is going.</div>" %
                     (where, "yes" if ARIA2 else "no, falling back to curl", MODELS)),
        urls,
        widgets.HBox([default_folder, hf]),
        widgets.HBox([stage_btn, start_btn, drop_btn]),
        stage_box,
        out,
        widgets.HTML("<h3 style='margin:16px 0 2px'>Already on disk</h3>"
                     "<div style='color:#888;font-size:12px'>Newest first. Move anything the "
                     "guess got wrong.</div>"),
        widgets.HBox([pick, reload_btn]),
        widgets.HBox([move_to, move_sub, move_btn]),
        move_out,
    ], layout=WIDE))


build_downloader_ui()

# ============================================================================
# PART 2 - DOCTOR
# ============================================================================
rule("DOCTOR")
findings = []
fixes = []


def note(level, area, msg):
    findings.append([level, area, msg, False])
    {"OK": ok, "WARN": warn, "FAIL": fail}.get(level, info)("%-14s %s" % (area, msg))
    say("%-5s %-14s %s" % (level, area, msg))


def fixed(msg):
    """Always called straight after the note it repairs, so it marks that one."""
    fixes.append(msg)
    if findings:
        findings[-1][3] = True
    info("   fixed: " + msg)
    say("      FIXED %s" % msg)


say("=== frizzy comfy doctor ===")
say("when      %s" % time.strftime("%Y-%m-%d %H:%M:%S"))

# ---------------------------------------------------------------- hardware
step("Hardware")
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        note("OK", "gpu", "%s  %.1f GB  compute %d.%d  torch %s"
             % (p.name, p.total_memory / 1024 ** 3, p.major, p.minor, torch.__version__))
        if p.major < 8:
            info("compute < 8.0: no bf16, no fp8, no FlashAttention or Sage. Prefer Q4_K_M.")
    else:
        note("FAIL", "gpu", "torch cannot see a GPU - Runtime > Change runtime type > T4")
except Exception as e:
    note("FAIL", "torch", "not importable: %s" % str(e)[:90])

try:
    smi = subprocess.run("nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu "
                         "--format=csv,noheader", shell=True, capture_output=True,
                         text=True).stdout.strip()
    if smi:
        say("nvidia-smi  %s" % smi)
        info("nvidia-smi: %s" % smi)
except Exception:
    pass

free = free_gb()
note("OK" if free > 12 else ("WARN" if free > 5 else "FAIL"),
     "disk", "%.1f GB free on the session disk" % free)

# ---------------------------------------------------------------- install
step("Install")
if not (ROOT / "main.py").exists():
    note("FAIL", "comfyui", "%s has no main.py - run cell 1" % ROOT)
else:
    note("OK", "comfyui", str(ROOT))

try:
    import comfyui_frontend_package as _fp
    _static = Path(_fp.__file__).parent / "static"
    if (_static / "index.html").is_file():
        note("OK", "frontend", "%s  %s" % (getattr(_fp, "__version__", "?"), _static))
    else:
        note("FAIL", "frontend", "the package is installed but static/index.html is missing")
        if AUTO_FIX:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--force-reinstall", "comfyui-frontend-package"])
            fixed("reinstalled comfyui-frontend-package - restart the runtime, then cell 1")
except Exception:
    note("FAIL", "frontend", "comfyui_frontend_package is not importable - this is the")
    info("blank page with a healthy server. Cell 1 repairs it.")

# The frontend is a separate pip package pinned by ComfyUI's requirements.txt. A
# mismatch does not stop generation: the server runs, files land in output/, and
# the UI quietly fails to open dialogs or show results. That is worth checking
# before anything else, because it looks exactly like a network problem.
req = ROOT / "requirements.txt"
want = ""
if req.exists():
    m = re.search(r"comfyui[-_]frontend[-_]package\s*==\s*([0-9][^\s#]*)",
                  req.read_text(errors="ignore"), re.I)
    want = m.group(1) if m else ""
have = ""
try:
    import comfyui_frontend_package as _fp2
    have = getattr(_fp2, "__version__", "") or ""
except Exception:
    pass
if not have:
    try:
        import importlib.metadata as _md
        have = _md.version("comfyui-frontend-package")
    except Exception:
        have = ""
if want and have and want != have:
    note("FAIL", "frontend pin", "this ComfyUI wants %s, %s is installed" % (want, have))
    info("that breaks dialogs and the execution UI while generation keeps working")
    if AUTO_FIX:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "comfyui-frontend-package==%s" % want],
                           capture_output=True, text=True)
        if r.returncode == 0:
            fixed("installed comfyui-frontend-package==%s - rerun cell 2" % want)
        else:
            info("   pip refused: %s" % (r.stderr or "")[-140:].strip())
elif want:
    note("OK", "frontend pin", "%s, matches requirements.txt" % have)

mgr = ROOT / "custom_nodes" / "ComfyUI-Manager"
if mgr.is_dir():
    r = subprocess.run("git -C '%s' log -1 --format=%%h\\ %%cr" % mgr, shell=True,
                       capture_output=True, text=True)
    line = (r.stdout or "").strip()
    old_mgr = bool(re.search(r"(\d+)\s+(months|years)", line))
    note("WARN" if old_mgr else "OK", "manager", line or "not a git checkout")
    if old_mgr:
        info("an old Manager imports frontend APIs that no longer exist, and one")
        info("throw during extension setup takes out every panel after it")
        if AUTO_FIX:
            g = subprocess.run("git -C '%s' pull --ff-only" % mgr, shell=True,
                               capture_output=True, text=True)
            if g.returncode == 0 and "Already up to date" not in (g.stdout or ""):
                fixed("updated ComfyUI-Manager - rerun cell 2")

u = ROOT / "user"
if u.is_symlink():
    note("FAIL", "user/", "symlinked to %s - /api/userdata stalls and the UI never loads"
         % os.readlink(u))
    if AUTO_FIX:
        tmp = Path("/content/_user_local")
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        subprocess.run('rsync -a "%s/" "%s/"' % (u, tmp), shell=True, capture_output=True)
        u.unlink()
        shutil.move(str(tmp), str(u))
        fixed("user/ is a real local directory again")
elif u.is_dir():
    note("OK", "user/", "local directory, %d file(s)"
         % sum(1 for _ in u.rglob("*") if _.is_file()))

nodes = sorted(p for p in (ROOT / "custom_nodes").glob("*") if p.is_dir()) \
    if (ROOT / "custom_nodes").is_dir() else []
note("OK", "custom nodes", "%d installed" % len(nodes))

# ---------------------------------------------------------------- speed layer
step("Downloader and cache layer")
payload = {}
try:
    payload = json.loads(Path("/content/frizzy_ext.json").read_text())
except Exception:
    pass

stale = []
if not NODE.is_dir():
    stale = list(payload.keys())
    note("FAIL" if payload else "WARN", "downloader",
         "not installed - a model clicked inside ComfyUI would land on your laptop"
         + ("" if payload else ". Run cell 2, it installs it."))
else:
    for name, body in payload.items():
        f = NODE / name
        if not f.exists() or f.read_text(encoding="utf8") != body:
            stale.append(name)
    if stale:
        note("WARN", "downloader", "%d file(s) out of date" % len(stale))
    else:
        note("OK", "downloader", "installed at %s" % NODE.name)

if stale and payload and AUTO_FIX:
    (NODE / "web").mkdir(parents=True, exist_ok=True)
    for name in stale:
        (NODE / name).write_text(payload[name], encoding="utf8")
    fixed("rewrote %d extension file(s) - rerun cell 2 to load them" % len(stale))

compat = Path("/content/comfyui_frontend_compat")
served = compat if compat.is_dir() else (_static if "_static" in globals() else None)
if compat.is_dir():
    marker = compat / ".target"
    note("OK", "frontend build", "rewritten for %s"
         % (marker.read_text().strip() if marker.exists() else "an older browser"))

gz = 0
try:
    if served is not None and Path(served).is_dir():
        gz = sum(1 for _ in Path(served).rglob("*.gz"))
except Exception:
    gz = 0
if gz:
    note("OK", "precompressed", "%d .gz assets - the app arrives at about a third of its size" % gz)
else:
    note("WARN", "precompressed", "no .gz assets yet, cell 2 makes them")

# ---------------------------------------------------------------- models
step("Models")
bad, parts, total_bytes, per = [], [], 0, {}
if MODELS.is_dir():
    for f in MODELS.rglob("*"):
        if not f.is_file():
            continue
        n = f.stat().st_size
        if f.suffix in (".part", ".download", ".aria2"):
            parts.append(f)
            continue
        total_bytes += n
        per.setdefault(f.parent.name, [0, 0])
        per[f.parent.name][0] += 1
        per[f.parent.name][1] += n
        if f.suffix.lower() in (".safetensors", ".gguf", ".ckpt", ".pt", ".pth", ".bin"):
            try:
                with open(f, "rb") as fh:
                    magic = fh.read(8)
            except Exception:
                continue
            if n < 4096 or magic[:4] in (b"<!DO", b"<htm", b"<HTM", b'{"er', b"<!do"):
                bad.append(f)
for k in sorted(per):
    info("%-20s %d file(s)  %s" % (k + "/", per[k][0], human(per[k][1])))
note("OK", "models", "%s across %d folder(s)" % (human(total_bytes), len(per)))
if bad:
    note("FAIL", "corrupt", "%d file(s) are error pages saved with a model name" % len(bad))
    for f in bad[:6]:
        info("   %s" % f)
    if AUTO_FIX:
        for f in bad:
            try:
                f.unlink()
            except Exception:
                pass
        fixed("deleted %d error page(s) - download them again" % len(bad))
if parts:
    note("WARN", "leftovers", "%d unfinished download(s), %s"
         % (len(parts), human(sum(p.stat().st_size for p in parts))))
    if AUTO_FIX and free < 12:
        for f in parts:
            try:
                f.unlink()
            except Exception:
                pass
        fixed("removed unfinished downloads to free space")
    else:
        info("kept - aria2 resumes from these. AUTO_FIX clears them when disk is tight.")

# ---------------------------------------------------------------- drive
step("Drive")
if os.path.ismount("/content/drive"):
    d = Path("/content/drive/MyDrive/frizzy-comfy")
    snap = d / "snapshots" / "latest.json"
    note("OK", "drive", str(d) if d.is_dir() else "mounted, but %s does not exist yet" % d)
    if snap.exists():
        try:
            s = json.loads(snap.read_text())
            note("OK", "snapshot", "%s, %d node(s)" % (s.get("when", "?"), len(s.get("nodes", {}))))
        except Exception:
            note("WARN", "snapshot", "unreadable")
    else:
        note("WARN", "snapshot", "none yet - cell 3 writes one")
else:
    note("FAIL", "drive", "not mounted - nothing will persist. Rerun cell 1.")

# ---------------------------------------------------------------- server
if CHECK_SERVER:
    step("Server")
    if not port_open(PORT):
        note("WARN", "server", "nothing listening on %d - run cell 2" % PORT)
    elif not SERVER_UP:
        note("FAIL", "server", "port %d is held but /system_stats does not answer" % PORT)
        if AUTO_FIX:
            kill_all(PORT)
            fixed("killed the dead process holding the port - now run cell 2")
    else:
        stats = api("/system_stats") or {}
        dev = (stats.get("devices") or [{}])[0]
        note("OK", "server", "up: %s, %s free VRAM"
             % (dev.get("name", "?")[:40], human(dev.get("vram_free", 0))))
        if _LIB:
            try:
                hrows, worst = comfy_health(timeout=15)
                for dt, status, size, path, label, timed_out in hrows:
                    line = ("     TIMEOUT   %s" % path) if timed_out else \
                           ("   %7.2fs %3s %9s  %s" % (dt, status or "-", human(size), path))
                    print(line)
                    say(line)
                v = health_verdict(worst, timeout=15)
                note("WARN" if v else "OK", "boot endpoints",
                     v[1] if v else "all answer quickly")
            except Exception as e:
                note("WARN", "boot endpoints", "could not time them: %s" % str(e)[:70])
        # The frontend cannot parse a mangled body, and a failed /api/settings is
        # a settings dialog that never opens. Checked locally AND through the
        # tunnel, because only the tunnel can re-encode what it forwards.
        for path in ("/api/settings", "/api/object_info"):
            try:
                with urllib.request.urlopen(BASE + path, timeout=25) as r:
                    raw = r.read()
                json.loads(raw)
                note("OK", "api" + path[4:], "valid JSON, %s" % human(len(raw)))
            except Exception as e:
                note("FAIL", "api" + path[4:], "%s: %s" % (type(e).__name__, str(e)[:70]))

        tf = Path("/content/tunnel_url.txt")
        if tf.exists():
            parts = tf.read_text().split()
            live = parts[0] if parts else ""
            note("OK", "tunnel", "%s (%s)" % (live, parts[1] if len(parts) > 1 else "?"))
            try:
                import gzip
                rq = urllib.request.Request(live.rstrip("/") + "/api/settings",
                                            headers={"Accept-Encoding": "gzip",
                                                     "User-Agent": "frizzy-doctor"})
                with urllib.request.urlopen(rq, timeout=30) as r:
                    raw = r.read()
                    enc = r.headers.get("Content-Encoding", "")
                body = gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw
                json.loads(body)
                note("OK", "api via tunnel", "valid JSON%s" % (" (" + enc + ")" if enc else ""))
            except Exception as e:
                note("FAIL", "api via tunnel",
                     "%s: %s - this is the broken settings dialog"
                     % (type(e).__name__, str(e)[:60]))
                info("set FRIZZY_API_GZIP off (it is off by default) and relaunch")
        else:
            info("no tunnel URL recorded yet - cell 2 writes it when it hands you one")

        # is the cache layer actually answering?
        try:
            html = urllib.request.urlopen(BASE + "/", timeout=10).read().decode("utf8", "ignore")
            m = re.search(r'src="([^"]*assets/index-[^"]+\.js)"', html)
            if m:
                req = urllib.request.Request(BASE + "/" + m.group(1).lstrip("/"), method="HEAD")
                with urllib.request.urlopen(req, timeout=10) as r:
                    cc = r.headers.get("Cache-Control", "")
                note("OK" if "max-age" in cc else "WARN", "asset caching",
                     cc or "no Cache-Control header")
                if "no-store" in cc:
                    info("the whole app is re-downloaded on every page load. Rerun cell 2.")
        except Exception as e:
            note("WARN", "asset caching", "could not check: %s" % str(e)[:60])

# ---------------------------------------------------------------- log
step("Log")
if LOG.exists():
    body = LOG.read_text(errors="ignore")
    imports = [l for l in body.splitlines() if "IMPORT FAILED" in l or "Cannot import" in l]
    if imports:
        note("WARN", "custom nodes", "%d failed to import" % len(imports))
        for l in imports[-5:]:
            info("   " + l[:130])
            say("   " + l[:130])
    if _LIB and explain(body, quiet=True) is None:
        note("OK", "log", "nothing in the error taxonomy matched")
    errs = [l for l in body.splitlines()
            if any(k in l for k in ("Traceback", "ERROR", "CUDA out of memory", "Killed"))]
    if errs:
        note("WARN", "log", "%d error line(s)" % len(errs))
        for l in errs[-6:]:
            info("   " + l[:130])
            say("   " + l[:130])
    say("")
    say("--- last %d log lines ---" % LOG_LINES)
    for l in body.splitlines()[-LOG_LINES:]:
        print("   " + l[:150])
        say(l[:150])
else:
    note("WARN", "log", "no log yet - cell 2 has not run in this session")

# ---------------------------------------------------------------- cleaning
if DEEP_CLEAN:
    step("Deep clean")
    before = free_gb()
    subprocess.run("pip cache purge", shell=True, capture_output=True)
    hfc = Path.home() / ".cache/huggingface"
    if hfc.exists():
        shutil.rmtree(hfc, ignore_errors=True)
    fixed("caches cleared, %.2f GB back" % (free_gb() - before))

# ---------------------------------------------------------------- report
rule("REPORT")
open_fails = [f for f in findings if f[0] == "FAIL" and not f[3]]
open_warns = [f for f in findings if f[0] == "WARN" and not f[3]]
print("  %d ok, %d warning(s), %d failure(s) left, %d thing(s) fixed"
      % (sum(1 for f in findings if f[0] == "OK"), len(open_warns), len(open_fails), len(fixes)))
if fixes:
    print("\n  Fixed:")
    for f in fixes:
        print("    - " + f)
if open_fails:
    print("\n  Still wrong:")
    for lvl, area, msg, _ in open_fails:
        print("    - %-14s %s" % (area, msg))
if open_warns:
    print("\n  Worth knowing:")
    for lvl, area, msg, _ in open_warns:
        print("    - %-14s %s" % (area, msg))
print("\n  Next: %s" % ("rerun cell 2" if (fixes or open_fails)
                        else "nothing, this session is healthy"))
print("\n" + "-" * 74)
print("  Copy everything between the lines if you want to paste it back to me.")
print("-" * 74)
print("\n".join(REPORT))
print("-" * 74)


---

## Transports

| | Websockets | Needs |
|---|---|---|
| cloudflared | yes | nothing, but quick tunnels get throttled per IP |
| pinggy | yes | nothing, 60 minutes per tunnel |
| localhost.run | yes | nothing, same ssh mechanism, different operator |
| ngrok | yes | free authtoken in Secrets as `NGROK_TOKEN` |
| Colab proxy | **no** | nothing - cannot run ComfyUI, off by default |

All of them are quick tunnels that can vanish mid-session. The watchdog in cell 2 rebuilds
whichever one you were on and prints the new URL, so losing a tunnel no longer means losing
the session.

## When it breaks

| Symptom | Real cause |
|---|---|
| `CUDA out of memory` | VRAM - drop a quant, or `lowvram` |
| Process `Killed`, no traceback | host RAM, not VRAM - keep `--cache-none` on |
| `error while deserializing header` | the download was an error page - cell 4 finds and deletes those |
| `Torch not compiled with CUDA` | a pip line replaced torch; restart the runtime |
| Logo forever | cell 2 names the hung endpoint |
| Generation runs but the UI shows nothing | the websocket, or a frontend/backend version mismatch - cell 4 checks both |
| Settings dialog will not open | `/api/*` arriving unparseable, or an outdated Manager - cell 4 checks both |
| Model downloaded to your laptop | the Models panel is not loaded - rerun cell 2 |
| Everything in `checkpoints` | old builds; move them in cell 4 or the Models panel |
| Tunnel URL dead | the watchdog prints the replacement; also in `/content/tunnel_url.txt` |
| Node missing after restart | fresh session - cell 1 restores from your snapshot |

If a page still will not load, open `/check.html` on the same URL. It runs the whole
capability, asset, API and websocket check inside the browser and prints the result on
screen, no developer tools needed.

## Licence note

MiniMax H3's Community Licence excludes local deployment in the EU, UK, US and South Korea
without written authorisation. Wan, LTX, Flux, Qwen and Z-Image carry no such territory
clause.
